In [ ]:
# ==============================================================================
# 4CBOn2 — Kaggle5: the between-game learning loop
#
# This notebook is the daily driver. On a schedule it: pulls the ledger,
# reads yesterday's score back, picks the next one-knob experiment from
# the fixed queue, smoke-filters it on five toy games, writes the pending
# row, pushes the GPU kernel and stops.
#
# THE MACHINE NEVER SUBMITS. Finalize submissions manually in the Kaggle app.
# Accelerator: None (CPU-only) | Internet: ON
# ==============================================================================
print('4CBOn2 Kaggle5: Daily driver initialized. Ready.')

In [ ]:
# ## 1 — The Kaggle CLI
#
# Install it only if the image lacks it; the driver probes every flag it relies on against the CLI actually present, each run (>= 2.2,< 3: flags verified live on 2.2.4).
pass


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'kaggle>=2.2,<3'], check=True)
print(subprocess.run(['kaggle', '--version'], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
# ## 2 — Get the driver, the lever, the gym and the agent
#
# The GitHub repo stays the source of truth; anything fetched is hashed and printed. Files that fail to fetch fall back to the embedded byte-verified copies in section 2b — **loudly**.
pass


In [ ]:
import hashlib, os, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
BASE_K5 = 'https://raw.githubusercontent.com/mohamtur1/arc-agi-3-agent/main/kaggle5'
BASE_4CB = 'https://raw.githubusercontent.com/mohamtur1/4CBOn2/main/Kaggle'
FAILED = []
for url, name in [(f'{BASE_K5}/{f}', f) for f in ['kaggle5.py', 'lever2-grid.patch', 'lever3_apply.py', 'lever3_runtime.py', 'k5_gym.py']] + [(f'{BASE_4CB}/program_synthesis_agent_v4.py', 'program_synthesis_agent_v4.py')]:
    rc = os.system(f'curl -fsSL {url} -o {D}/{name}')
    if rc == 0:
        data = (D / name).read_bytes()
        print('fetched  %-32s %7d bytes  sha256 %s'
              % (name, len(data), hashlib.sha256(data).hexdigest()[:16]))
    else:
        FAILED.append(name)
        print('FETCH FAILED (rc=%d) %s -> embedded fallback will be used' % (rc, name))
print()
if FAILED:
    print('!' * 60)
    print('FETCH FAILURES:', ', '.join(FAILED))
    print('using the embedded byte-verified copies in the cells below')
    print('!' * 60)
else:
    print('all 5 files fetched from GitHub (source of truth).')
print(sorted(p.name for p in D.iterdir()))

In [ ]:
# ## 2b — Embedded fallbacks
#
# Each cell writes its file only if the fetch above did not, then prints the sha256 of whatever bytes are in play.
pass


In [ ]:
import hashlib, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
t = D / 'kaggle5.py'
EMBEDDED = r'''
"""kaggle5 — the between-game learning loop driver. Pushes daily, never submits.

Runs INSIDE a scheduled Kaggle notebook (CPU-only, internet ON) and drives the
competition kernel (GPU, internet OFF) purely through the Kaggle CLI.

What a day looks like now
  1. pull the ledger (dataset download; falls back loudly if unavailable)
  2. reconcile: read yesterday's submission rows back and attach them to the
     pending experiment (per-game telemetry when the kernel exposes it;
     absent / zero / errored submissions are all recorded as data)
  3. plan: fold the ledger, pick the next one-knob experiment from the fixed
     queue (with its documented backtrack rule), and log exactly why
  4. filter: run the local toy gym (k5_gym.py) — a smoke filter, never a
     score — and skip obviously-broken variants before they burn a day
  5. write the pending row (append-only, durable BEFORE the push)
  6. push the kernel with the knob cell injected — AND STOP. The machine
     never calls `competitions submit`. The human finalizes in the Kaggle
     app; the run prints exactly which variant was prepared and what to tap.

Stages
  check    free: pull, verify inputs, probe CLI flags, dry-run the injection,
           scan for the K5 knob channel, then probe submission-limits.
  plan     reconcile + print what today's experiment would be. Pushes nothing.
  gym      run the local filter for an ad-hoc --set KNOB=VALUE (offline OK).
  push     manual one-shot: ad-hoc experiment via --set, or lever-only repush
  score    reconcile + print the experiment table. Pushes nothing.
  daily    the scheduled cell: the full loop above, idempotent within a day.
  submit / all   REMOVED. They print why and exit non-zero. A human submits.

`daily` keeps no hidden state: everything lives in the append-only ledger
(experiments.jsonl + game_rows.jsonl), which is synced to a private Kaggle
dataset at the start and end of every run — and once more right before the
push, so a crash can never lose a pending row.

Run `python kaggle5.py --self-test` to exercise the whole loop offline
against _stub_kaggle.py: multi-day reconciliation, backtracking, dummies,
stale rows, refusals. No auth, no network, no Kaggle.

One honest fixed fact: this system measures. It does not compound. A knob
sweep is expected to find small deltas (and some negative ones); the value
is one trustworthy measurement per day and a ledger that explains itself.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path

# --------------------------------------------------------------------------
# CONFIG — edit only this block
# --------------------------------------------------------------------------

OWNER = "mohamedmalikturay"
KERNEL_SLUG = "arc-agi-3-milestone-2-solution-3e9adb"
KERNEL_URL = "https://www.kaggle.com/code/%s/%s" % (OWNER, KERNEL_SLUG)
COMPETITION = "arc-prize-2026-arc-agi-3"
SUBMISSION_FILE = "submission.parquet"

LEDGER_DATASET = "%s/kaggle5-ledger" % OWNER
LEDGER_FILES = ("experiments.jsonl", "game_rows.jsonl")

AGENT_FILE = "program_synthesis_agent_v4.py"
AGENT_REPO_URL = ("https://raw.githubusercontent.com/mohamtur1/4CBOn2/main/Kaggle/"
                  + AGENT_FILE)
AGENT_SHA256 = "8f447e454134d4a55dd0dfe3227c6273fa70c595f24773a0c6c84e1ce7b440a0"
GYM_FILE = "k5_gym.py"
KNOB_JSON_NAME = "k5_knobs.json"
AGENT_CHANNEL_MARKER = "K5_KNOBS_JSON"   # present in any agent with the K1 loader

DEVICE = "franzen-harness-v1"
ANCHOR_VARIANT = "franzen-base"
HOOK_HEADING_SUBSTR = "customization hook"

LEVER_TAG = "lever-2"
LEVER_PATCH_FILE = "lever2-grid.patch"     # the git patch that IS the lever

# The lever patches harness SOURCE, so it must land after section 1 (which
# copies the bundle and runs `git apply harness-changes.patch`) and before
# section 3 (which installs the runtime). Section 2's heading is the insert
# point; verified against the real notebook, which puts the lever at cells 5-7.
INJECT_AFTER_HEADING = "1. environment and submission mode"
INJECT_BEFORE_HEADING = "2. precaching"

# The applier cell reads these two names. They are defined in section 1.
REQUIRED_UPSTREAM_NAMES = ("WORKING_DIR", "BUNDLE_DIR")

# Only used when machine_shape comes back empty. Verified value, taken from the
# notebook's own metadata. The CLI docs write it NvidiaRtxPro6000.
FALLBACK_ACCELERATOR = "nvidiaRtxPro6000"

# The 5 inputs. push refuses if any is missing.
EXPECTED_COMPETITION_SOURCES = ["arc-prize-2026-arc-agi-3"]
EXPECTED_DATASET_SOURCES = [
    "dfranzen/pennyroyal-v253",
    "dfranzen/taaf-kaggle-source-bundle-copy",
]
EXPECTED_MODEL_SOURCES = [
    "dfranzen/intel-qwen3.8-flash-next-w4a16-autoround",
    "dfranzen/albucino-qwen3-8-flash-next-drafter",
]

STATUS_POLL_SECONDS = 60
STATUS_MAX_SECONDS = 9 * 3600 + 1800       # ARC-AGI-3 notebooks cap at 9 h
SCORE_POLL_SECONDS = 120
SCORE_MAX_SECONDS = 6 * 3600

STALE_AFTER_DAYS = 2                  # pending this long unanswered -> stale

# Scheduled runs fire in UTC. Freetown is UTC+0 all year, so 0 is correct there.
LOCAL_UTC_OFFSET_HOURS = 0

# --------------------------------------------------------------------------
# THE KNOBS — full default state (must mirror the agent module exactly;
# self-test imports the agent and refuses to run on drift). GRID/CELL are
# structural and are permanently OFF the sweep list.
# --------------------------------------------------------------------------

KNOB_SURFACE = {
    "analyzer_timeout": {"type": "float", "min": 0.0, "default": 900.0},
    "max_runtime_s_per_game": {"type": "float", "min": 0.0, "default": 31920.0},
    "max_actions_per_game": {"type": "int_or_none", "min": 1, "default": None},
    "concurrency": {"type": "int", "min": 1, "default": 120},
}

DEFAULT_KNOBS = {
    "analyzer_timeout": 900.0,
    "max_runtime_s_per_game": 31920.0,
    "max_actions_per_game": None,
    "concurrency": 120,
}

EXPERIMENT_QUEUE = [
    ("analyzer_timeout", [1200.0],
     "up: 900.0 -> 1200.0 gives analyzer 33% more time per puzzle; respects |log2(1200/900)| <= 1"),
    ("max_actions_per_game", [500],
     "cap: None -> 500 caps the uncapped action side to prevent runaway infinite loops"),
    ("max_runtime_s_per_game", [31320.0],
     "trim: 31920 -> 31320 trims ~2% (600s / 10m buffer) per game"),
    ("concurrency", [60],
     "down: 120 -> 60 halves concurrency to reduce memory pressure / contention"),
]

# Historical anchors, seeded once into an empty ledger. Scores from the
# owner's first two submissions (v1 -> 0.06, v2 -> 0.07); agent file hashes
# were not recorded at the time, which is exactly why the ledger exists now.
IMPORTED_HISTORY = [
    {"variant": "v1-imported", "score": 0.06,
     "note": "historical import: fixed-bag-of-transforms agent; predates the loop"},
    {"variant": "v2-imported", "score": 0.07,
     "note": "historical import: learning machinery (EpisodeLedger/GameKnowledge/"
             "RuleRecord); predates the loop"},
]

# CLI subcommands + flags this driver relies on, probed against the INSTALLED
# cli (bug D: no inherited flag assumptions; verified present on 2.2.4 but the
# probe checks whatever is actually in the container, every run).
REQUIRED_CLI = {
    ("kernels", "pull"): ["-p", "-m"],
    ("kernels", "push"): ["-p", "--accelerator"],
    ("kernels", "status"): [],
    ("kernels", "files"): ["--format"],
    ("kernels", "output"): ["-p"],
    ("competitions", "submissions"): ["--format", "--page-size"],
    ("competitions", "submission-limits"): ["--json"],
    ("datasets", "download"): ["-p", "--unzip"],
    ("datasets", "create"): ["-p"],
    ("datasets", "version"): ["-p", "-m", "-t"],
}

# --------------------------------------------------------------------------


def say(msg: str = "") -> None:
    print(msg, flush=True)


def die(msg: str) -> None:
    say("")
    say("STOP — " + msg)
    raise SystemExit(1)


def norm(s: str) -> str:
    return re.sub(r"[\s\-_:]+", " ", (s or "").strip().lower())


def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()


def canon(value) -> str:
    """Canonical string for a knob value (list vs tuple safe)."""
    return json.dumps(value, sort_keys=True)


# --------------------------------------------------------------------------
# CLI plumbing
# --------------------------------------------------------------------------

def check_auth() -> None:
    """Report the credential path. The abrasive version lives in the
    notebook's auth cell (bug A); here we only describe what was found. The
    last resort is Kaggle's ambient notebook token — never an abort."""
    if os.environ.get("KAGGLE_API_TOKEN"):
        say("auth: KAGGLE_API_TOKEN is set")
        return
    if os.environ.get("KAGGLE_API_V1_TOKEN"):
        say("auth: KAGGLE_API_V1_TOKEN is set (Kaggle-injected v1 token)")
        return
    for p in ("~/.kaggle/access_token", "~/.kaggle/access_token.txt",
              "~/.kaggle/kaggle.json"):
        f = Path(os.path.expanduser(p))
        if f.is_file() and f.read_text().strip():
            say("auth: using %s" % p)
            return
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        say("auth: using legacy KAGGLE_USERNAME/KAGGLE_KEY")
        return
    say("auth: no explicit credentials. Inside a Kaggle notebook the SDK reads")
    say("      the token Kaggle injects, so this is expected and may still work.")


def kaggle(args: list[str], capture: bool = True, timeout: int = 900,
           allow_fail: bool = False, quiet: bool = False) -> subprocess.CompletedProcess:
    """Run the kaggle CLI.

    KAGGLE_BIN overrides the command (space separated) — used for the test
    double, and handy for pinning a specific install.
    """
    override = os.environ.get("KAGGLE_BIN", "").strip()
    if override:
        base = override.split()
    elif shutil.which("kaggle"):
        base = [shutil.which("kaggle")]
    else:
        base = [sys.executable, "-m", "kaggle"]
    cmd = base + args
    say("$ " + " ".join(cmd))
    r = subprocess.run(
        cmd,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
        text=True,
        timeout=timeout,
    )
    if capture and r.stdout and not quiet:
        say(r.stdout.rstrip())
    if r.returncode != 0 and not allow_fail:
        if quiet and r.stdout:
            say(r.stdout.rstrip())
        die("kaggle %s exited %d" % (args[0], r.returncode))
    return r


def probe_cli_flags() -> None:
    """Bug D: verify every subcommand + flag this driver uses against the
    CLI actually installed in the container, not against a version note."""
    if os.environ.get("KAGGLE_BIN", "").strip():
        say("  cli flag probe: skipped (KAGGLE_BIN override in use)")
        return
    missing = []
    for (grp, sub), flags in sorted(REQUIRED_CLI.items()):
        r = kaggle([grp, sub, "--help"], allow_fail=True, quiet=True)
        if r.returncode != 0:
            missing.append("%s %s (subcommand absent)" % (grp, sub))
            continue
        for fl in flags:
            if fl not in (r.stdout or ""):
                missing.append("%s %s %s" % (grp, sub, fl))
    if missing:
        die("the installed kaggle CLI does not provide everything this driver "
            "needs:\n  " + "\n  ".join(missing) +
            "\n  Do not patch around this silently: fix the driver or pin the CLI.")
    say("  cli flag probe: all %d subcommands and flags present on the "
        "installed CLI" % len(REQUIRED_CLI))


def probe_submission_limits() -> None:
    """Bug E: `competitions submission-limits` (present on 2.2.4 — verified),
    best-effort and non-blocking: printed for the human, never parsed into
    control flow."""
    r = kaggle(["competitions", "submission-limits", COMPETITION, "--json"],
               allow_fail=True, quiet=True)
    out = (r.stdout or "").strip()
    if r.returncode != 0 or not out:
        say("  submission-limits: unavailable on this CLI/competition; "
            "assuming the usual 1/day (informational only)")
        return
    say("  submission-limits (informational): %s" % out[:400])


def _json_from(stdout: str):
    try:
        return json.loads(stdout[stdout.index("["): stdout.rindex("]") + 1])
    except (ValueError, json.JSONDecodeError):
        return None


# --------------------------------------------------------------------------
# pull + guards
# --------------------------------------------------------------------------

def pull(work: Path) -> tuple[Path, dict]:
    work.mkdir(parents=True, exist_ok=True)
    out = kaggle(["kernels", "pull", "-p", str(work), "-m",
                  f"{OWNER}/{KERNEL_SLUG}"]).stdout
    meta_path = work / "kernel-metadata.json"
    if not meta_path.is_file():
        die("pull produced no kernel-metadata.json:\n" + out)
    meta = json.loads(meta_path.read_text())
    code = work / meta.get("code_file", "")
    if not code.is_file():
        die("pull produced no notebook file (expected %s)" % code.name)
    return code, meta


def guard_metadata(meta: dict) -> None:
    say("")
    say("--- data sources on the pulled kernel ---")
    for key in ("competition_sources", "dataset_sources", "kernel_sources",
                "model_sources"):
        say("  %-22s %s" % (key, meta.get(key) or []))
    say("  %-22s %s" % ("enable_gpu", meta.get("enable_gpu")))
    say("  %-22s %s" % ("enable_internet", meta.get("enable_internet")))
    say("  %-22s %r" % ("machine_shape", meta.get("machine_shape")))

    def missing(expected: list[str], actual: list[str]) -> list[str]:
        have = [norm(x).lstrip("c/") for x in (actual or [])]
        out = []
        for e in expected:
            ne = norm(e)
            if not any(h == ne or h.startswith(ne + "/") for h in have):
                out.append(e)
        return out

    problems = []
    if m := missing(EXPECTED_COMPETITION_SOURCES, meta.get("competition_sources")):
        problems.append("competition_sources missing %s" % m)
    if m := missing(EXPECTED_DATASET_SOURCES, meta.get("dataset_sources")):
        problems.append("dataset_sources missing %s" % m)
    if m := missing(EXPECTED_MODEL_SOURCES, meta.get("model_sources")):
        problems.append("model_sources missing %s" % m)
    if meta.get("enable_internet") is not False:
        problems.append("enable_internet is %r, must be false"
                        % meta.get("enable_internet"))

    if problems:
        die(
            "the pulled kernel is not configured the way your baseline is.\n"
            "  " + "\n  ".join(problems) + "\n"
            "  Pushing now would silently drop inputs and burn a run."
        )
    # The real notebook carries accelerator=nvidiaRtxPro6000 while enable_gpu is
    # false. The accelerator field is what selects hardware, so this is only a note.
    if not meta.get("enable_gpu"):
        say("  note: enable_gpu is falsy, machine_shape=%r. The accelerator field"
            % meta.get("machine_shape"))
        say("        selects hardware on this kernel, so that combination is expected.")
    say("  -> all 5 inputs present, internet off. OK to push.")


def guard_slug(meta: dict) -> None:
    title = (meta.get("title") or "").strip()
    kslug = (meta.get("id") or "").split("/")[-1]
    if not title or not kslug:
        return
    slug = re.sub(r"-+", "-", re.sub(r"[^a-z0-9]+", "-", title.lower())).strip("-")
    if slug and slug != kslug.lower():
        say("  note: title %r slugifies to %r but the id slug is %r."
            % (title, slug, kslug))
        say("        The id in the metadata wins; check the URL the push prints.")


def needs_accelerator_flag(meta: dict) -> bool:
    return not (meta.get("machine_shape") or "").strip()


def find_hook_heading(nb: dict) -> int | None:
    for i, c in enumerate(nb.get("cells", [])):
        if _is_md(c):
            t = cell_text(c).lower()
            if HOOK_HEADING_SUBSTR in t:
                return i
    return None


def has_knob_channel(nb: dict) -> bool:
    for c in nb.get("cells", []):
        if AGENT_CHANNEL_MARKER in cell_text(c):
            return True
    return find_hook_heading(nb) is not None


def is_hand_lever_cell(cell: dict) -> bool:
    if cell.get("cell_type") != "code":
        return False
    src = cell_text(cell)
    if "Lever 2 — add .grid" in src or "sb = python_tool_sandbox._SANDBOX_BOOTSTRAP" in src:
        return True
    if "from inference.agent import prompts, python_tool_sandbox, tool_agent" in src:
        return True
    return False


def is_empty_code_cell(cell: dict) -> bool:
    if cell.get("cell_type") != "code":
        return False
    src = cell_text(cell).strip()
    outputs = cell.get("outputs", [])
    return not src and not outputs


def is_misclassified_markdown(cell: dict) -> bool:
    if cell.get("cell_type") != "code":
        return False
    src = cell_text(cell).strip()
    if not src:
        return False
    if src.startswith("%%") or "writefile" in src or "_K5" in src or LEVER_TAG in src:
        return False
    lines = [l.strip() for l in src.splitlines() if l.strip()]
    if not lines:
        return False
    first = lines[0]
    is_md_header = first.startswith(("# ", "## ", "### ", "**", "- "))
    has_bold_or_update = any(l.startswith(("**", "* ", "- ", "> ")) or "update (v" in l.lower() for l in lines)
    if not (is_md_header and has_bold_or_update):
        return False
    try:
        compile(src, "<cell>", "exec")
        return False
    except SyntaxError:
        return True


def sanitize_notebook(nb: dict) -> dict:
    nb = json.loads(json.dumps(nb))
    cells = nb.get("cells", [])
    drop = set()
    for i, c in enumerate(cells):
        if is_empty_code_cell(c):
            say("  [sanitizer] removed empty output-less code cell at index %d" % i)
            drop.add(i)
        elif is_hand_lever_cell(c):
            say("  [sanitizer] removed hand lever-2 prototype cell at index %d" % i)
            drop.add(i)
        elif is_misclassified_markdown(c):
            say("  [sanitizer] converted misclassified markdown cell at index %d to markdown" % i)
            c["cell_type"] = "markdown"
            c.pop("execution_count", None)
            c.pop("outputs", None)
    if drop:
        nb["cells"] = [c for i, c in enumerate(cells) if i not in drop]
    return nb


def find_knob_cell_index(nb: dict) -> int | None:
    for i, c in enumerate(nb.get("cells", [])):
        if c.get("cell_type") == "code" and (knob_marker() in cell_text(c) or AGENT_CHANNEL_MARKER in cell_text(c)):
            return i
    return None


# --------------------------------------------------------------------------
# notebook surgery
# --------------------------------------------------------------------------

def cell_text(cell: dict) -> str:
    src = cell.get("source", "")
    return src if isinstance(src, str) else "".join(src)


def _is_md(cell: dict) -> bool:
    return cell.get("cell_type") == "markdown"


def _heading_match(cell: dict, target: str, mode: str) -> str | None:
    if not _is_md(cell):
        return None
    for line in cell_text(cell).splitlines():
        stripped = line.lstrip()
        if not stripped.startswith("#"):
            continue
        head = norm(stripped.lstrip("#"))
        if (mode == "eq" and head == target) or (mode == "prefix"
                                                 and head.startswith(target)):
            return line.strip()
    return None


def find_injection_point(nb: dict, explicit_index: int | None) -> int:
    """Index to insert at: end of the anchor section, i.e. just before the next
    markdown heading. Inserting before the next heading rather than at
    `heading + 1` keeps this correct when a section holds several code cells."""
    cells = nb.get("cells", [])
    if explicit_index is not None:
        if not 0 <= explicit_index <= len(cells):
            die("--inject-at %d out of range (notebook has %d cells)"
                % (explicit_index, len(cells)))
        return explicit_index

    hit = None
    for i, c in enumerate(cells):
        line = _heading_match(c, norm(INJECT_AFTER_HEADING), "eq")
        if line:
            hit = ("after %r" % line, i)
            break
    if hit is None:
        for i, c in enumerate(cells):
            line = _heading_match(c, norm(INJECT_BEFORE_HEADING), "prefix")
            if line:
                say("  fallback anchor: heading %r at cell %d -> insert at %d"
                    % (line, i, i))
                return i
    if hit is None:
        die(
            "no anchor found. Run stage=check, read the heading map, then pass --inject-at N.\n"
            "  Expected a markdown heading matching %r or starting %r."
            % (INJECT_AFTER_HEADING, INJECT_BEFORE_HEADING)
        )

    label, i = hit
    for j in range(i + 1, len(cells)):
        if _is_md(cells[j]):
            say("  anchor: %s at cell %d, next heading at cell %d -> insert at %d"
                % (label, i, j, j))
            return j
    say("  anchor: %s at cell %d, no later heading -> insert at end (%d)"
        % (label, i, len(cells)))
    return len(cells)


def guard_upstream_names(nb: dict, at: int) -> None:
    """The applier cell reads WORKING_DIR and BUNDLE_DIR. If section 1 does not
    define them the lever would NameError nine hours into a run."""
    before = "".join(cell_text(c) for c in nb.get("cells", [])[:at])
    absent = [n for n in REQUIRED_UPSTREAM_NAMES
              if not re.search(r"\b%s\s*=" % n, before)]
    if absent:
        die(
            "the cells before the injection point do not define: %s\n"
            "  The lever applier reads those names. Check that section 1 is intact,\n"
            "  or pass --inject-at N to place the lever after the cell that defines them."
            % ", ".join(absent)
        )
    say("  upstream names present before the lever: %s"
        % ", ".join(REQUIRED_UPSTREAM_NAMES))


def applier_marker(lever_tag: str) -> str:
    """Goes in the applier cell. The patch cell cannot carry a comment marker:
    `%%writefile` is a cell magic and must be the first line, so a leading
    comment would turn it into plain text and write nothing."""
    return "# ===== %s apply (kaggle5) =====" % lever_tag


def knob_marker() -> str:
    return "# ===== k5 knobs (kaggle5) ====="


def _is_writefile(cell: dict, filename: str) -> bool:
    lines = cell_text(cell).splitlines()
    return bool(lines) and lines[0].startswith("%%writefile") \
        and filename in lines[0]


def lever_anchor(nb: dict, lever_tag: str) -> int | None:
    """Index of the applier cell, or None. The patch cell is the one before it."""
    mk = applier_marker(lever_tag)
    for i, c in enumerate(nb.get("cells", [])):
        if c.get("cell_type") == "code" and mk in cell_text(c):
            return i
    return None


def lever_cells_present(nb: dict, lever_tag: str) -> list[int]:
    i = lever_anchor(nb, lever_tag)
    if i is None:
        return []
    cells = nb.get("cells", [])
    out = []
    if i > 0 and _is_writefile(cells[i - 1], LEVER_PATCH_FILE):
        out.append(i - 1)
    out.append(i)
    if i + 1 < len(cells) and knob_marker() in cell_text(cells[i + 1]):
        out.append(i + 1)
    return out


VARIANT_LINE = 'VARIANT = "lever3"  # k5-variant-line'


def set_variant(nb: dict, variant: str) -> dict:
    """Bake the requested variant into the applier cell (idempotent).

    lever3 : memory + hard prune gate + prompt directives (default)
    kernel : memory view present, gate off, prompt untouched (A/B rollback)
    """
    if variant == "lever3":
        return nb
    anchor = lever_anchor(nb, LEVER_TAG)
    if anchor is None:
        die("variant rewrite: no applier cell found in the pulled notebook")
    cell = nb["cells"][anchor]
    src = cell_text(cell)
    if VARIANT_LINE not in src:
        die("variant rewrite: the applier cell has no %r line; refusing to "
            "rewrite blindly" % VARIANT_LINE)
    cell["source"] = src.replace(
        VARIANT_LINE, 'VARIANT = "%s"  # k5-variant-line' % variant, 1)
    say("  variant=%s baked into the applier cell" % variant)
    return nb


def build_lever_cells(patch_text: str, applier: str, lever_tag: str) -> list[dict]:
    if ("'" * 3) in patch_text:
        die(LEVER_PATCH_FILE
            + " contains a triple quote; cannot embed it in a writefile cell")
    body = patch_text if patch_text.endswith(chr(10)) else patch_text + chr(10)
    write = "%%writefile /kaggle/working/" + LEVER_PATCH_FILE + chr(10) + body
    app = applier_marker(lever_tag) + chr(10) + applier.strip() + chr(10)
    return [
        {"cell_type": "code", "metadata": {"trusted": True,
                                           "kaggle5": lever_tag + "-patch"},
         "execution_count": None, "outputs": [], "source": write},
        {"cell_type": "code", "metadata": {"trusted": True,
                                           "kaggle5": lever_tag + "-apply"},
         "execution_count": None, "outputs": [], "source": app},
    ]


def build_knob_cell(payload: dict) -> dict:
    src = (
        knob_marker() + chr(10) +
        "# Injected after ## 7. Customization hook" + chr(10) +
        "# Applies assignments allowlist-only, loudly refuses unknown names / uncaps / bools," + chr(10) +
        "# dies if the upstream harness renamed an attribute, prints '[k5] bm.solver.x: old -> new'" + chr(10) +
        "# per knob, and writes /kaggle/working/k5_knobs.json." + chr(10) +
        "import json as _k5_json" + chr(10) +
        "import pathlib as _k5_pathlib" + chr(10) + chr(10) +
        "# JSON literal aliases so valid JSON is valid Python at runtime" + chr(10) +
        "null = None" + chr(10) +
        "false = False" + chr(10) +
        "true = True" + chr(10) + chr(10) +
        "_K5 = " + json.dumps(payload, indent=2, sort_keys=True) + chr(10) + chr(10) +
        "_K5_SURFACE = {" + chr(10) +
        "    'analyzer_timeout': float," + chr(10) +
        "    'max_runtime_s_per_game': float," + chr(10) +
        "    'max_actions_per_game': (int, type(None))," + chr(10) +
        "    'concurrency': int," + chr(10) +
        "}" + chr(10) + chr(10) +
        "if 'bm' not in globals() or not hasattr(bm, 'solver'):" + chr(10) +
        "    raise RuntimeError('[k5] bm.solver not found in globals')" + chr(10) + chr(10) +
        "for _k, _v in _K5.get('knobs', {}).items():" + chr(10) +
        "    if _k not in _K5_SURFACE:" + chr(10) +
        "        raise ValueError(f'[k5] Refusing unknown knob: {_k}')" + chr(10) +
        "    if isinstance(_v, bool):" + chr(10) +
        "        raise TypeError(f'[k5] Refusing boolean knob value: {_k}={_v}')" + chr(10) +
        "    if not hasattr(bm.solver, _k):" + chr(10) +
        "        raise AttributeError(f'[k5] Upstream harness has no attribute bm.solver.{_k}')" + chr(10) +
        "    _old = getattr(bm.solver, _k)" + chr(10) +
        "    if _old is not None and _v is None:" + chr(10) +
        "        raise ValueError(f'[k5] Refusing uncap for {_k}')" + chr(10) +
        "    setattr(bm.solver, _k, _v)" + chr(10) +
        "    print(f'[k5] bm.solver.{_k}: {_old} -> {_v}')" + chr(10) + chr(10) +
        "_k5_pathlib.Path('/kaggle/working').mkdir(parents=True, exist_ok=True)" + chr(10) +
        "_k5_pathlib.Path('/kaggle/working/" + KNOB_JSON_NAME + "').write_text(" + chr(10) +
        "    _k5_json.dumps(_K5, indent=1, sort_keys=True)" + chr(10) +
        ")" + chr(10) +
        "print('[k5] run binding: variant=%r changed=%r'" + chr(10) +
        "      % (_K5.get('variant'), _K5.get('changed')))" + chr(10) +
        "del _k5_json, _k5_pathlib, _K5, _K5_SURFACE" + chr(10)
    )
    return {"cell_type": "code",
            "metadata": {"trusted": True, "kaggle5": "knobs"},
            "execution_count": None, "outputs": [], "source": src}


def inject(nb: dict, patch_text: str, applier: str, lever_tag: str,
           explicit_index: int | None = None,
           knob_payload: dict | None = None) -> dict:
    nb = json.loads(json.dumps(nb))
    nb = sanitize_notebook(nb)

    lever_cells = build_lever_cells(patch_text, applier, lever_tag)
    present = lever_cells_present(nb, lever_tag)
    if present:
        say("  %s already present at cells %s — replacing in place, no duplicates"
            % (lever_tag, present))
        drop = set(present)
        keep = [c for i, c in enumerate(nb["cells"]) if i not in drop]
        at = min(present)
        keep[at:at] = lever_cells
        nb["cells"] = keep
    else:
        at = find_injection_point(nb, explicit_index)
        guard_upstream_names(nb, at)
        nb["cells"][at:at] = lever_cells
        say("  inserted %s at cells %d-%d" % (lever_tag, at, at + len(lever_cells) - 1))

    if knob_payload is not None:
        k_cell = build_knob_cell(knob_payload)
        existing_k_idx = find_knob_cell_index(nb)
        if existing_k_idx is not None:
            nb["cells"][existing_k_idx] = k_cell
            say("  replaced existing knob cell at cell %d in place" % existing_k_idx)
        else:
            h_idx = find_hook_heading(nb)
            if h_idx is not None:
                code_idx = None
                for j in range(h_idx + 1, len(nb["cells"])):
                    if _is_md(nb["cells"][j]):
                        break
                    if nb["cells"][j].get("cell_type") == "code":
                        code_idx = j
                ins_idx = (code_idx + 1) if code_idx is not None else (h_idx + 1)
                nb["cells"].insert(ins_idx, k_cell)
                say("  inserted knob cell at cell %d (after hook heading)" % ins_idx)
            else:
                anchor = lever_anchor(nb, lever_tag)
                ins_idx = (anchor + 1) if anchor is not None else len(nb["cells"])
                nb["cells"].insert(ins_idx, k_cell)
                say("  inserted knob cell at cell %d" % ins_idx)

    return nb


def verify_injection(nb: dict, patch_text: str, lever_tag: str,
                     knob_payload: dict | None = None) -> None:
    anchor = lever_anchor(nb, lever_tag)
    if anchor is None:
        die("post-check: no applier cell found")
    cells = nb.get("cells", [])
    if anchor == 0 or not _is_writefile(cells[anchor - 1], LEVER_PATCH_FILE):
        die("post-check: the cell before the applier is not a %%writefile "
            + LEVER_PATCH_FILE + " cell")
    if patch_text.strip() not in cell_text(cells[anchor - 1]):
        die("post-check: patch body not intact in the writefile cell")
    apply_src = cell_text(cells[anchor])
    if "git" not in apply_src or "apply" not in apply_src:
        die("post-check: the applier cell does not git-apply the patch")
    say("  post-check OK: writefile cell + applier, patch intact")

    if knob_payload is not None:
        k_idx = find_knob_cell_index(nb)
        if k_idx is None:
            die("post-check: expected knob cell not found")
        knob_src = cell_text(cells[k_idx])
        m = re.search(r"_K5 = (\{.*?\})" + chr(10) + chr(10) + "_K5_SURFACE", knob_src, re.S)
        if not m:
            die("post-check: could not find _K5 payload dictionary in knob cell")
        try:
            got_payload = json.loads(m.group(1))
        except json.JSONDecodeError as exc:
            die("post-check: knob cell payload does not parse as JSON: %s" % exc)
        if got_payload.get("format") != 2:
            die("post-check: expected format 2 payload in knob cell, got %r" % got_payload.get("format"))
        say("  knob cell present and parses OK")


# --------------------------------------------------------------------------
# stateless cycle discovery + submission table (read-only)
# --------------------------------------------------------------------------

def kernel_status() -> str:
    out = kaggle(["kernels", "status", f"{OWNER}/{KERNEL_SLUG}"],
                 quiet=True).stdout
    m = re.search(r'has status "([^"]+)"', out)
    return (m.group(1) if m else out.strip()).lower()


def kernel_has_submission_file(version: int | None = None) -> bool:
    spec = f"{OWNER}/{KERNEL_SLUG}" + (f"/{version}" if version else "")
    r = kaggle(["kernels", "files", spec, "--format", "json"],
               allow_fail=True, quiet=True)
    if r.returncode != 0 or not r.stdout.strip():
        return False
    try:
        data = _json_from(r.stdout)
        if isinstance(data, list):
            return any(f.get("name") == SUBMISSION_FILE for f in data)
    except Exception:
        pass
    return False


def submissions() -> list[dict]:
    r = kaggle(["competitions", "submissions", COMPETITION, "--format", "json",
                "--page-size", "20"], allow_fail=True, quiet=True)
    if r.returncode != 0:
        say("  competitions submissions: unavailable (rc=%d) — starting with empty submissions" % r.returncode)
        if r.stdout and r.stdout.strip():
            say("  " + r.stdout.strip().splitlines()[0])
        return []
    data = _json_from(r.stdout)
    if not isinstance(data, list):
        say("  competitions submissions: response not a JSON list — treating as empty")
        return []
    return data


def local_today(offset_hours: int = LOCAL_UTC_OFFSET_HOURS) -> str:
    env = os.environ.get("K5_TODAY", "").strip()   # testing hook, documented
    if env:
        return env
    return (datetime.now(timezone.utc)
            + timedelta(hours=offset_hours)).strftime("%Y-%m-%d")


def date_of(sub: dict) -> str:
    return str(sub.get("date") or "")[:10]


def to_float(x) -> float | None:
    try:
        if x is None or x == "":
            return None
        return float(x)
    except (TypeError, ValueError):
        return None


def _days_old(ymd: str, today: str) -> int:
    try:
        a = datetime.strptime(ymd, "%Y-%m-%d")
        b = datetime.strptime(today, "%Y-%m-%d")
        return (b - a).days
    except ValueError:
        return 999


# --------------------------------------------------------------------------
# PIECE ONE — the ledger that survives the night
#
# Append-only JSON lines, event-sourced: nothing is ever rewritten, new facts
# are new lines. `experiments.jsonl` carries one experiment row and its life
# events (pushed -> scored/zero/error / filtered / stale / abandoned).
# `game_rows.jsonl` carries the per-game telemetry. Every experiment row
# records the FULL knob state plus the agent source hash, so any row is
# reproducible from the ledger alone.
# --------------------------------------------------------------------------

EXP_FIELDS = ("exp", "variant", "device", "date", "changed", "knobs", "agent_sha256",
              "reason")


class Ledger:
    def __init__(self, ldir: Path):
        self.dir = ldir
        self.dir.mkdir(parents=True, exist_ok=True)
        for name in LEDGER_FILES:
            f = self.dir / name
            if not f.exists():
                f.write_text("")

    def path(self, name: str) -> Path:
        return self.dir / name

    # ---- raw IO (append-only; read tolerates and counts bad lines) --------
    def read_lines(self, name: str) -> list[str]:
        p = self.path(name)
        return [ln for ln in p.read_text().splitlines() if ln.strip()]

    def events(self, name: str = "experiments.jsonl") -> list[dict]:
        out, bad = [], 0
        for ln in self.read_lines(name):
            try:
                obj = json.loads(ln)
                if isinstance(obj, dict):
                    out.append(obj)
                    continue
            except json.JSONDecodeError:
                pass
            bad += 1
        if bad:
            say("  ledger: %d unreadable line(s) in %s (kept untouched, skipped)"
                % (bad, name))
        return out

    def append(self, obj: dict, name: str = "experiments.jsonl") -> None:
        obj = dict(obj)
        obj.setdefault("at",
                       datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))
        line = json.dumps(obj, sort_keys=True)
        with self.path(name).open("a") as fh:
            fh.write(line + "\n")

    # ---- remote sync ------------------------------------------------------
    def merge_remote_dir(self, remote: Path) -> int:
        """Union of exact lines, local order first, remote additions appended.
        Append-only sets make this idempotent and order-stable."""
        added = 0
        for name in LEDGER_FILES:
            rp = remote / name
            if not rp.is_file():
                continue
            have = set(self.read_lines(name))
            new = [ln for ln in rp.read_text().splitlines()
                   if ln.strip() and ln not in have]
            if new:
                with self.path(name).open("a") as fh:
                    for ln in new:
                        fh.write(ln + "\n")
                added += len(new)
        return added

    def sync_in(self) -> None:
        """Pull the ledger dataset. Absent dataset / no auth / first run are
        all normal: the local files are the source of truth either way."""
        tmp = Path(tempfile.mkdtemp(prefix="k5_ledger_in_"))
        r = kaggle(["datasets", "download", "-d", LEDGER_DATASET,
                    "-p", str(tmp), "--unzip", "-q"],
                   allow_fail=True, quiet=True)
        if r.returncode != 0:
            say("  ledger sync-in: dataset %s not downloaded (first run? no "
                "auth yet?) — starting from the local copy" % LEDGER_DATASET)
            return
        added = self.merge_remote_dir(tmp)
        say("  ledger sync-in: merged %d remote line(s) from %s"
            % (added, LEDGER_DATASET))

    def sync_out(self, note: str) -> None:
        """Push a new dataset version. Tried, in order: datasets version ->
        datasets create -> GitHub commit (only if a token env var exists) ->
        a loud manual instruction. The local copy is always intact regardless.
        """
        meta = {
            "title": "kaggle5-ledger",
            "id": LEDGER_DATASET,
            "licenses": [{"name": "CC0-1.0"}],
        }
        (self.dir / "dataset-metadata.json").write_text(json.dumps(meta, indent=1))
        (self.dir / "datasets-metadata.json").write_text(json.dumps(meta, indent=1))
        r = kaggle(["datasets", "version", "-p", str(self.dir),
                    "-m", note[:250], "-t", "-q"], allow_fail=True, quiet=True)
        if r.returncode == 0:
            say("  ledger sync-out: new dataset version (%s)" % note)
            return
        r = kaggle(["datasets", "create", "-p", str(self.dir), "-t", "-q"],
                   allow_fail=True, quiet=True)
        if r.returncode == 0:
            say("  ledger sync-out: dataset created (%s)" % LEDGER_DATASET)
            return
        if r.stdout and r.stdout.strip():
            say("  ledger create note: %s" % r.stdout.strip().splitlines()[0])
        if self._github_fallback(note):
            return
        say("")
        say("!" * 66)
        say("LEDGER SYNC FAILED — the ledger is safe locally at")
        say("  %s" % self.dir)
        say("but it did NOT reach the dataset or GitHub. Rescue it by hand:")
        say("  1. open this run's output files on kaggle.com (they persist), or")
        say("  2. paste the two .jsonl files into the kaggle5-ledger dataset, or")
        say("  3. run `kaggle datasets version -p <dir> -m manual -t` anywhere.")
        say("!" * 66)

    def _github_fallback(self, note: str) -> bool:
        """Commit the ledger to the repo. Only attempted when a token was
        deliberately provided as an env var; never silently skipped — the
        caller announces loudly when this path is used."""
        token = (os.environ.get("K5_GITHUB_TOKEN")
                 or os.environ.get("GITHUB_TOKEN") or "").strip()
        if not token:
            return False
        say("  ledger sync-out: dataset path failed; committing to GitHub "
            "(K5_GITHUB_TOKEN is set)")
        ok = True
        for name in LEDGER_FILES:
            ok = self._github_put(
                "mohamtur1/arc-agi-3-agent", "kaggle5/ledger/" + name,
                self.path(name).read_bytes(), "kaggle5 ledger: " + note[:100],
                token) and ok
        if ok:
            say("  ledger sync-out: committed to GitHub kaggle5/ledger/")
        else:
            say("  ledger sync-out: GitHub commit failed too")
        return ok

    @staticmethod
    def _github_put(repo: str, path: str, content: bytes, message: str,
                    token: str) -> bool:
        import base64
        api = "https://api.github.com/repos/%s/contents/%s" % (repo, path)
        headers = {"Authorization": "Bearer " + token,
                   "Accept": "application/vnd.github+json",
                   "User-Agent": "kaggle5-ledger"}
        try:
            req = urllib.request.Request(api, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as resp:
                sha = json.loads(resp.read().decode()).get("sha")
        except Exception:
            sha = None
        body = {"message": message, "branch": "main",
                "content": base64.b64encode(content).decode()}
        if sha:
            body["sha"] = sha
        try:
            req = urllib.request.Request(
                api, data=json.dumps(body).encode(), method="PUT",
                headers=dict(headers, **{"Content-Type": "application/json"}))
            with urllib.request.urlopen(req, timeout=30) as resp:
                return 200 <= resp.status < 300
        except Exception as exc:
            say("    github put %s failed: %r" % (path, exc))
            return False

    # ---- fold: event log -> current view of the world ---------------------
    def fold(self) -> "View":
        view = View()
        for ev in self.events("experiments.jsonl"):
            kind = ev.get("kind")
            if kind == "import":
                view.imports.append(ev)
            elif kind == "experiment":
                exp = {k: ev.get(k) for k in EXP_FIELDS}
                exp.update(status="pending", score=None, version=None,
                           outcome=None, adopted=None, ref=None, per_game=[])
                view.experiments[ev["exp"]] = exp
                view.order.append(ev["exp"])
            elif kind == "pushed":
                exp = view.experiments.get(ev.get("exp"))
                if exp is not None:
                    exp["version"] = ev.get("version")
            elif kind == "score":
                exp = view.experiments.get(ev.get("exp"))
                if exp is not None:
                    exp.update(status=ev.get("outcome_status", "scored"),
                               score=ev.get("score"), ref=ev.get("ref"),
                               outcome=ev.get("verdict"),
                               adopted=ev.get("adopted"))
                    if ev.get("adopted"):
                        view.incumbent_knobs = dict(exp.get("knobs") or {})
                        view.incumbent_score = ev.get("score")
                        view.anchor_done = True
                    if ev.get("knob") is not None and (
                            ev.get("verdict") in ("kept", "worse", "equal")
                            or ev.get("score") is not None):
                        view.tried.add((canon(ev.get("knob")),
                                        canon(ev.get("value"))))
                    view.matched_refs.add(str(ev.get("ref")))
            elif kind == "gym":
                exp = view.experiments.get(ev.get("exp"))
                if exp is not None and exp["status"] == "pending":
                    exp["status"] = "filtered"
                if ev.get("verdict") == "FAIL":
                    view.tried.add((canon(ev.get("knob")), canon(ev.get("value"))))
            elif kind == "stale":
                exp = view.experiments.get(ev.get("exp"))
                if exp is not None and exp["status"] == "pending":
                    exp["status"] = "stale"
            elif kind == "abandon":
                exp = view.experiments.get(ev.get("exp"))
                if exp is not None and exp["status"] == "pending":
                    exp["status"] = "abandoned"
        for ev in self.events("game_rows.jsonl"):
            exp = view.experiments.get(ev.get("exp"))
            if exp is not None:
                exp["per_game"] = list(ev.get("rows") or [])
                exp["per_game_dummy"] = bool(ev.get("dummy"))
        return view


class View:
    """The folded state. Everything the planner needs, nothing else."""
    def __init__(self):
        self.imports = []
        self.experiments = {}
        self.order = []
        self.incumbent_knobs = dict(DEFAULT_KNOBS)
        self.incumbent_score = None
        self.anchor_done = False
        self.tried = set()          # {(canon(knob), canon(value))} measured-or-filtered
        self.matched_refs = set()

    def open_pending(self):
        for eid in self.order:
            exp = self.experiments[eid]
            if exp["status"] == "pending":
                return exp
        return None

    def next_exp_id(self) -> int:
        n = 0
        for eid in self.order:
            m = re.match(r"exp-(\d+)$", eid)
            if m:
                n = max(n, int(m.group(1)))
        return n + 1

    def table(self) -> str:
        lines = ["  %-8s %-9s %-26s %-24s %-9s %s"
                 % ("exp", "status", "change", "agent", "score", "verdict")]
        for eid in self.order:
            e = self.experiments[eid]
            ch = ", ".join("%s %r->%r" % (k, ov, nv)
                           for k, (ov, nv) in (e.get("changed") or {}).items()) \
                or "(baseline)"
            sc = "-" if e.get("score") is None else "%.4g" % e["score"]
            lines.append("  %-8s %-9s %-26s %-24s %-9s %s"
                         % (eid, e["status"], ch[:26],
                            str(e.get("agent_sha256") or "?")[:12] + "…",
                            sc, e.get("outcome") or ""))
        return "\n".join(lines)


# --------------------------------------------------------------------------
# PIECE TWO — read yesterday's score back
#
# The submission artifact is per-game (submission.parquet: one row per game),
# so an aggregate alone would throw away most of the signal. Two sources:
#   aggregate: `competitions submissions` — the public score per submission.
#   per-game : `kernels output <kernel>/<version>` -> k5_per_game.csv, the
#              telemetry file the v4 agent writes during the rerun.
# Parquet is never parsed here (stdlib-only contract); if only the parquet
# is present we record per-game as unavailable and say so.
# --------------------------------------------------------------------------

MSG_RE_TEMPLATE = r"kaggle5\s+{variant}\s+v(\d+)"
SENTINEL_GAME_IDS = {"1", "0", "dummy", "test", "example", "sample"}


def rows_look_dummy(parsed: list[dict]) -> bool:
    """The interactive (Phase-A) artifact is a hardcoded one-row submission
    so the commit succeeds; real scoring only happens on competition rerun.
    Dummy rows are labelled, never folded into results. Rule: a single row
    is always the dummy; otherwise only file-wide sentinel ids count, so a
    real game that happens to reuse a sentinel-looking id is never thrown
    away by mistake."""
    if len(parsed) <= 1:
        return True
    ids = [str(rw.get("game_id", "")).strip().lower() for rw in parsed]
    return bool(ids) and all(i in SENTINEL_GAME_IDS for i in ids)


def fetch_per_game(exp_id: str, version: int, led: Ledger) -> None:
    """Best-effort per-game telemetry for a scored experiment. Chain of
    custody: only ever read the output of the version that was pushed for
    THIS experiment and then competition-rerun (a completed submission was
    just reconciled against it). A Phase-A (interactive) output contains a
    hardcoded DUMMY submission — the sentinel filter labels such rows and
    they are never folded into results."""
    tmp = Path(tempfile.mkdtemp(prefix="k5_out_"))
    r = kaggle(["kernels", "output", f"{OWNER}/{KERNEL_SLUG}/{version}",
                "-p", str(tmp), "-q"], allow_fail=True, quiet=True)
    csv_path = tmp / "k5_per_game.csv"
    info_path = tmp / "k5_run_info.json"
    if r.returncode != 0 or not csv_path.is_file():
        led.append({"kind": "note", "exp": exp_id,
                    "text": "per-game telemetry unavailable for v%d "
                            "(old agent, or rerun outputs not exposed); "
                            "aggregate score was still recorded" % version})
        say("  per-game: no k5_per_game.csv for v%d (recorded as unavailable)"
            % version)
        return
    rows = []
    for ln in csv_path.read_text().splitlines():
        if ln.startswith("#") or not ln.strip():
            continue
        rows.append(ln)
    if not rows:
        dummy, parsed = True, []
    else:
        try:
            parsed = list(csv.DictReader(rows))
        except csv.Error:
            parsed = []
        dummy = rows_look_dummy(parsed)
    led.append({"kind": "game_rows", "exp": exp_id, "version": version,
                "source": "kernel_output", "dummy": dummy, "rows": parsed},
               name="game_rows.jsonl")
    if dummy:
        say("  per-game: rows for v%d look like the interactive DUMMY — "
            "labelled dummy=true, not folded into results" % version)
    if info_path.is_file():
        try:
            info = json.loads(info_path.read_text())
        except json.JSONDecodeError:
            info = None
        if info:
            led.append({"kind": "note", "exp": exp_id, "text":
                        "run self-report: agent_sha256=%s variant=%r applied=%r"
                        % (str(info.get("agent_sha256"))[:16],
                           info.get("variant"),
                           info.get("applied_overrides"))})


def reconcile(led: Ledger, subs: list[dict], today: str) -> int:
    """Attach new submission outcomes to pending experiments. Returns the
    number of score events written. Matching: (1) the message I ask the
    human to paste (variant+version, strongest), (2) FIFO by date for
    anything unmatched (submission date must be on/after the push date).
    Zero and errored submissions are recorded as data."""
    view = led.fold()
    pend_ids = [eid for eid in view.order
                if view.experiments[eid]["status"] == "pending"]
    fresh = []
    for s in subs:
        ref = str(s.get("ref", ""))
        if ref and ref in view.matched_refs:
            continue
        status = str(s.get("status") or "").lower()
        if status not in ("complete", "error"):
            continue
        fresh.append(s)
    fresh.sort(key=date_of)                      # chronological application

    written = 0
    for s in fresh:
        desc = str(s.get("description") or "")
        ref = str(s.get("ref", ""))
        status = str(s.get("status") or "").lower()
        score = to_float(s.get("publicScore"))
        pick = None
        for eid in pend_ids:
            exp = view.experiments[eid]
            m = re.search(MSG_RE_TEMPLATE.format(variant=re.escape(
                str(exp["variant"]))), desc)
            if m and (exp["version"] is None
                      or int(m.group(1)) == int(exp["version"])):
                pick = eid
                break
        if pick is None:
            for eid in pend_ids:                 # FIFO fallback, date-guarded
                if date_of(s) >= (view.experiments[eid]["date"] or ""):
                    pick = eid
                    break
        if pick is None:
            led.append({"kind": "note", "text":
                        "submission ref=%s (%s, score=%s) matched no pending "
                        "experiment — recorded as a note only"
                        % (ref, desc[:60], s.get("publicScore"))})
            continue
        exp = view.experiments[pick]
        pend_ids.remove(pick)

        inc = view.incumbent_score
        if status == "error":
            outcome_status, verdict, adopted = "error", "worse", False
        else:
            outcome_status = "zero" if score == 0.0 else "scored"
            if inc is None:
                verdict, adopted = "anchor", True
            elif score is None:
                verdict, adopted = "unknown", False
            elif score >= inc - 1e-9:
                verdict = "kept" if score > inc + 1e-9 else "equal"
                adopted = True
            else:
                verdict, adopted = "worse", False

        changed = exp.get("changed") or {}
        knob = next(iter(changed), None)
        ev = {"kind": "score", "exp": pick, "ref": ref,
              "submission_date": date_of(s), "status": status,
              "outcome_status": outcome_status, "score": score,
              "incumbent_score_before": inc, "verdict": verdict,
              "adopted": adopted,
              "knob": knob, "value": (changed.get(knob) or [None, None])[1]
              if knob else None}
        led.append(ev)
        written += 1
        say("  reconciled %s: submission ref=%s status=%s score=%s "
            "(incumbent was %s) -> %s%s"
            % (pick, ref, status,
               ("%.4g" % score) if score is not None else "?",
               ("%.4g" % inc) if inc is not None else "-",
               verdict,
               " — this value is the new incumbent" if adopted else ""))
        if view.experiments[pick]["version"] is not None and status != "error":
            fetch_per_game(pick, int(view.experiments[pick]["version"]), led)
        # keep the local view current for the next match in this batch
        view = led.fold()

    # stale watch: pendings that outlived their welcome
    for eid in list(pend_ids):
        exp = view.experiments[eid]
        age = _days_old(exp["date"], today)
        if age > STALE_AFTER_DAYS:
            led.append({"kind": "stale", "exp": eid,
                        "reason": "pending %d days with no submission "
                                  "(>%d). Tombstoned; the value may be "
                                  "re-planned later." % (age, STALE_AFTER_DAYS)})
            say("  %s is %d days pending with no submission -> stale" % (eid, age))
    return written


# --------------------------------------------------------------------------
# PIECE THREE — pick the next experiment from the ledger
# Fixed queue, one knob at a time, documented backtrack. The choice is a pure
# function of the ledger, and the reason is written into the pending row.
# --------------------------------------------------------------------------

def plan_next(view: View, today: str):
    """Return the plan dict, or None when the queue is exhausted."""
    if not view.anchor_done:
        return {"kind": "anchor", "variant": ANCHOR_VARIANT,
                "changed": {}, "knobs": dict(DEFAULT_KNOBS),
                "reason": "anchor: first measured run of %s; knobs at the "
                          "documented defaults; binds the ledger to the kernel's "
                          "own bytes before any knob moves" % DEVICE}

    for knob, ladder, why in EXPERIMENT_QUEUE:
        dead = False
        for value in ladder:
            key = (canon(knob), canon(value))
            if key in view.tried:
                continue
            if dead:
                continue
            # was any EARLIER entry on this ladder scored worse?
            for eid in view.order:
                e = view.experiments[eid]
                if (e["status"] in ("scored", "zero")
                        and e.get("outcome") == "worse"):
                    ch = e.get("changed") or {}
                    if knob in ch:
                        dead = True
            if dead:
                continue
            old = view.incumbent_knobs.get(knob, DEFAULT_KNOBS.get(knob))
            knobs = dict(view.incumbent_knobs)
            knobs[knob] = value
            if canon(old) == canon(value):
                view.tried.add(key)
                continue
            eid = "exp-%03d" % view.next_exp_id()
            return {"kind": "experiment", "exp": eid, "variant": eid,
                    "changed": {knob: [old, value]}, "knobs": knobs,
                    "reason": "queue: %s | %s" % (knob, why)}
    return None


def block_experiment_event(plan: dict, today: str) -> dict:
    """The pending row. Written BEFORE the push and synced, so a crash can
    never lose what today was supposed to measure."""
    eid = plan.get("exp") or "exp-%03d" % 0
    ev = {"kind": "experiment", "exp": plan.get("exp", eid),
          "variant": plan["variant"], "device": DEVICE, "date": today,
          "changed": plan["changed"], "knobs": plan["knobs"],
          "agent_file": AGENT_FILE, "agent_sha256": AGENT_SHA256,
          "reason": plan["reason"]}
    if plan["kind"] == "anchor":
        ev["exp"] = "exp-000"
        ev["variant"] = ANCHOR_VARIANT
    return ev


def knob_payload_for(plan: dict, today: str) -> dict:
    return {"format": 2, "variant": plan["variant"], "device": DEVICE,
            "date": today, "changed": plan["changed"], "reason": plan["reason"],
            "knobs": plan["knobs"]}


def validate_static_gate(incumbent_knobs: dict, plan: dict) -> tuple[bool, str]:
    """Static gate: RULES ONLY.
    - exactly one knob changed (unless anchor)
    - knob in surface
    - no booleans
    - never uncap (None -> value allowed, value -> None refused)
    - |log2(new/old)| <= 1 for positive numerics
    - canon-equal refused
    THE HARNESS HAS NO LOCAL PROXY (wallet-budget LLM solver, 9h GPU run).
    Never present filter results as predictions or scores.
    """
    variant = plan.get("variant", "")
    changed = plan.get("changed", {})
    new_knobs = plan.get("knobs", {})

    if variant == ANCHOR_VARIANT or plan.get("kind") == "anchor" or not changed:
        if new_knobs != DEFAULT_KNOBS:
            return False, "anchor knobs do not match baseline surface defaults"
        return True, "anchor accepted: full surface at baselines"

    if len(changed) != 1:
        return False, f"gate refused: expected exactly 1 knob changed, got {len(changed)} ({list(changed.keys())})"

    knob, (old_val, new_val) = next(iter(changed.items()))
    if knob not in KNOB_SURFACE:
        return False, f"gate refused: knob {knob!r} not in KNOB_SURFACE {sorted(KNOB_SURFACE.keys())}"

    if isinstance(new_val, bool) or isinstance(old_val, bool):
        return False, f"gate refused: boolean values not permitted on {knob!r} ({new_val!r})"

    if canon(old_val) == canon(new_val):
        return False, f"gate refused: proposed value {new_val!r} is canonically equal to old value {old_val!r}"

    if old_val is not None and new_val is None:
        return False, f"gate refused: uncap attempted on {knob!r} ({old_val!r} -> None is refused)"

    if (isinstance(old_val, (int, float)) and old_val > 0 and
        isinstance(new_val, (int, float)) and new_val > 0):
        ratio = float(new_val) / float(old_val)
        log2_val = abs(math.log2(ratio))
        if log2_val > 1.0 + 1e-9:
            return False, f"gate refused: |log2({new_val}/{old_val})| = {log2_val:.4f} > 1.0"

    return True, f"gate passed: knob {knob} valid ({old_val} -> {new_val})"


# --------------------------------------------------------------------------
# PIECE TWO-B — the local filter. A smoke test in front of the real queue;
# never presented, stored, or read as a leaderboard number.
# --------------------------------------------------------------------------

def gym_json(out: str) -> dict:
    """Extract the gym's summary JSON from stdout. The agent prints its
    '[k5] knob overrides' line(s) too, so try full text, then text without
    those lines, then from the last line that opens a brace."""
    for cand in (out,
                 "\n".join(ln for ln in out.splitlines()
                           if not ln.startswith("[k5]"))):
        try:
            return json.loads(cand)
        except (ValueError, json.JSONDecodeError):
            pass
        if "\n{" in cand:
            try:
                return json.loads(cand[cand.rindex("\n{") + 1:])
            except (ValueError, json.JSONDecodeError):
                pass
    raise ValueError("no JSON object found in gym output")


def run_gym_gate(lever_dir: Path, incumbent_knobs: dict, plan: dict) -> dict:
    """Run k5_gym.py: incumbent vs candidate on five toy games. Returns a
    dict with verdict PASS/FAIL. A crash, a hang, a lost toy win, or a 2+
    toy-level drop FAILs the filter; anything else passes. Filter numbers
    are stored under local_filter in the ledger and are labelled there as
    NOT a score."""
    gym = lever_dir / GYM_FILE
    agent = lever_dir / AGENT_FILE
    if not gym.is_file() or not agent.is_file():
        return {"verdict": "SKIP",
                "verdict_reasons": ["gym or agent file missing next to kaggle5.py"]}
    tmp = Path(tempfile.mkdtemp(prefix="k5_gym_"))
    inc = tmp / "incumbent.json"
    cand = tmp / "candidate.json"
    inc.write_text(json.dumps({"variant": "incumbent",
                               "knobs": incumbent_knobs}, indent=1))
    cand.write_text(json.dumps({"variant": plan["variant"],
                                "knobs": plan["knobs"]}, indent=1))
    cmd = [sys.executable, str(gym), "--agent", str(agent),
           "--base-knobs", str(inc), "--knobs", str(cand), "--json"]
    say("$ " + " ".join(cmd))
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    except subprocess.TimeoutExpired:
        return {"verdict": "FAIL", "verdict_reasons": ["gym ran over 600s (hang?)"]}
    out = r.stdout or ""
    try:
        data = gym_json(out)
    except ValueError:
        blob = out + (r.stderr or "")
        # A missing dependency means the FILTER could not run. That is an
        # environment fact, not evidence about the variant. Conflating the two
        # silences the loop forever on any host without the gym's imports
        # (a clean CI runner, for example), so it gets its own verdict.
        if "ModuleNotFoundError" in blob or "ImportError" in blob:
            missing = sorted(set(
                line.split("No module named", 1)[1].strip().strip("'\"")
                for line in blob.splitlines() if "No module named" in line))
            return {"verdict": "UNAVAILABLE",
                    "verdict_reasons": [
                        "gym could not run on this host: missing %s"
                        % (", ".join(m for m in missing if m) or "a dependency"),
                        blob[-300:]]}
        return {"verdict": "FAIL",
                "verdict_reasons": ["gym output unreadable", blob[-300:]]}
    say("  gym filter (a smoke test, NOT a score): verdict=%s %s"
        % (data.get("verdict"), "; ".join(data.get("verdict_reasons") or [])))
    for tag in ("incumbent", "candidate"):
        blk = data.get(tag) or {}
        say("    %-9s toy-levels=%s toy-wins=%s wall=%ss"
            % (tag, blk.get("levels"), blk.get("wins"), blk.get("wall_seconds")))
    return data


# --------------------------------------------------------------------------
# agent file guard — the pin that keeps source drift loud instead of silent
# --------------------------------------------------------------------------

def guard_agent_file(lever_dir: Path) -> str:
    p = lever_dir / AGENT_FILE
    if not p.is_file():
        die("%s not found next to kaggle5.py (fetch or embed step must place "
            "it there)" % AGENT_FILE)
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    if digest != AGENT_SHA256:
        die(
            "%s sha256 %s… != pinned %s…\n"
            "  The agent changed without a re-pin. That is how the 0.06->0.07\n"
            "  delta became guesswork. If this change is intended: update\n"
            "  AGENT_SHA256 in kaggle5.py CONFIG (and AGENT_FILE if the name\n"
            "  changed), commit, and let tomorrow's run pick it up."
            % (AGENT_FILE, digest[:16], AGENT_SHA256[:16]))
    say("  agent pin OK: %s sha256 %s…" % (AGENT_FILE, digest[:16]))
    return digest


def check_defaults_drift(lever_dir: Path) -> None:
    """DEFAULT_KNOBS must match the baseline solver attributes — the anchor
    run must mean 'no overrides' (franzen-base)."""
    expected = {
        "analyzer_timeout": 900.0,
        "max_runtime_s_per_game": 31920.0,
        "max_actions_per_game": None,
        "concurrency": 120,
    }
    drift = {k: (DEFAULT_KNOBS.get(k), v)
             for k, v in expected.items()
             if canon(DEFAULT_KNOBS.get(k)) != canon(v)}
    if drift:
        die("DEFAULT_KNOBS drifted from baseline solver surface: %r" % drift)
    say("  defaults check OK: DEFAULT_KNOBS == baseline solver attributes")


# --------------------------------------------------------------------------
# stages
# --------------------------------------------------------------------------

VERSION_RE = re.compile(r"Kernel version (\d+) successfully")


def parse_push_version(stdout: str) -> int:
    m = VERSION_RE.search(stdout or "")
    if not m:
        die("no version number in the push output. Look for "
            "'Kernel version N successfully pushed'.")
    return int(m.group(1))


def print_human_card(plan: dict, version: int, today: str) -> None:
    changed = plan.get("changed") or {}
    delta = ", ".join("%s %r -> %r" % (k, ov, nv)
                      for k, (ov, nv) in changed.items()) or "(no knob change — anchor)"
    say("")
    say("=" * 66)
    say("PUSH COMPLETE — THE MACHINE STOPS HERE. IT NEVER SUBMITS.")
    say("=" * 66)
    say("  variant prepared : %s" % plan["variant"])
    say("  knob changed     : %s" % delta)
    say("  why (from ledger): %s" % plan["reason"])
    say("  kernel version   : v%d" % version)
    if plan.get("gym"):
        say("  local filter     : %s (a smoke test, NOT a score)"
            % plan["gym"].get("verdict"))
    say("")
    say("  YOUR THREE TAPS (the daily ~1-slot allowance is yours to spend):")
    say("  1. open  %s" % KERNEL_URL)
    say("  2. wait for version v%d to finish running, then" % version)
    say("     '...' menu -> Submit to %s -> pick version v%d"
        % (COMPETITION, version))
    say("  3. paste this exact message (it is how tomorrow matches the score):")
    say("       kaggle5 %s v%d" % (plan["variant"], version))
    say("")
    say("  Tomorrow's scheduled run reads the result and plans from it.")
    say("  If you skip today, this driver WAITS (no new push) for %d days,"
        % STALE_AFTER_DAYS)
    say("  then marks the row stale and moves on.")
    say("=" * 66)


def do_check(work: Path, patch_path: Path, applier_path: Path,
             inject_at: int | None, lever_dir: Path,
             variant: str = "lever3") -> None:
    probe_cli_flags()
    guard_agent_file(lever_dir)
    check_defaults_drift(lever_dir)
    code, meta = pull(work)
    guard_metadata(meta)
    guard_slug(meta)

    nb = json.loads(code.read_text())
    cells = nb.get("cells", [])
    say("")
    say("--- notebook map: %d cells ---" % len(cells))
    for i, c in enumerate(cells):
        first = cell_text(c).strip().splitlines()[0] if cell_text(c).strip() else ""
        say("  [%3d] %-4s %7dch  %s"
            % (i, "md" if _is_md(c) else "code", len(cell_text(c)), first[:66]))

    say("")
    say("  knob channel present in pulled notebook: %s"
        % ("YES" if has_knob_channel(nb) else
           "NO — ensure section 7 Customization hook heading is present"))

    try:
        at = find_injection_point(nb, inject_at)
    except SystemExit:
        say("no anchor matched — pass --inject-at N using the map above, then re-run check")
        return
    guard_upstream_names(nb, at)

    say("lever cells already present: %s" % (lever_cells_present(nb, LEVER_TAG) or "no"))

    if not patch_path.is_file() or not patch_path.read_text().strip():
        die("%s missing or empty — the lever is the patch, it must exist"
            % patch_path.name)
    applier = applier_path.read_text() if applier_path.is_file() else ""
    if not applier.strip():
        die("%s missing or empty" % applier_path.name)

    demo = {"kind": "experiment", "exp": "exp-000", "variant": "check-dry-run",
            "changed": {"EXPLORATION_CAP": [12, 24]}, "reason": "dry run",
            "knobs": dict(DEFAULT_KNOBS, EXPLORATION_CAP=24)}
    nb2 = inject(nb, patch_path.read_text(), applier, LEVER_TAG, inject_at,
                 knob_payload=knob_payload_for(demo, local_today()))
    nb2 = set_variant(nb2, variant)
    verify_injection(nb2, patch_path.read_text(), LEVER_TAG,
                     knob_payload=knob_payload_for(demo, local_today()))
    where = lever_cells_present(nb2, LEVER_TAG)
    say("dry-run injection: %d cells -> %d cells, lever+knobs at cells %s"
        % (len(cells), len(nb2["cells"]), where))
    say("")
    probe_submission_limits()
    say("")
    say("nothing was pushed. Next: stage=plan")


def do_push(work: Path, patch_path: Path, applier_path: Path,
            inject_at: int | None, allow_same: bool,
            plan: dict | None = None, allow_inert: bool = False,
            variant: str = "lever3") -> int:
    patch_text = patch_path.read_text() if patch_path.is_file() else ""
    applier = applier_path.read_text() if applier_path.is_file() else ""
    if not patch_text.strip():
        die("%s missing or empty" % patch_path.name)
    if not applier.strip():
        die("%s missing or empty" % applier_path.name)

    knob_payload = knob_payload_for(plan, local_today()) if plan else None

    code, meta = pull(work)
    nb = json.loads(code.read_text())
    say("pulled %s: %d cells" % (code.name, len(nb.get("cells", []))))
    guard_metadata(meta)
    guard_slug(meta)

    if plan and plan.get("changed") and not has_knob_channel(nb):
        msg = ("the pulled notebook has no knob channel (marker %r not found "
               "and hook heading not found), so %r would change NOTHING.\n"
               "  Pushing it would spend a run to produce zero information.\n"
               "  Fix: ensure section '## 7. Customization hook' is present in the kernel.\n"
               "  Or pass --allow-inert to push an inert variant anyway."
               % (AGENT_CHANNEL_MARKER, plan["variant"]))
        if allow_inert:
            say("  WARNING --allow-inert: " + msg.replace("\n", "\n  "))
        else:
            die(msg)
    elif plan and plan.get("changed") and allow_inert:
        pass

    before = json.dumps(nb, sort_keys=True)
    nb2 = inject(nb, patch_text, applier, LEVER_TAG, inject_at,
                 knob_payload=knob_payload)
    nb2 = set_variant(nb2, variant)
    verify_injection(nb2, patch_text, LEVER_TAG, knob_payload=knob_payload)

    if before == json.dumps(nb2, sort_keys=True):
        if not allow_same:
            die("the injected notebook is identical to what is already on Kaggle.\n"
                "  A push would create a version with no change. Edit the lever, or pass --allow-same.")
        say("  --allow-same given: pushing an unchanged notebook")

    code.write_text(json.dumps(nb2, indent=1))
    say("  wrote %s (%d cells)" % (code.name, len(nb2["cells"])))

    args = ["kernels", "push", "-p", str(work)]
    if needs_accelerator_flag(meta):
        say("  machine_shape empty on the pulled metadata; passing --accelerator %s"
            % FALLBACK_ACCELERATOR)
        args += ["--accelerator", FALLBACK_ACCELERATOR]
    out = kaggle(args, timeout=600).stdout
    return parse_push_version(out)


def do_wait(max_seconds: int = STATUS_MAX_SECONDS) -> bool:
    say("waiting on %s/%s (poll %ds, max %.1fh)"
        % (OWNER, KERNEL_SLUG, STATUS_POLL_SECONDS, max_seconds / 3600))
    deadline = time.time() + max_seconds
    last = None
    while time.time() < deadline:
        st = kernel_status()
        if st != last:
            say("  [%s] %s" % (time.strftime("%H:%M:%S"), st))
            last = st
        if st in ("complete", "cancelled", "error"):
            if st != "complete":
                die("the run ended as %r — do NOT submit. Read it with:\n"
                    "  kaggle kernels logs %s/%s" % (st, OWNER, KERNEL_SLUG))
            return True
        time.sleep(STATUS_POLL_SECONDS)
    say("still %r after %.1fh" % (last, max_seconds / 3600))
    return False


def refuse_submit(stage: str) -> None:
    say("")
    say("=" * 66)
    say("THE MACHINE NEVER PRESSES THE BUTTON.")
    say("=" * 66)
    say(" stage=%r no longer exists. `competitions submit` is not in this" % stage)
    say(" driver at all: grep the source. Every submission is finalized by a")
    say(" human in the Kaggle app, exactly like the first two.")
    say("")
    say(" To submit today's pushed version: open %s" % KERNEL_URL)
    say(" -> '...' -> Submit to the competition -> choose the version -> go.")
    raise SystemExit(1)


def do_daily(work: Path, patch_path: Path, applier_path: Path,
             inject_at: int | None, lever_dir: Path, skip_gym: bool,
             allow_inert: bool, variant: str = "lever3") -> None:
    """The scheduled loop. Idempotent within a day; safe to interrupt at
    any line; never submits."""
    today = local_today()
    say("=" * 66)
    say("daily run for %s (local, UTC%+d)" % (today, LOCAL_UTC_OFFSET_HOURS))
    say("=" * 66)

    probe_cli_flags()
    agent_sha = guard_agent_file(lever_dir)
    check_defaults_drift(lever_dir)

    led = Ledger(work / "ledger")
    led.sync_in()

    if not any(ev.get("kind") == "import" for ev in led.events()):
        for h in IMPORTED_HISTORY:
            eid = "import-%s" % h["variant"]
            led.append({"kind": "import", "exp": eid, "variant": h["variant"],
                        "score": h["score"], "date": "historical",
                        "note": h["note"]})
        say("  ledger: seeded %d historical anchors (v1=0.06, v2=0.07; "
            "source hashes unrecorded at the time — that is why this file "
            "now exists)" % len(IMPORTED_HISTORY))

    # ---- 1) read yesterday back ----
    say("")
    say("--- 1) reconcile: reading yesterday's submission(s) back ---")
    n = reconcile(led, submissions(), today)
    if n:
        led.sync_out("reconcile %s: %d score row(s)" % (today, n))   # write early
    say("  %d new score row(s) reconciled" % n)
    probe_submission_limits()

    view = led.fold()

    # ---- 2) hold / stale window for an unanswered pending row ----
    pend = view.open_pending()
    if pend is not None:
        st = kernel_status()
        ver = pend.get("version")
        has_sub = kernel_has_submission_file(ver)
        say("  pushed kernel status: %r (version %s), has submission file: %s"
            % (st, ver, has_sub))
        if any(w in st for w in ("running", "queued")) or "running" in st or "queued" in st:
            say("")
            say("--- pending row %s (%s) is still running on GPU (status %r) ---"
                % (pend["exp"], pend["variant"], st))
            say("  pushed as version v%s on %s; hold until it finishes."
                % (ver, pend.get("date")))
            say("  The driver pushes nothing today, on purpose.")
            led.sync_out("daily %s: holding on %s (running)" % (today, pend["exp"]))
            return
        elif not has_sub:
            say("")
            say("  pushed kernel version v%s completed without producing %s (status %r)."
                % (ver, SUBMISSION_FILE, st))
            say("  auto-abandoning %s to unblock re-planning." % pend["exp"])
            led.append({"kind": "abandon", "exp": pend["exp"],
                        "reason": "kernel run completed without %s (status %s)"
                                  % (SUBMISSION_FILE, st)})
            led.sync_out("abandon %s (kernel failed)" % pend["exp"])
            view = led.fold()
        else:
            say("")
            say("--- pending row %s (%s) is ready for manual submission ---"
                % (pend["exp"], pend["variant"]))
            say("  pushed as version v%s on %s; finished with %s."
                % (ver, pend.get("date"), SUBMISSION_FILE))
            say("  Either the human has not submitted (your move — see the card in")
            say("  that day's run log) or scoring is still in flight. The driver")
            say("  pushes nothing today, on purpose.")
            led.sync_out("daily %s: holding on %s (awaiting human submit)"
                         % (today, pend["exp"]))
            return

    # ---- 3) plan ----
    say("")
    say("--- 3) plan ---")
    plan = plan_next(view, today)
    if plan is None:
        say("  experiment queue exhausted. The incumbent IS the answer this")
        say("  system has to give. Incumbent knobs and score:")
        say("    score %.4g  knobs %s"
            % (view.incumbent_score, json.dumps(view.incumbent_knobs,
                                                sort_keys=True)))
        say("")
        say(view.table())
        say("")
        say("  nothing pushed. The loop stops here by design — no silent")
        say("  re-runs, no invented experiments.")
        led.sync_out("daily %s: queue exhausted" % today)
        return
    say("  planned %s: %s" % (plan["variant"],
                              json.dumps(plan["changed"])))
    say("  reason: %s" % plan["reason"])

    # ---- 4) local filter (piece two-b): a smoke test, never a score ----
    if plan["kind"] != "anchor" and not skip_gym:
        say("")
        say("--- 4) local filter (toy games; filters only the obviously-broken) ---")
        gate = run_gym_gate(lever_dir, view.incumbent_knobs, plan)
        plan["gym"] = gate
        knob, val = next(iter(plan["changed"].items()))
        led.append({"kind": "gym", "exp": plan.get("exp", plan["variant"]),
                    "variant": plan["variant"], "knob": knob, "value": val[1],
                    "verdict": gate.get("verdict"),
                    "verdict_reasons": gate.get("verdict_reasons"),
                    "note": "local_filter: toy-gym smoke counters, NOT a "
                            "leaderboard score",
                    "summary": {t: {k: (gate.get(t) or {}).get(k)
                                    for k in ("levels", "wins", "wall_seconds")}
                                for t in ("incumbent", "candidate")}})
        if gate.get("verdict") == "UNAVAILABLE":
            say("  " + "!" * 62)
            say("  FILTER UNAVAILABLE — this says nothing about the variant.")
            say("  reason: " + "; ".join((gate.get("verdict_reasons") or [""])[:1]))
            say("  Proceeding WITHOUT the filter, exactly as --skip-gym would.")
            say("  Install the gym's imports to restore the filter.")
            say("  " + "!" * 62)
        if gate.get("verdict") == "FAIL":
            say("  FILTERED: %s rejected before spending a day: %s"
                % (plan["variant"], "; ".join(gate.get("verdict_reasons") or [])))
            say("  recorded as tried-without-measurement; tomorrow plans the")
            say("  next queue entry. Nothing pushed today.")
            led.sync_out("daily %s: %s filtered" % (today, plan["variant"]))
            return

    # ---- 5) pending row FIRST (durable), then push, then stop ----
    say("")
    say("--- 5) pending row + push + STOP ---")
    ev = block_experiment_event(plan, today)
    led.append(ev)
    led.sync_out("pending %s for %s (pre-push)" % (ev["exp"], today))
    version = do_push(work, patch_path, applier_path, inject_at,
                      allow_same=False, plan=plan, allow_inert=allow_inert,
                      variant=variant)
    led.append({"kind": "pushed", "exp": ev["exp"], "version": version})
    led.sync_out("pushed v%d for %s" % (version, ev["exp"]))
    say("PUSHED version %d of %s/%s" % (version, OWNER, KERNEL_SLUG))
    print_human_card(plan, version, today)


def do_plan(lever_dir: Path, work: Path, today: str) -> None:
    led = Ledger(work / "ledger")
    led.sync_in()
    reconcile(led, submissions(), today)
    view = led.fold()
    say("")
    say(view.table())
    say("")
    plan = plan_next(view, today)
    if plan is None:
        say("queue exhausted; incumbent score %.4g" % (view.incumbent_score or -1))
        return
    say("next experiment: %s  %s" % (plan["variant"], json.dumps(plan["changed"])))
    say("reason: %s" % plan["reason"])
    say("nothing pushed (stage=plan is read-only).")


def do_score_read(lever_dir: Path, work: Path, today: str) -> None:
    led = Ledger(work / "ledger")
    led.sync_in()
    n = reconcile(led, submissions(), today)
    say("reconciled %d new score row(s)" % n)
    view = led.fold()
    say("")
    say(view.table())
    say("")
    say("incumbent score: %s" % (("%.4g" % view.incumbent_score)
                                 if view.incumbent_score is not None else "-"))
    led.sync_out("score read %s" % today)


def parse_set(pairs: list[str]) -> dict:
    out = {}
    for p in pairs:
        if "=" not in p:
            die("--set expects KNOB=JSON_VALUE, got %r" % p)
        k, v = p.split("=", 1)
        k = k.strip()
        if k not in DEFAULT_KNOBS:
            die("--set %s: not a sweepable knob (sweepable: %s)"
                % (k, ", ".join(sorted(DEFAULT_KNOBS))))
        try:
            out[k] = json.loads(v)
        except json.JSONDecodeError as exc:
            die("--set %s: value %r is not JSON (%s)" % (k, v, exc))
    return out


# --------------------------------------------------------------------------
# self test — the whole loop, offline, against _stub_kaggle.py
# --------------------------------------------------------------------------

FIXTURE_HEADINGS = [
    "# About this notebook",
    "## 0. Harness patch",
    "## 1. Environment and submission mode",
    "## 2. Precaching",
    "## 3. Install the ARC runtime",
    "## 4. Import the bundled source",
    "## 5. Start serving",
    "## 6. Load the benchmark",
    "## 7. Customization hook",
    "## 8. Run the system monitor",
]

PATCH = ("diff --git a/ARC3-Inference/inference/agent/prompts.py "
         "b/ARC3-Inference/inference/agent/prompts.py\n"
         "index 52351db..3053e65 100644\n"
         "--- a/ARC3-Inference/inference/agent/prompts.py\n"
         "+++ b/ARC3-Inference/inference/agent/prompts.py\n"
         "@@ -139,7 +139,7 @@\n"
         '-    "- The raw numeric grid is intentionally not exposed.\\n"\n'
         '+    "- `current_frame.grid` is the exact numeric color grid.\\n"\n')

APPLIER = ("import subprocess\n"
           "VARIANT = \"lever3\"  # k5-variant-line\n"
           "subprocess.run(['git', 'apply', '-v', str(PATCH)], "
           "cwd=str(BUNDLE_DIR / 'src'), check=True)\n"
           "print('lever 3 applied')\n")


def _fixture() -> dict:
    cells = [{"cell_type": "markdown", "metadata": {},
              "source": FIXTURE_HEADINGS[0] + "\n\nprose"}]
    for h in FIXTURE_HEADINGS[1:]:
        cells.append({"cell_type": "markdown", "metadata": {},
                      "source": h + "\n\nprose"})
        body = "import os\n"
        if h.startswith("## 1."):
            body = ("import subprocess\n"
                    "WORKING_DIR = Path('/kaggle/working')\n"
                    "BUNDLE_DIR = Path('/kaggle/taaf-kaggle-source-share')\n"
                    "subprocess.run(['git', 'apply', '/kaggle/harness-changes.patch'], check=True)\n")
        elif h.startswith("## 7."):
            body = ("# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.\n"
                    "class _Solver:\n"
                    "    analyzer_timeout = 900.0\n"
                    "    max_runtime_s_per_game = 31920.0\n"
                    "    max_actions_per_game = None\n"
                    "    concurrency = 120\n"
                    "class _BM:\n"
                    "    solver = _Solver()\n"
                    "bm = _BM()\n")
        cells.append({"cell_type": "code", "metadata": {}, "execution_count": None,
                      "outputs": [], "source": body})
    return {"cells": cells, "metadata": {"kaggle": {"accelerator": "nvidiaRtxPro6000"}},
            "nbformat": 4, "nbformat_minor": 4}


def _t(ok: bool, label: str) -> None:
    if not ok:
        raise AssertionError("self-test failed: " + label)
    say("  " + label + " OK")


def _test_injection() -> None:
    say("self-test: injection surgery")
    nb = _fixture()
    expect = FIXTURE_HEADINGS.index("## 2. Precaching") * 2 - 1
    at = find_injection_point(nb, None)
    _t(at == expect, "anchor: insert at %d" % at)

    # lever-only (legacy) mode: exactly 2 cells
    nb2 = inject(nb, PATCH, APPLIER, LEVER_TAG)
    verify_injection(nb2, PATCH, LEVER_TAG)
    _t(len(lever_cells_present(nb2, LEVER_TAG)) == 2
       and len(nb2["cells"]) == len(nb["cells"]) + 2,
       "lever-only injection adds exactly 2 cells")

    # with the knob payload: format 2, hook placement
    payload = {"format": 2, "variant": "exp-001", "device": DEVICE,
               "date": "2026-10-16",
               "changed": {"analyzer_timeout": [900.0, 1200.0]},
               "reason": "t",
               "knobs": dict(DEFAULT_KNOBS, analyzer_timeout=1200.0)}
    nb3 = inject(nb, PATCH, APPLIER, LEVER_TAG, knob_payload=payload)
    verify_injection(nb3, PATCH, LEVER_TAG, knob_payload=payload)

    # variant baking: lever3 is a no-op, kernel rewrites exactly one line
    anchor = lever_anchor(nb3, LEVER_TAG)
    before_src = cell_text(nb3["cells"][anchor])
    _t('VARIANT = "lever3"' in before_src, "applier cell carries the variant line")
    nb_kept = set_variant(json.loads(json.dumps(nb3)), "lever3")
    _t(cell_text(nb_kept["cells"][lever_anchor(nb_kept, LEVER_TAG)]) == before_src,
       "variant=lever3 leaves the applier cell byte-identical")
    nb_kernel = set_variant(json.loads(json.dumps(nb3)), "kernel")
    kernel_src = cell_text(nb_kernel["cells"][lever_anchor(nb_kernel, LEVER_TAG)])
    _t('VARIANT = "kernel"' in kernel_src and 'VARIANT = "lever3"' not in kernel_src,
       "variant=kernel bakes the kernel variant into the applier cell")
    _t(kernel_src.count("k5-variant-line") == 1,
       "variant rewrite does not duplicate the marker")

    # hook placement test
    h_idx = find_hook_heading(nb3)
    k_idx = find_knob_cell_index(nb3)
    _t(h_idx is not None and k_idx is not None and k_idx > h_idx,
       "hook placement: knob cell placed after '## 7. Customization hook'")

    # payload AST round-trip
    import ast
    knob_src = cell_text(nb3["cells"][k_idx])
    parsed_ast = ast.parse(knob_src)
    m = re.search(r"_K5 = (\{.*?\})" + chr(10) + chr(10) + "_K5_SURFACE", knob_src, re.S)
    _t(bool(parsed_ast) and m is not None and json.loads(m.group(1))["format"] == 2,
       "payload AST round-trip")

    # idempotent replace
    _t(json.dumps(inject(nb3, PATCH, APPLIER, LEVER_TAG, knob_payload=payload),
                  sort_keys=True) == json.dumps(nb3, sort_keys=True),
       "re-inject with same payload is a no-op")
    nb4 = inject(nb3, PATCH, APPLIER, LEVER_TAG,
                 knob_payload=dict(payload, variant="exp-002"))
    _t(find_knob_cell_index(nb4) is not None
       and len(nb4["cells"]) == len(nb3["cells"]),
       "re-inject replaces in place, never duplicates")

    # sanitizer on a live-shaped fixture
    nb_live = _fixture()
    nb_live["cells"].insert(4, {"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [], "source": "   " + chr(10) + chr(10)})
    proto_src = ("# Lever 2 — add .grid to the sandbox; prompt tweaks are best" + chr(10) +
                 "import re" + chr(10) +
                 "from inference.agent import prompts, python_tool_sandbox, tool_agent" + chr(10) +
                 "sb = python_tool_sandbox._SANDBOX_BOOTSTRAP" + chr(10))
    nb_live["cells"].insert(12, {"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [], "source": proto_src})
    nb_san = sanitize_notebook(nb_live)
    _t(not any(is_empty_code_cell(c) for c in nb_san["cells"])
       and not any(is_hand_lever_cell(c) for c in nb_san["cells"]),
       "sanitizer on a live-shaped fixture")

    # refusal firing when the hook is renamed
    nb_renamed = _fixture()
    for c in nb_renamed["cells"]:
        if _is_md(c) and "customization hook" in cell_text(c).lower():
            c["source"] = "## 7. Something else"
    _t(not has_knob_channel(nb_renamed), "refusal firing when the hook is renamed")

    nb5 = _fixture()
    for c in nb5["cells"]:
        c["source"] = cell_text(c).replace("BUNDLE_DIR = ", "_unused = ")
    try:
        inject(nb5, PATCH, APPLIER, LEVER_TAG)
    except SystemExit:
        say("  upstream-name guard rejects a section 1 that lost BUNDLE_DIR OK")
    else:
        raise AssertionError("upstream-name guard did not fire")

    _t(parse_push_version("Kernel version 17 successfully pushed.  x") == 17,
       "push version parse")


def _test_ledger_and_planner(tmp: Path) -> None:
    say("self-test: ledger, reconcile, planner")
    led = Ledger(tmp / "led")
    today = "2026-10-16"

    view0 = led.fold()
    plan = plan_next(view0, today)
    _t(plan["kind"] == "anchor" and plan["variant"] == ANCHOR_VARIANT
       and plan["changed"] == {}, "empty ledger plans the anchor first")

    ev0 = block_experiment_event(plan, today)
    _t(set(ev0["knobs"]) == set(DEFAULT_KNOBS) and ev0["device"] == DEVICE
       and ev0["reason"], "pending row carries full knob state + device")

    led.append(ev0)
    led.append({"kind": "pushed", "exp": "exp-000", "version": 5})
    subs = [{"ref": 1001, "date": "2026-10-15 09:00:00",
             "description": "kaggle5 %s v5" % ANCHOR_VARIANT, "status": "complete",
             "publicScore": 0.07}]
    n = reconcile(led, subs, today)
    view1 = led.fold()
    _t(n == 1 and view1.incumbent_score == 0.07 and view1.anchor_done,
       "anchor reconciliation: message-matched, 0.07 becomes incumbent")
    _t(view1.experiments["exp-000"]["status"] == "scored",
       "anchor status scored")

    plan1 = plan_next(view1, today)
    _t(plan1 is not None and plan1["changed"] == {"analyzer_timeout": [900.0, 1200.0]}
       and plan1["exp"] == "exp-001",
       "first queue step: analyzer_timeout 900.0->1200.0")
    _t(plan1["reason"].startswith("queue: analyzer_timeout"),
       "plan reason names the knob and cites the queue")

    led.append(block_experiment_event(plan1, "2026-10-16"))
    led.append({"kind": "pushed", "exp": "exp-001", "version": 6})
    subs1 = [{"ref": 1002, "date": "2026-10-16 09:00:00",
              "description": "kaggle5 exp-001 v6", "status": "complete",
              "publicScore": 0.05}]
    reconcile(led, subs1, "2026-10-16")
    view2 = led.fold()
    _t(view2.incumbent_score == 0.07,
       "worse result (0.05 < 0.07): incumbent unchanged")
    _t((canon("analyzer_timeout"), canon(1200.0)) in view2.tried,
       "tried set records the measured value")

    plan2 = plan_next(view2, "2026-10-17")
    _t(plan2 is not None and plan2["changed"] == {"max_actions_per_game": [None, 500]},
       "backtrack rule: worse kills the rest of analyzer_timeout's ladder; next knob is max_actions_per_game")

    led.append(block_experiment_event(plan2, "2026-10-17"))
    led.append({"kind": "pushed", "exp": "exp-002", "version": 7})
    subs2 = [{"ref": 1003, "date": "2026-10-17 09:00:00",
              "description": "unlabelled submission", "status": "complete",
              "publicScore": 0.07}]
    reconcile(led, subs2, "2026-10-17")
    view3 = led.fold()
    _t(view3.experiments["exp-002"]["outcome"] == "equal"
       and view3.incumbent_knobs["max_actions_per_game"] == 500,
       "FIFO fallback matched the unlabelled submission; equal adopts")
    _t("1003" in view3.matched_refs,
       "matched refs accumulate (no double counting)")
    _t(reconcile(led, subs2, "2026-10-17") == 0,
       "re-reconcile is a no-op (idempotent)")

    plan3 = plan_next(view3, "2026-10-18")
    led.append(block_experiment_event(plan3, "2026-10-18"))
    led.append({"kind": "pushed", "exp": "exp-003", "version": 8})
    subs3 = [{"ref": 1004, "date": "2026-10-18 09:00:00",
              "description": "kaggle5 exp-003 v8", "status": "complete",
              "publicScore": 0.0}]
    reconcile(led, subs3, "2026-10-18")
    view4 = led.fold()
    _t(view4.experiments["exp-003"]["status"] == "zero",
       "a zero is recorded as data, never dropped")

    led.append({"kind": "experiment", "exp": "exp-099", "variant": "exp-099",
                "date": "2026-10-01", "changed": {}, "knobs": dict(DEFAULT_KNOBS),
                "reason": "old"})
    reconcile(led, [], "2026-10-10")
    view5 = led.fold()
    _t(view5.experiments["exp-099"]["status"] == "stale",
       "pending > 2 days -> stale tombstone")

    dummy_rows = [{"game_id": "ls20-dummy", "levels_completed": "1"}]
    _t(rows_look_dummy(dummy_rows),
       "dummy: single row or file-wide sentinels labelled; mixed files kept")

    for knob, ladder, _why in EXPERIMENT_QUEUE:
        for val in ladder:
            view5.tried.add((canon(knob), canon(val)))
    _t(plan_next(view5, "2026-10-19") is None,
       "queue exhausted -> plan None (the loop stops by design)")

    before = (tmp / "led" / "experiments.jsonl").read_text()
    _ = led.fold()
    _t((tmp / "led" / "experiments.jsonl").read_text() == before,
       "reading/folding never rewrites the ledger (append-only)")


def _test_static_gate() -> None:
    say("self-test: static gate edge cases")
    incumbent = dict(DEFAULT_KNOBS)

    # 1. multi-knob changed
    p_multi = {"variant": "exp-bad1", "changed": {"analyzer_timeout": [900.0, 1200.0], "concurrency": [120, 60]}, "knobs": {}}
    ok, msg = validate_static_gate(incumbent, p_multi)
    _t(not ok and "expected exactly 1 knob changed" in msg, "multi-knob plan refused")

    # 2. unknown knob
    p_unk = {"variant": "exp-bad2", "changed": {"unknown_knob": [1, 2]}, "knobs": {}}
    ok, msg = validate_static_gate(incumbent, p_unk)
    _t(not ok and "not in KNOB_SURFACE" in msg, "unknown knob refused")

    # 3. boolean value
    p_bool = {"variant": "exp-bad3", "changed": {"concurrency": [120, True]}, "knobs": {}}
    ok, msg = validate_static_gate(incumbent, p_bool)
    _t(not ok and "boolean" in msg, "boolean value refused")

    # 4. uncap value -> None
    p_uncap = {"variant": "exp-bad4", "changed": {"max_actions_per_game": [500, None]}, "knobs": {}}
    ok, msg = validate_static_gate(incumbent, p_uncap)
    _t(not ok and "uncap" in msg, "uncap (value -> None) refused")

    # 5. uncap None -> value allowed
    p_cap = {"variant": "exp-ok1", "changed": {"max_actions_per_game": [None, 500]}, "knobs": dict(incumbent, max_actions_per_game=500)}
    ok, msg = validate_static_gate(incumbent, p_cap)
    _t(ok, "uncap (None -> value) allowed")

    # 6. |log2| > 1
    p_ratio = {"variant": "exp-bad5", "changed": {"analyzer_timeout": [900.0, 2000.0]}, "knobs": {}}
    ok, msg = validate_static_gate(incumbent, p_ratio)
    _t(not ok and "|log2" in msg, "ratio > 2.0 (|log2| > 1) refused")

    # 7. canon-equal
    p_eq = {"variant": "exp-bad6", "changed": {"analyzer_timeout": [900.0, 900.0]}, "knobs": {}}
    ok, msg = validate_static_gate(incumbent, p_eq)
    _t(not ok and "canonically equal" in msg, "canon-equal change refused")


def _test_gym_verdicts(lever_dir: Path) -> None:
    """A filter that cannot run must never be mistaken for a rejection."""
    say("self-test: gym verdict classification")
    tmp = Path(tempfile.mkdtemp(prefix="k5_gymv_"))
    (tmp / AGENT_FILE).write_text("x = 1\n")   # run_gym_gate needs this name
    plan = {"variant": "t", "knobs": {}, "changed": {"analyzer_timeout": [1.0, 2.0]}}

    gym = tmp / GYM_FILE
    gym.write_text(
        "import sys" + chr(10) +
        "import numpy_that_does_not_exist" + chr(10) +
        "print('{}')" + chr(10))
    v = run_gym_gate(tmp, {}, plan)
    _t(v.get("verdict") == "UNAVAILABLE",
       "a gym that cannot import its deps is UNAVAILABLE, not FAIL")
    _t(any("numpy_that_does_not_exist" in str(r)
           for r in v.get("verdict_reasons") or []),
       "UNAVAILABLE names the missing module")

    gym.write_text(
        "import sys" + chr(10) +
        "sys.stderr.write('Traceback...\nRuntimeError: real crash\n')" +
        chr(10) + "sys.exit(3)" + chr(10))
    v = run_gym_gate(tmp, {}, plan)
    _t(v.get("verdict") == "FAIL",
       "a gym that really crashes is still FAIL (the filter still bites)")

    gym.write_text(
        "import json" + chr(10) +
        "print(json.dumps({'verdict': 'PASS', 'verdict_reasons': [],"
        " 'incumbent': {'levels': 5, 'wins': 3, 'wall_seconds': 1.0},"
        " 'candidate': {'levels': 5, 'wins': 3, 'wall_seconds': 1.0}}))" + chr(10))
    v = run_gym_gate(tmp, {}, plan)
    _t(v.get("verdict") == "PASS", "a healthy gym still passes")
    shutil.rmtree(tmp, ignore_errors=True)


def _test_guards(lever_dir: Path) -> None:
    say("self-test: guards and refusals")
    _src = Path(__file__).read_text()
    _bad1 = "competitions" + chr(34) + ", " + chr(34) + "submit"
    _bad2 = "competitions" + chr(39) + ", " + chr(39) + "submit"
    _t(_bad1 not in _src and _bad2 not in _src,
       "driver source contains no call-shaped `competitions submit` text")
    try:
        refuse_submit("submit")
    except SystemExit as exc:
        _t(exc.code == 1, "--stage submit refuses loudly (exit 1)")
    else:
        raise AssertionError("refuse_submit did not raise")
    try:
        refuse_submit("all")
    except SystemExit:
        say("  --stage all refuses too OK")
    else:
        raise AssertionError("refuse_submit did not raise")

    good = {"competition_sources": EXPECTED_COMPETITION_SOURCES,
            "dataset_sources": EXPECTED_DATASET_SOURCES,
            "model_sources": EXPECTED_MODEL_SOURCES,
            "enable_internet": False, "enable_gpu": False,
            "machine_shape": "nvidiaRtxPro6000"}
    guard_metadata(good)
    say("  metadata guard accepts the real config OK")

    if (lever_dir / AGENT_FILE).is_file():
        guard_agent_file(lever_dir)
        check_defaults_drift(lever_dir)
        say("  agent pin + defaults-drift guards pass on the shipped agent OK")


def _test_stub_end_to_end(lever_dir: Path, tmp: Path) -> None:
    say("self-test: multi-day offline end-to-end against _stub_kaggle.py")
    stub = lever_dir / "_stub_kaggle.py"
    for f in (stub, lever_dir / LEVER_PATCH_FILE, lever_dir / "lever3_apply.py"):
        if not f.is_file():
            say("  stub end-to-end: SKIPPED (%s missing beside kaggle5.py)"
                % f.name)
            return
    env_base = dict(os.environ)
    env_base["KAGGLE_BIN"] = "%s %s" % (sys.executable, stub)
    env_base["STUB_STATE"] = str(tmp / "state.json")
    env_base["STUB_LOG"] = str(tmp / "stub.log")
    env_base["STUB_DATA"] = str(tmp / "stubdata")

    def run(day, stage="daily", extra=()):
        env = dict(env_base, K5_TODAY=day)
        r = subprocess.run(
            [sys.executable, str(Path(__file__).resolve()), "--stage", stage,
             "--work", str(tmp / "work"), *extra],
            env=env, capture_output=True, text=True, timeout=600)
        return r

    def human_submits(ref, day, desc, score, version, per_game=True):
        state = json.loads((tmp / "state.json").read_text())
        state["subs"].insert(0, {
            "ref": ref, "fileName": SUBMISSION_FILE,
            "date": day + " 09:00:00", "description": desc,
            "status": "complete", "publicScore": score, "privateScore": ""})
        if per_game:
            state.setdefault("k5_outputs", {})[str(version)] = {
                "k5_per_game.csv": (
                    "# k5 telemetry v1" + chr(10) +
                    "run_variant,run_agent_sha256,run_knobs_sha256,game_id," +
                    "levels_completed,win_levels,win,actions_used," +
                    "decision_events,final_state" + chr(10) +
                    "%(v)s,abc123,def456,ls20-%(v)s,1,3,0,80,212,NOT_FINISHED" + chr(10) +
                    "%(v)s,abc123,def456,ft09-%(v)s,2,2,1,37,140,WIN" + chr(10)
                    % {"v": desc.split()[1]}),
                "k5_run_info.json": json.dumps(
                    {"variant": desc.split()[1], "device": DEVICE,
                     "applied_overrides": {}}),
            }
        (tmp / "state.json").write_text(json.dumps(state))

    def stub_versions():
        return json.loads((tmp / "state.json").read_text())["version"]

    def ledger_lines():
        return (tmp / "work" / "ledger" / "experiments.jsonl").read_text()

    # ---- day 1: anchor ----
    r = run("2026-10-15")
    _t(r.returncode == 0, "day1 daily exits 0")
    _t('"exp": "exp-000"' in ledger_lines() and '"kind": "pushed"'
       in ledger_lines(), "day1: anchor pending row + pushed row")
    _t("THE MACHINE STOPS HERE" in r.stdout, "day1: human card printed")
    v1 = stub_versions()
    human_submits(1001, "2026-10-15", "kaggle5 %s v%d" % (ANCHOR_VARIANT, v1), 0.07, v1)

    # ---- day 2: reconcile anchor, plan + push exp-001 ----
    r = run("2026-10-16")
    _t(r.returncode == 0, "day2 daily exits 0")
    _t("anchor" in r.stdout and "exp-001" in r.stdout
       and "analyzer_timeout" in r.stdout,
       "day2: anchor reconciled; exp-001 (analyzer_timeout 900.0->1200.0) planned")
    v2 = stub_versions()
    _t(v2 == v1 + 1, "day2: exactly one new push")
    human_submits(1002, "2026-10-16", "kaggle5 exp-001 v%d" % v2, 0.05, v2)

    # ---- day 3: worse -> backtrack rule picks max_actions_per_game ----
    r = run("2026-10-17")
    _t(r.returncode == 0, "day3 daily exits 0")
    _t("worse" in r.stdout and "max_actions_per_game" in r.stdout,
       "day3: 0.05 < 0.07 recorded as data; backtrack plans max_actions_per_game")
    v3 = stub_versions()

    # ---- idempotency: same day again -> hold, no push ----
    r = run("2026-10-17")
    _t(r.returncode == 0 and "pushes nothing today" in r.stdout,
       "day3 second run: holds on the unanswered pending row")
    _t(stub_versions() == v3, "day3 second run: no extra push")

    # ---- per-game rows landed for the reconciled experiments ----
    game_rows = (tmp / "work" / "ledger" / "game_rows.jsonl").read_text()
    _t("ls20-" in game_rows and "ft09-" in game_rows,
       "per-game telemetry rows stored (not just an aggregate)")
    _t('"dummy": false' in game_rows, "real rows labelled dummy=false")

    # ---- the machine never submitted, across all three days ----
    log = (tmp / "stub.log").read_text() if (tmp / "stub.log").is_file() else ""
    _t("competitions submit" not in log, "never-submits: zero submit calls "
                                         "across the whole simulated week")
    _t("kernels push" in log and "datasets version" in log,
       "pushes and ledger syncs did happen")

    # ---- abandoned pending -> re-plan next day ----
    env = dict(env_base, K5_TODAY="2026-10-17")
    r = subprocess.run([sys.executable, str(Path(__file__).resolve()),
                        "--abandon", "--work", str(tmp / "work")],
                       env=env, capture_output=True, text=True, timeout=60)
    _t(r.returncode == 0, "--abandon tombstones the pending row")


def self_test(lever_dir: Path) -> None:
    _test_injection()
    _test_ledger_and_planner(Path(tempfile.mkdtemp(prefix="k5_st_")))
    _test_static_gate()
    _test_gym_verdicts(lever_dir)
    _test_guards(lever_dir)
    _test_stub_end_to_end(lever_dir, Path(tempfile.mkdtemp(prefix="k5_e2e_")))
    say("")
    say("SELF TEST PASSED")



def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--stage", default="check",
                    choices=["check", "plan", "gym", "push", "score", "daily",
                             "submit", "all"])
    ap.add_argument("-v", "--version", type=int, default=0)
    ap.add_argument("--lever-dir", default=None,
                    help="folder holding the patch, applier, agent and gym")
    ap.add_argument("--work", default=None)
    ap.add_argument("--inject-at", type=int, default=None)
    ap.add_argument("--allow-same", action="store_true")
    ap.add_argument("--allow-inert", action="store_true",
                    help="push even when the knob channel is missing (inert variant)")
    ap.add_argument("--skip-gym", action="store_true")
    ap.add_argument("--set", action="append", default=[],
                    metavar="KNOB=JSON", help="ad-hoc override for stage=gym/push")
    ap.add_argument("--variant", default="lever3", choices=["lever3", "kernel"],
                    help="lever3 = memory + hard prune gate (default); "
                         "kernel = memory present, gate off, prompt untouched")
    ap.add_argument("--abandon", action="store_true",
                    help="tombstone the current pending row (human decided "
                         "not to submit it) and exit")
    ap.add_argument("--self-test", action="store_true")
    a = ap.parse_args()

    default_work = ("/kaggle/working/kaggle5" if Path("/kaggle/working").is_dir()
                    else str(Path.cwd() / "kaggle5_work"))
    work = Path(a.work or default_work)
    lever_dir = Path(a.lever_dir) if a.lever_dir else (
        Path(__file__).resolve().parent
        if Path(__file__).resolve().parent.is_dir() else Path.cwd())
    patch_path = lever_dir / LEVER_PATCH_FILE
    applier_path = lever_dir / "lever3_apply.py"

    if a.self_test:
        self_test(lever_dir)
        return

    say("kaggle5  stage=%s  kernel=%s/%s  comp=%s"
        % (a.stage, OWNER, KERNEL_SLUG, COMPETITION))
    check_auth()

    if a.abandon:
        led = Ledger(work / "ledger")
        led.sync_in()
        view = led.fold()
        pend = view.open_pending()
        if pend is None:
            say("no pending row to abandon.")
            return
        led.append({"kind": "abandon", "exp": pend["exp"],
                    "reason": "human flagged --abandon (decided not to submit "
                              "this variant)"})
        led.sync_out("abandon %s" % pend["exp"])
        say("abandoned %s. Tomorrow's run will plan from the ledger." % pend["exp"])
        return

    if a.stage == "check":
        do_check(work, patch_path, applier_path, a.inject_at, lever_dir,
                 variant=a.variant)
    elif a.stage == "plan":
        do_plan(lever_dir, work, local_today())
    elif a.stage == "gym":
        guard_agent_file(lever_dir)
        over = parse_set(a.set)
        knobs = dict(DEFAULT_KNOBS, **over)
        plan = {"variant": "adhoc-gym", "changed":
                {k: [DEFAULT_KNOBS[k], v] for k, v in over.items()},
                "knobs": knobs, "reason": "ad-hoc --set", "exp": "adhoc"}
        gate = run_gym_gate(lever_dir, dict(DEFAULT_KNOBS), plan)
        if gate.get("verdict") in (None, "FAIL"):
            raise SystemExit(1)
    elif a.stage == "push":
        if a.set:
            over = parse_set(a.set)
            knobs = dict(DEFAULT_KNOBS, **over)
            plan = {"kind": "experiment", "variant": "manual-" + local_today(),
                    "changed": {k: [DEFAULT_KNOBS[k], v] for k, v in over.items()},
                    "knobs": knobs, "reason": "manual ad-hoc push via --set",
                    "exp": "manual-" + local_today()}
            led = Ledger(work / "ledger")
            led.sync_in()
            ev = block_experiment_event(plan, local_today())
            led.append(ev)
            led.sync_out("manual pending %s" % ev["exp"])
            ver = do_push(work, patch_path, applier_path, a.inject_at,
                          a.allow_same, plan=plan, allow_inert=a.allow_inert,
                          variant=a.variant)
            led.append({"kind": "pushed", "exp": ev["exp"], "version": ver})
            led.sync_out("manual pushed v%d" % ver)
            print_human_card(plan, ver, local_today())
        else:
            say("note: lever-only repush, OUTSIDE the experiment loop "
                "(no pending row written).")
            ver = do_push(work, patch_path, applier_path, a.inject_at,
                          a.allow_same, variant=a.variant)
            say("PUSHED version %d. The loop is untouched; use --set to push "
                "an ad-hoc experiment instead." % ver)
    elif a.stage == "score":
        do_score_read(lever_dir, work, local_today())
    elif a.stage == "daily":
        do_daily(work, patch_path, applier_path, a.inject_at, lever_dir,
                 skip_gym=a.skip_gym, allow_inert=a.allow_inert,
                 variant=a.variant)
    elif a.stage in ("submit", "all"):
        refuse_submit(a.stage)


if __name__ == "__main__":
    main()
'''.lstrip('\n')
if not t.is_file() or t.stat().st_size < 100:
    t.write_text(EMBEDDED)
    print('EMBEDDED FALLBACK used ->', t.name, t.stat().st_size, 'bytes')
else:
    cur = t.read_text()
    print('using fetched', t.name, t.stat().st_size, 'bytes',
          '(identical to embedded)' if cur == EMBEDDED else
          '(DIFFERS from embedded: the repo fetch is the source of truth)')
print('  sha256', hashlib.sha256(t.read_bytes()).hexdigest()[:16], t.name)


In [ ]:
import hashlib, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
t = D / 'lever2-grid.patch'
EMBEDDED = r'''
diff --git a/ARC3-Inference/README.md b/ARC3-Inference/README.md
index 07542dd..9f95aa3 100644
--- a/ARC3-Inference/README.md
+++ b/ARC3-Inference/README.md
@@ -64,6 +64,9 @@ duck inspects the board, writes small bits of code to reason about it, and calls
 The duck can use:
 
 - `current_frame.ascii` for a compact symbolic grid
+- `current_frame.grid` for the exact numeric color IDs (`grid[row][col]`)
+- `current_frame.memory` for the deterministic episodic state graph:
+  transitions already tried, and the no-op / dead-end / loop pruning gate
 - `current_frame.segmentation` for connected components, object hashes,
   boundaries, containment, and adjacency
 - `history`, `previous_frame`, `transitions`, and `last_transition` for
@@ -72,9 +75,10 @@ The duck can use:
 - `last_action_result` for fields such as `board_changed`, `level_completed`,
   `game_over`, `run_complete`, and `reward`
 
-The raw numeric grid is intentionally hidden from the Python tool. The preferred
-view is `current_frame.segmentation`; `current_frame.ascii` is there for small
-local checks.
+The raw numeric grid is available as `current_frame.grid` for exact cell-level
+checks and state hashing. The preferred view is still
+`current_frame.segmentation`; `current_frame.ascii` is there for small local
+checks, and `current_frame.memory` carries the run's transition history.
 
 The model-facing actions are:
 
diff --git a/ARC3-Inference/inference/agent/episodic_memory.py b/ARC3-Inference/inference/agent/episodic_memory.py
new file mode 100644
index 0000000..7968369
--- /dev/null
+++ b/ARC3-Inference/inference/agent/episodic_memory.py
@@ -0,0 +1,315 @@
+"""Deterministic episodic state memory for the ARC-3 tool agent (Lever-3).
+
+The Python tool sandbox is a *fresh subprocess on every call* (see
+``run_sandboxed_python``), so any object created inside it dies with the call.
+The only durable place for episodic memory is here, on the host, next to
+``tool_runtime_state.json`` -- the one place that observes every real
+``action(...)`` transition and survives across tool calls.
+
+Design rules
+  * deterministic: 64-bit FNV-1a hashes, no randomness, no clocks, no floats
+  * bounded: capped states/edges, oldest evicted first
+  * JSON-serialisable, so it can be persisted to disk and sent to the sandbox
+  * advisory to the model, authoritative at the gate: ``action()`` refuses a
+    known no-op / dead-end / loop edge, unless every valid action would be
+    refused -- the escape hatch that stops a level from ever bricking
+"""
+from __future__ import annotations
+
+import json
+from pathlib import Path
+from typing import Any
+
+MEMORY_FILENAME = "tool_episodic_memory.json"
+MEMORY_VERSION = 1
+
+# Set False to keep the memory view available but advisory (rollback variant).
+EPISODIC_MEMORY_GATE = True
+
+MAX_STATES = 512
+MAX_EDGES_PER_STATE = 12
+MAX_DEAD_ENDS = 512
+SNAPSHOT_MAX_STATES = 256
+SNAPSHOT_MAX_EDGES_PER_STATE = 8
+
+PRUNED_OUTCOMES = ("no_op", "dead_end", "loop")
+
+_FNV_OFFSET = 0xCBF29CE484222325
+_FNV_PRIME = 0x100000001B3
+_MASK64 = 0xFFFFFFFFFFFFFFFF
+_ROW_SALT = 0x2C
+_LEVEL_SALT = 0x9E3779B97F4A7C15
+
+
+def _mix(h: int, byte: int) -> int:
+    h ^= byte & 0xFF
+    return (h * _FNV_PRIME) & _MASK64
+
+
+def grid_hash(grid: Any, level: int = 0) -> int:
+    """Order-sensitive 64-bit FNV-1a over a numeric grid, salted by level.
+
+    Salting with the level keeps an identical board from two different levels
+    (common: a small repeated layout) from sharing one memory node.
+    """
+    h = (_FNV_OFFSET ^ ((int(level) & 0xFFFF) * _LEVEL_SALT)) & _MASK64
+    if not grid:
+        return h
+    for row in grid:
+        for cell in row:
+            try:
+                value = int(cell)
+            except (TypeError, ValueError):
+                value = 0
+            h = _mix(h, value)
+        h = _mix(h, _ROW_SALT)
+    return h
+
+
+def number_text(value: Any) -> str:
+    try:
+        return str(int(value))
+    except (TypeError, ValueError):
+        return str(value).strip()
+
+
+def action_key(action: Any) -> str:
+    """Canonical action identity. Must match ``MemoryView._action_key``."""
+    if isinstance(action, dict):
+        name = str(action.get("action", "") or "").strip().upper()
+        parts = [name or "?"]
+        if action.get("row") is not None:
+            parts.append("R%s" % number_text(action.get("row")))
+        if action.get("col") is not None:
+            parts.append("C%s" % number_text(action.get("col")))
+        return "|".join(parts)
+    return str(action or "").strip().upper()
+
+
+def action_name(action: Any) -> str:
+    """Just the action verb: 'MOUSE|R0|C0' -> 'MOUSE', 'LEFT' -> 'LEFT'."""
+    return action_key(action).split("|", 1)[0]
+
+
+def hash_key(value: Any) -> str:
+    """Normalise a state hash (int or hex string) to its 16-char JSON key."""
+    if isinstance(value, bool):
+        return str(int(value))
+    if isinstance(value, int):
+        return "%016x" % (value & _MASK64)
+    return str(value or "").strip().lower()
+
+
+class CompactStateGraph:
+    """Transition memory: (h(S_t), a_t) -> h(S_t+1), with outcome labels."""
+
+    def __init__(self) -> None:
+        self.states: dict[str, dict[str, dict[str, Any]]] = {}
+        self.dead_ends: list[str] = []
+        self.visits: dict[str, int] = {}
+        self.stats: dict[str, int] = {
+            "no_op": 0,
+            "dead_end": 0,
+            "loop": 0,
+            "progress": 0,
+            "active": 0,
+            "overrides": 0,
+            "pruned_calls": 0,
+        }
+
+    # ------------------------------------------------------------------ write
+    def observe(
+        self,
+        before_frame: Any,
+        action: Any,
+        result: dict[str, Any],
+        after_frame: Any,
+    ) -> str | None:
+        """Record one executed ``action`` and return its outcome label."""
+        if before_frame is None or after_frame is None:
+            return None
+        if not isinstance(result, dict) or not result.get("executed", True):
+            return None
+        name = action_key(action)
+        if not name:
+            return None
+
+        before_key = hash_key(grid_hash(before_frame.grid, before_frame.level))
+        after_key = hash_key(grid_hash(after_frame.grid, after_frame.level))
+
+        edge = self.states.setdefault(before_key, {})
+        prior = edge.get(name)
+        outcome = self._classify(before_key, after_key, prior, result)
+
+        edge[name] = {
+            "next": after_key,
+            "outcome": outcome,
+            "count": int((prior or {}).get("count", 0)) + 1,
+        }
+        while len(edge) > MAX_EDGES_PER_STATE:
+            edge.pop(next(iter(edge)))
+        if outcome == "dead_end" and after_key not in self.dead_ends:
+            self.dead_ends.append(after_key)
+            del self.dead_ends[:-MAX_DEAD_ENDS]
+        self.visits[before_key] = self.visits.get(before_key, 0) + 1
+        self.stats[outcome] = self.stats.get(outcome, 0) + 1
+        self._evict()
+        return outcome
+
+    def _classify(
+        self,
+        before_key: str,
+        after_key: str,
+        prior: dict[str, Any] | None,
+        result: dict[str, Any],
+    ) -> str:
+        if result.get("game_over"):
+            return "dead_end"
+        if before_key == after_key or result.get("board_changed") is False:
+            return "no_op"
+        if result.get("level_completed") or result.get("run_complete"):
+            return "progress"
+        if prior is not None and prior.get("next") == after_key:
+            return "loop"
+        return "active"
+
+    def _evict(self) -> None:
+        while len(self.states) > MAX_STATES:
+            self.states.pop(next(iter(self.states)))
+
+    # ------------------------------------------------------------------- read
+    def tried(self, state_hash: Any) -> list[dict[str, Any]]:
+        entries = self.states.get(hash_key(state_hash), {})
+        return [
+            {
+                "action": name,
+                "next": record.get("next"),
+                "outcome": record.get("outcome"),
+                "count": record.get("count"),
+            }
+            for name, record in entries.items()
+        ]
+
+    def is_pruned(self, state_hash: Any, action: Any) -> bool:
+        key = hash_key(state_hash)
+        record = self.states.get(key, {}).get(action_key(action))
+        if record is None:
+            return False
+        if record.get("outcome") in PRUNED_OUTCOMES:
+            return True
+        if record.get("next") in self.dead_ends:
+            return True
+        return record.get("next") == key
+
+    def safe_actions(self, state_hash: Any, valid_actions: Any) -> list[Any]:
+        return [
+            action
+            for action in (valid_actions or [])
+            if not self.is_pruned(state_hash, action)
+        ]
+
+    def gate(
+        self,
+        before_grid: Any,
+        before_level: int,
+        requested: Any,
+        valid_actions: Any,
+    ) -> dict[str, Any] | None:
+        """Return a refusal describing why, or None to let the action through."""
+        if not EPISODIC_MEMORY_GATE or not valid_actions or not requested:
+            return None
+        key = hash_key(grid_hash(before_grid, before_level))
+        # Police by verb, not by full key: a valid_actions list carries plain
+        # names ("MOUSE"), while a recorded edge may carry coordinates
+        # ("MOUSE|r0|c0"). Name matching keeps coordinate actions in scope.
+        valid_names = {action_name(action) for action in valid_actions}
+        requested_keys = [action_key(action) for action in requested]
+        considered = [name for name in requested_keys if action_name(name) in valid_names]
+        if not considered:
+            return None  # not ours to police: let the environment reject it
+        blocked = [name for name in considered if self.is_pruned(key, name)]
+        if not blocked:
+            return None
+        safe = self.safe_actions(key, valid_actions)
+        if not safe:
+            # Every legal action is pruned. Never brick the level: allow it,
+            # and keep the honest record that the gate stood down.
+            self.stats["overrides"] = self.stats.get("overrides", 0) + 1
+            return None
+        self.stats["pruned_calls"] = self.stats.get("pruned_calls", 0) + 1
+        return {"blocked": blocked, "safe_actions": safe, "tried": self.tried(key)}
+
+    # --------------------------------------------------------------- snapshot
+    def to_payload(self) -> dict[str, Any]:
+        keys = list(self.states.keys())[-SNAPSHOT_MAX_STATES:]
+        states: dict[str, Any] = {}
+        for key in keys:
+            rows = list(self.states[key].items())[-SNAPSHOT_MAX_EDGES_PER_STATE:]
+            states[key] = [
+                [name, record.get("next"), record.get("outcome"), record.get("count", 1)]
+                for name, record in rows
+            ]
+        return {
+            "version": MEMORY_VERSION,
+            "states": states,
+            "dead_ends": list(self.dead_ends[-MAX_DEAD_ENDS:]),
+            "stats": dict(self.stats),
+        }
+
+    @classmethod
+    def from_payload(cls, payload: Any) -> "CompactStateGraph":
+        memory = cls()
+        if not isinstance(payload, dict):
+            return memory
+        for key, rows in (payload.get("states") or {}).items():
+            if not isinstance(rows, (list, tuple)):
+                continue
+            edges: dict[str, dict[str, Any]] = {}
+            for row in rows:
+                if not isinstance(row, (list, tuple)) or len(row) < 3:
+                    continue
+                try:
+                    count = int(row[3])
+                except (IndexError, TypeError, ValueError):
+                    count = 1
+                edges[str(row[0])] = {
+                    "next": str(row[1]),
+                    "outcome": str(row[2]),
+                    "count": count,
+                }
+            if edges:
+                memory.states[str(key)] = edges
+        memory.dead_ends = [str(item) for item in (payload.get("dead_ends") or [])]
+        stats = payload.get("stats")
+        if isinstance(stats, dict):
+            for name, value in stats.items():
+                if isinstance(value, (int, float)) and not isinstance(value, bool):
+                    memory.stats[str(name)] = int(value)
+        memory._evict()
+        return memory
+
+    def save(self, path: Path | None) -> None:
+        if path is None:
+            return
+        try:
+            path.parent.mkdir(parents=True, exist_ok=True)
+            tmp = path.with_suffix(path.suffix + ".tmp")
+            tmp.write_text(json.dumps(self.to_payload(), indent=1), encoding="utf-8")
+            tmp.replace(path)
+        except OSError:
+            pass  # memory is an optimisation, never a crash source
+
+    @classmethod
+    def load(cls, path: Path | None) -> "CompactStateGraph":
+        if path is None or not path.is_file():
+            return cls()
+        try:
+            return cls.from_payload(json.loads(path.read_text(encoding="utf-8")))
+        except (OSError, ValueError):
+            return cls()
+
+    def __len__(self) -> int:
+        return sum(len(edges) for edges in self.states.values())
+
+    def __str__(self) -> str:
+        return "CompactStateGraph(states=%d, edges=%d)" % (len(self.states), len(self))
diff --git a/ARC3-Inference/inference/agent/prompts.py b/ARC3-Inference/inference/agent/prompts.py
index 2e59574..fd749e9 100644
--- a/ARC3-Inference/inference/agent/prompts.py
+++ b/ARC3-Inference/inference/agent/prompts.py
@@ -37,7 +37,8 @@ VISUAL_GAME_ADDENDUM = (
 STRUCTURED_RUNTIME_STATE_ADDENDUM = (
     "\n\nRuntime variables inside every `python` tool call:\n"
     "- `current_frame` is a lightweight frame view for the latest environment state.\n"
-    "- `current_frame` exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\n"
+    "- `current_frame` exposes `.ascii`, `.grid`, `.step`, `.level`, `.shape`, `.segmentation`, and `.memory`.\n"
+    "- `current_frame.grid` is the exact numeric color grid as an immutable tuple of row tuples, indexed `grid[row][col]`. Use it for precise cell/color tests and for `current_frame.memory.hash_grid(...)`.\n"
     "- `current_frame.ascii` is a single newline-delimited string containing the latest board rendered with the letter-coded ARC color symbols.\n"
     "- `current_frame.segmentation` parses the board into objects. It returns `{'nodes': [...], 'adjacency_list': [...]}`.\n"
     "- Each node in `segmentation['nodes']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object's color and shape that ignores its position -- equal hashes mean the same object regardless of where it is, so use it to track an object across frames or to spot multiple identical objects in one frame), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]`), and `children` (ids of objects fully enclosed by this one).\n"
@@ -46,7 +47,7 @@ STRUCTURED_RUNTIME_STATE_ADDENDUM = (
     "- `current_frame.step` is the current environment step count.\n"
     "- `current_frame.level` is the current level number.\n"
     "- `current_frame.shape` is a `(rows, cols)` tuple.\n"
-    "- The raw numeric grid is intentionally not exposed. Use `current_frame.segmentation` as your primary view of the board -- objects, colors, shapes, containment, adjacency, and cross-frame object hashes. Use `current_frame.ascii` only to read a small, specific region; do not scan the whole board with it.\n"
+    "- Use `current_frame.segmentation` as your primary view of the board -- objects, colors, shapes, containment, adjacency, and cross-frame object hashes. Use `current_frame.grid` for exact numeric checks. Use `current_frame.ascii` only to read a small, specific region; do not scan the whole board with it.\n"
 
     "- `history` is a chronological list of action/frame snapshots.\n"
     "- `history` is a Python list of objects, not a dict.\n"
@@ -66,6 +67,19 @@ STRUCTURED_RUNTIME_STATE_ADDENDUM = (
     "- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_result` are refreshed.\n"
 )
 
+EPISODIC_MEMORY_ADDENDUM = (
+    "\n\nEpisodic memory (deterministic, host-side, survives every tool call):\n"
+    "- `current_frame.memory` is a `MemoryView` over every transition this run has actually executed. It is rebuilt from the host after each action, so it persists across `python` calls.\n"
+    "- `current_frame.memory.hash_grid(current_frame.grid)` returns the 64-bit state hash for the current board. The current level is mixed in automatically; pass a level explicitly only if you need another one.\n"
+    "- `current_frame.memory.tried(h)` lists the actions already executed from state `h` as dicts with `action`, `next`, `outcome`, and `count`. Outcomes are `active`, `progress`, `no_op`, `loop`, and `dead_end`.\n"
+    "- `current_frame.memory.is_pruned(h, action)` is `True` when that action is a known `no_op`, `dead_end`, or `loop` from state `h`.\n"
+    "- `current_frame.memory.safe_actions(h, valid_actions)` filters a candidate list down to actions that are not pruned.\n"
+    "- The gate is enforced in code, not prose: `action(...)` refuses a pruned action and returns `stop_reason='k5_pruned'` with the blocked actions, the transitions already recorded from this state, and the safe alternatives. Choose from `safe_actions` and continue in the same tool call.\n"
+    "- If every valid action is pruned, the gate stands down and lets the action through, so a level can never brick.\n"
+    "- Read memory before choosing: `h = current_frame.memory.hash_grid(current_frame.grid); print(current_frame.memory.tried(h))`.\n"
+)
+
+
 MULTIMODAL_CONTEXT_ADDENDUM = (
     "\n\nMultimodal context:\n"
     "- User turns include an attached image of the current ARC grid.\n"
diff --git a/ARC3-Inference/inference/agent/python_tool_sandbox.py b/ARC3-Inference/inference/agent/python_tool_sandbox.py
index ac24747..cf3b6a8 100644
--- a/ARC3-Inference/inference/agent/python_tool_sandbox.py
+++ b/ARC3-Inference/inference/agent/python_tool_sandbox.py
@@ -131,7 +131,16 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
             self.level = level
             self.shape = tuple(shape)
             self._grid = grid
+            self._grid_view = None
             self._segmentation = None
+            self.memory = None
+
+        @property
+        def grid(self):
+            # Immutable numeric colour grid; cached so callers cannot mutate it.
+            if self._grid_view is None:
+                self._grid_view = tuple(tuple(row) for row in self._grid)
+            return self._grid_view
 
         @property
         def segmentation(self):
@@ -146,6 +155,109 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
         __repr__ = __str__
 
 
+    _FNV_OFFSET = 0xCBF29CE484222325
+    _FNV_PRIME = 0x100000001B3
+    _MASK64 = 0xFFFFFFFFFFFFFFFF
+    _ROW_SALT = 0x2C
+    _LEVEL_SALT = 0x9E3779B97F4A7C15
+    _PRUNED_OUTCOMES = ("no_op", "dead_end", "loop")
+
+    def _mix(h, byte):
+        h ^= byte & 0xFF
+        return (h * _FNV_PRIME) & _MASK64
+
+    def _grid_hash(grid, level=0):
+        h = (_FNV_OFFSET ^ ((int(level) & 0xFFFF) * _LEVEL_SALT)) & _MASK64
+        for row in grid or ():
+            for cell in row:
+                try:
+                    value = int(cell)
+                except (TypeError, ValueError):
+                    value = 0
+                h = _mix(h, value)
+            h = _mix(h, _ROW_SALT)
+        return h
+
+    def _num_text(value):
+        try:
+            return str(int(value))
+        except (TypeError, ValueError):
+            return str(value).strip()
+
+    def _action_key(action):
+        if isinstance(action, dict):
+            name = str(action.get("action", "") or "").strip().upper()
+            parts = [name or "?"]
+            if action.get("row") is not None:
+                parts.append("R%s" % _num_text(action.get("row")))
+            if action.get("col") is not None:
+                parts.append("C%s" % _num_text(action.get("col")))
+            return "|".join(parts)
+        return str(action or "").strip().upper()
+
+    def _hash_key(value):
+        if isinstance(value, bool):
+            return str(int(value))
+        if isinstance(value, int):
+            return "%016x" % (value & _MASK64)
+        return str(value or "").strip().lower()
+
+
+    class MemoryView:
+        # Read-only view of the host-side episodic memory for this run.
+        # (No docstring: this block lives inside the sandbox bootstrap text.)
+        def __init__(self, payload, level=0):
+            payload = payload if isinstance(payload, dict) else {}
+            self.version = int(payload.get("version", 0) or 0)
+            self.stats = dict(payload.get("stats") or {})
+            self._level = int(level or 0)
+            self._states = {}
+            for key, rows in (payload.get("states") or {}).items():
+                entries = {}
+                for row in rows or []:
+                    if not isinstance(row, (list, tuple)) or len(row) < 3:
+                        continue
+                    entries[str(row[0])] = (
+                        str(row[1]),
+                        str(row[2]),
+                        int(row[3]) if len(row) > 3 else 1,
+                    )
+                self._states[str(key)] = entries
+            self._dead = set(str(item) for item in (payload.get("dead_ends") or []))
+
+        def hash_grid(self, grid, level=None):
+            return _grid_hash(grid, self._level if level is None else level)
+
+        def tried(self, state_hash):
+            entries = self._states.get(_hash_key(state_hash), {})
+            return [
+                {"action": name, "next": record[0], "outcome": record[1], "count": record[2]}
+                for name, record in entries.items()
+            ]
+
+        def is_pruned(self, state_hash, action):
+            key = _hash_key(state_hash)
+            record = self._states.get(key, {}).get(_action_key(action))
+            if record is None:
+                return False
+            if record[1] in _PRUNED_OUTCOMES:
+                return True
+            if record[0] in self._dead:
+                return True
+            return record[0] == key
+
+        def safe_actions(self, state_hash, valid_actions):
+            return [a for a in (valid_actions or []) if not self.is_pruned(state_hash, a)]
+
+        def __len__(self):
+            return sum(len(entries) for entries in self._states.values())
+
+        def __str__(self):
+            return "MemoryView(states=%d, edges=%d)" % (len(self._states), len(self))
+
+        __repr__ = __str__
+
+
     class HistoryEntryView:
         def __init__(self, *, action, frame):
             self.action = action
@@ -352,6 +464,15 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
             runtime_globals["last_action"] = last_transition.action if last_transition is not None else None
             runtime_globals["valid_actions"] = [str(item) for item in state_payload.get("valid_actions", [])]
             runtime_globals["last_action_result"] = action_result
+            memory_payload = state_payload.get("memory")
+            memory_view = (
+                MemoryView(memory_payload, current_frame.level)
+                if isinstance(memory_payload, dict) and current_frame is not None
+                else None
+            )
+            if current_frame is not None:
+                current_frame.memory = memory_view
+            runtime_globals["memory"] = memory_view
 
         def action(actions):
             normalized_actions = _normalize_actions(actions)
diff --git a/ARC3-Inference/inference/agent/tool_agent.py b/ARC3-Inference/inference/agent/tool_agent.py
index 59655e9..bb3a5a6 100644
--- a/ARC3-Inference/inference/agent/tool_agent.py
+++ b/ARC3-Inference/inference/agent/tool_agent.py
@@ -19,6 +19,7 @@ from inference.agent.prompts import (
     GAME_OVERVIEW_ADDENDUM,
     PYTHON_ADDENDUM,
     STRUCTURED_RUNTIME_STATE_ADDENDUM,
+    EPISODIC_MEMORY_ADDENDUM,
     MULTIMODAL_CONTEXT_ADDENDUM,
     TOOL_CALL_FORMAT_GUIDANCE,
     VISUAL_GAME_ADDENDUM,
@@ -31,6 +32,11 @@ from inference.agent.vision_context import (
 
 from inference.agent.python_tool_sandbox import run_sandboxed_python
 from inference.agent.runtime_state import Frame, HistoryEntry, RUNTIME_STATE_FILENAME, load_runtime_state
+from inference.agent.episodic_memory import (
+    CompactStateGraph,
+    EPISODIC_MEMORY_GATE,
+    MEMORY_FILENAME,
+)
 from inference.utils.openai_compat import build_chat_payload, build_headers
 
 log = logging.getLogger(__name__)
@@ -351,6 +357,7 @@ def _build_system_prompt(*, tool_output_tokens: int) -> str:
     prompt = "You are a coding agent solving a grid-based puzzle game."
     prompt += GAME_OVERVIEW_ADDENDUM
     prompt += STRUCTURED_RUNTIME_STATE_ADDENDUM
+    prompt += EPISODIC_MEMORY_ADDENDUM
     if current_grid_image_enabled():
         prompt += MULTIMODAL_CONTEXT_ADDENDUM
     prompt += VISUAL_GAME_ADDENDUM
@@ -955,6 +962,8 @@ class ToolAgent:
         self._last_step_summary: dict[str, Any] | None = None
         self._last_action_result: dict[str, Any] | None = None
         self._summarized_knowledge = _empty_world_model()
+        self._memory = CompactStateGraph()
+        self._memory_path: Path | None = None
 
     def _headers(self) -> dict[str, str]:
         api_key = (
@@ -982,6 +991,8 @@ class ToolAgent:
             self._last_step_summary = None
             self._last_action_result = None
             self._summarized_knowledge = _empty_world_model()
+            self._memory_path = runtime_dir / MEMORY_FILENAME
+            self._memory = CompactStateGraph.load(self._memory_path)
 
     @property
     def total_tokens(self) -> int:
@@ -1488,6 +1499,7 @@ class ToolAgent:
                     if isinstance(persisted_action_result, dict)
                     else {}
                 ),
+                "memory": self._memory.to_payload(),
             }
 
         terminal_action_result: dict[str, Any] | None = None
@@ -1526,10 +1538,74 @@ class ToolAgent:
                         last_action_result=compact_payload,
                     ),
                 }
+            before_frame = load_runtime_state(state_path)[0]
+            if EPISODIC_MEMORY_GATE and before_frame is not None and valid_actions:
+                refusal_gate = self._memory.gate(
+                    before_frame.grid,
+                    before_frame.level,
+                    normalized_actions,
+                    valid_actions,
+                )
+                if refusal_gate is not None:
+                    blocked_text = ", ".join(str(item) for item in refusal_gate["blocked"])
+                    tried_text = "; ".join(
+                        "%s->%s" % (item["action"], item["outcome"])
+                        for item in refusal_gate["tried"]
+                    ) or "none"
+                    stop_detail = (
+                        "Episodic memory pruned %s: from this exact board state those actions "
+                        "are known no-ops, dead ends, or loops. Safe actions now: %s. "
+                        "Transitions already recorded from this state: %s. "
+                        "Pick from safe_actions and continue in this same tool call."
+                        % (blocked_text, ", ".join(str(a) for a in refusal_gate["safe_actions"]) or "none", tried_text)
+                    )
+                    gate_result = {
+                        "executed": False,
+                        "action_num": (self._last_action_result or {}).get("action_num"),
+                        "level": before_frame.level,
+                        "score": (self._last_action_result or {}).get("score"),
+                        "reward": 0.0,
+                        "state": (self._last_action_result or {}).get("state"),
+                        "valid_actions": list(valid_actions),
+                        "board_changed": False,
+                        "done": False,
+                        "level_completed": False,
+                        "game_over": False,
+                        "run_complete": False,
+                        "requested_count": len(normalized_actions),
+                        "executed_count": 0,
+                        "stopped_early": True,
+                        "stop_reason": "k5_pruned",
+                        "stop_detail": stop_detail,
+                        "k5_pruned": {
+                            "blocked": list(refusal_gate["blocked"]),
+                            "safe_actions": list(refusal_gate["safe_actions"]),
+                            "tried": list(refusal_gate["tried"]),
+                        },
+                    }
+                    self._last_action_result = dict(gate_result)
+                    return {
+                        "action_result": gate_result,
+                        "state": _serialized_runtime_state(
+                            next_valid_actions=valid_actions,
+                            last_action_result=gate_result,
+                        ),
+                    }
+
             raw_payload = self._step_env_callback({"actions": normalized_actions})
             if not isinstance(raw_payload, dict):
                 raise RuntimeError("action(actions) did not return a JSON-like payload.")
             compact_payload = self._compact_action_result(raw_payload)
+            if len(normalized_actions) == 1 and compact_payload.get("executed"):
+                after_frame = load_runtime_state(state_path)[0]
+                if before_frame is not None and after_frame is not None:
+                    self._memory.observe(
+                        before_frame,
+                        normalized_actions[0],
+                        compact_payload,
+                        after_frame,
+                    )
+                    self._memory.save(self._memory_path)
             next_valid_actions = raw_payload.get("valid_actions")
             if isinstance(next_valid_actions, list):
                 self._current_valid_actions = _normalize_valid_actions(next_valid_actions)
'''.lstrip('\n')
if not t.is_file() or t.stat().st_size < 100:
    t.write_text(EMBEDDED)
    print('EMBEDDED FALLBACK used ->', t.name, t.stat().st_size, 'bytes')
else:
    cur = t.read_text()
    print('using fetched', t.name, t.stat().st_size, 'bytes',
          '(identical to embedded)' if cur == EMBEDDED else
          '(DIFFERS from embedded: the repo fetch is the source of truth)')
print('  sha256', hashlib.sha256(t.read_bytes()).hexdigest()[:16], t.name)


In [ ]:
import hashlib, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
t = D / 'lever3_apply.py'
EMBEDDED = r'''
# CELL 4B-2 of 2 — install Lever-3 at runtime.
#
# No upstream file is edited. An earlier revision shipped a byte-pinned
# `git apply` patch; it failed on the live kernel with
# `patch does not apply ... tool_agent.py:19` because the kernel's harness is a
# different revision than the dataset copy the patch was cut against.
#
# This installs Lever-3 by NAME instead:
#   * lever3_runtime.py is written beside this cell by the 4b-1b cell
#   * install() either patches the already-imported harness, or arms a
#     meta-path watcher that patches it the moment it is first imported
#   * if a name it needs is missing, it says so and patches nothing
#   * memory is keyed by the frame's ASCII text, so no `.grid` is required
#
# VARIANT
#   lever3 : memory + hard prune gate + prompt directives  (default)
#   kernel : memory view present, gate off, prompt untouched (A/B rollback)
import importlib
import os
import sys
import traceback
from pathlib import Path

VARIANT = "lever3"  # k5-variant-line

# ------------------------------------------------------------------- shim
# The GPU kernel receives exactly two cells: the patch writer and this one.
# It has no internet, so it cannot fetch lever3_runtime.py from the repo.
# The shim therefore travels inside this cell, base64-encoded (it contains
# triple quotes, so it cannot be embedded as a plain literal).
SHIM_SHA256 = "ceaa28567b98ea821d7746cf5cd2132e6f030c5b7667ce1108d8944f69583def"  # k5-shim-sha
SHIM_B64 = "..."  # k5-shim-b64


def _materialize_shim(target_dir):
    import base64
    import hashlib
    raw = base64.b64decode(SHIM_B64)
    digest = hashlib.sha256(raw).hexdigest()
    if digest != SHIM_SHA256:
        raise RuntimeError(
            "lever 3 FAILED: embedded shim is corrupt (sha256 %s != %s). "
            "Rebuild the applier with kaggle5/sync_shim.py." % (digest, SHIM_SHA256))
    target_dir.mkdir(parents=True, exist_ok=True)
    dest = target_dir / "lever3_runtime.py"
    dest.write_bytes(raw)
    print("lever 3: wrote lever3_runtime.py (%d bytes, sha256 %s)"
          % (len(raw), digest[:16]))
    return dest


# ------------------------------------------------------------------ locate
def _first_existing(paths):
    for p in paths:
        try:
            if p and Path(p).is_dir():
                return Path(p)
        except OSError:
            continue
    return None


WORKING_DIR = _first_existing([
    globals().get("WORKING_DIR"),
    os.environ.get("WORKING_DIR"),
    "/kaggle/working",
    str(Path.cwd()),
]) or Path.cwd()

# lever3_runtime.py is written next to the driver by the cell above
CANDIDATES = [
    globals().get("K5_DIR"),
    os.environ.get("K5_DIR"),
    WORKING_DIR / "kaggle5",
    Path.cwd(),
    Path(__file__).resolve().parent if "__file__" in globals() else None,
]
# Prefer a directory that already has the shim; otherwise write it somewhere
# writable. Never depend on a previous cell having run.
K5_DIR = None
for cand in CANDIDATES:
    if cand and (Path(cand) / "lever3_runtime.py").is_file():
        K5_DIR = Path(cand)
        break
if K5_DIR is None:
    for cand in CANDIDATES:
        if not cand:
            continue
        try:
            Path(cand).mkdir(parents=True, exist_ok=True)
            K5_DIR = Path(cand)
            break
        except OSError:
            continue
if K5_DIR is None:
    import tempfile
    K5_DIR = Path(tempfile.mkdtemp(prefix="k5_shim_"))
_materialize_shim(K5_DIR)

if str(K5_DIR) not in sys.path:
    sys.path.insert(0, str(K5_DIR))

# ------------------------------------------------------------------ install
try:
    import lever3_runtime
    importlib.reload(lever3_runtime)          # pick up a re-pushed shim
except Exception:
    print("lever 3 FAILED to import lever3_runtime:")
    traceback.print_exc()
    raise

runtime = lever3_runtime.install(gate=(VARIANT == "lever3"))
for event in runtime.events:
    print("lever 3: %s" % event)

# ------------------------------------------------------------- verification
# Importing the harness now is not a side effect to avoid: if it imports, the
# watcher patches it immediately and we can prove Lever-3 is live before the
# games start. If it cannot import yet, the watcher stays armed.
try:
    ta = importlib.import_module("inference.agent.tool_agent")
    runtime = lever3_runtime._RUNTIME
    for event in runtime.events[len(runtime.events) - 1:]:
        print("lever 3: %s" % event)
except Exception as exc:
    print("lever 3: harness not importable yet (%s: %s)" % (type(exc).__name__, exc))
    print("lever 3: armed — it will be patched when first imported")

if not runtime.installed:
    print("lever 3: ARMED — the harness is not imported yet, which is normal")
    print("lever 3: here. The patch fires when the harness first loads and")
    print("lever 3: prints '[k5] lever 3: PATCHED ... -- episodic memory is live'.")
    print("lever 3: Search this log for that line. If it never appears and the")
    print("lever 3: games run, Lever-3 was NOT in effect.")
else:
    missing = []
    for hook, check in (
        ("_run_python_tool", lambda m: hasattr(m.ToolAgent, "_run_python_tool")),
        ("run_sandboxed_python",
         lambda m: "run_sandboxed_python" in getattr(m, "__dict__", {})),
    ):
        try:
            if not check(ta if "ta" in dir() else sys.modules["inference.agent.tool_agent"]):
                missing.append(hook)
        except Exception:
            missing.append(hook)
    print("lever 3: variant=%s gate=%s" % (VARIANT, runtime.gate))
    if missing:
        print("lever 3: WARNING hooks not confirmed: %s" % ", ".join(missing))
    else:
        print("lever 3: LIVE (memory + gate installed on %s)" % runtime.patched_module)
print("lever 3: done")
'''.lstrip('\n')
if not t.is_file() or t.stat().st_size < 100:
    t.write_text(EMBEDDED)
    print('EMBEDDED FALLBACK used ->', t.name, t.stat().st_size, 'bytes')
else:
    cur = t.read_text()
    print('using fetched', t.name, t.stat().st_size, 'bytes',
          '(identical to embedded)' if cur == EMBEDDED else
          '(DIFFERS from embedded: the repo fetch is the source of truth)')
print('  sha256', hashlib.sha256(t.read_bytes()).hexdigest()[:16], t.name)


In [ ]:
import hashlib, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
t = D / 'lever3_runtime.py'
EMBEDDED = r'''
"""Lever-3 at runtime: deterministic episodic memory + a hard prune gate.

Why this shape
--------------
An earlier version shipped a byte-pinned `git apply` patch. It broke twice:
once because the bundle path was wrong, and once because the kernel's harness
is a different revision than the dataset copy the patch was cut against
(`patch does not apply` at tool_agent.py:19). Byte-pinning assumes you can see
the file you are patching; on this kernel we cannot.

This module installs Lever-3 by *name*, at import time:

  * no upstream file is edited, so there is nothing to drift
  * it works whether or not the harness was already imported
  * if a name it needs is missing, it says so loudly and does nothing else

Three hooks, all verified before use:

  1. ToolAgent._run_python_tool  -- tells us which game directory is live, so
     memory is per-game.
  2. tool_agent.run_sandboxed_python -- wraps the action handler, which is the
     only place that sees every real action and both frames around it.
     This is also where the gate lives: the refusal happens host-side, so a
     small model cannot talk its way past it.
  3. tool_agent._build_system_prompt -- appends the memory instructions.

State identity is the frame's ASCII rendering, not `.grid`: ASCII is present in
every revision of the payload, and hashing it makes memory work even where
`.grid` does not exist.
"""
from __future__ import annotations

import importlib
import importlib.abc
import importlib.machinery
import json
import sys
import traceback
from pathlib import Path

TARGET_MODULE = "inference.agent.tool_agent"
MEMORY_FILENAME = "tool_episodic_memory.json"

MAX_STATES = 512
MAX_EDGES_PER_STATE = 12
MAX_DEAD_ENDS = 512
SNAPSHOT_STATES = 256
SNAPSHOT_EDGES = 8
PRUNED = ("no_op", "dead_end", "loop")

_MASK64 = 0xFFFFFFFFFFFFFFFF
_FNV_OFFSET = 0xCBF29CE484222325
_FNV_PRIME = 0x100000001B3
_LEVEL_SALT = 0x9E3779B97F4A7C15

ADDENDUM = (
    "\n\nEpisodic memory (deterministic, host-side, survives every tool call):\n"
    "- `current_frame.memory.hash_grid(current_frame.ascii)` returns the 64-bit "
    "state hash for this board. Hash the ASCII, not a numeric grid.\n"
    "- `current_frame.memory.tried(h)` lists actions already executed from that "
    "state: dicts with `action`, `next`, `outcome`, `count`. Outcomes are "
    "`active`, `progress`, `no_op`, `loop`, `dead_end`.\n"
    "- `current_frame.memory.is_pruned(h, action)` is True when that action is a "
    "known no-op, loop, or dead end from that state.\n"
    "- `current_frame.memory.safe_actions(h, valid_actions)` filters a candidate "
    "list down to actions that are not pruned.\n"
    "- This is enforced in code, not prose: `action(...)` refuses a pruned action "
    "and returns `stop_reason='k5_pruned'` with the blocked actions, the "
    "transitions already recorded from this state, and the safe alternatives. "
    "Pick from `safe_actions` and continue in the same tool call.\n"
    "- If every valid action is pruned the gate stands down and lets the action "
    "through, so a level can never brick.\n"
    "- Read it before choosing: `h = current_frame.memory.hash_grid("
    "current_frame.ascii); print(current_frame.memory.tried(h))`.\n"
)


# --------------------------------------------------------------- state hash
def _ascii_text(frame_payload) -> str:
    if isinstance(frame_payload, dict):
        return str(frame_payload.get("ascii") or "")
    return ""


def _level_of(frame_payload) -> int:
    if isinstance(frame_payload, dict):
        try:
            return max(1, int(frame_payload.get("level") or 1))
        except (TypeError, ValueError):
            return 1
    return 1


def state_hash(ascii_text: str, level: int = 1) -> str:
    h = (_FNV_OFFSET ^ ((int(level) & 0xFFFF) * _LEVEL_SALT)) & _MASK64
    for byte in str(ascii_text).encode("utf-8", "replace"):
        h ^= byte
        h = (h * _FNV_PRIME) & _MASK64
    return "%016x" % h


def _num(value) -> str:
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value).strip()


def action_key(action) -> str:
    if isinstance(action, dict):
        name = str(action.get("action") or "").strip().upper()
        parts = [name or "?"]
        if action.get("row") is not None:
            parts.append("R%s" % _num(action.get("row")))
        if action.get("col") is not None:
            parts.append("C%s" % _num(action.get("col")))
        return "|".join(parts)
    return str(action or "").strip().upper()


def action_name(action) -> str:
    return action_key(action).split("|", 1)[0]


# ------------------------------------------------------------------- memory
class CompactStateGraph:
    def __init__(self) -> None:
        self.states: dict[str, dict[str, dict]] = {}
        self.dead_ends: list[str] = []
        self.stats = {k: 0 for k in
                      ("active", "progress", "no_op", "loop", "dead_end",
                       "overrides", "pruned_calls")}

    def observe(self, before: str, level: int, action, result: dict,
                after: str) -> str | None:
        if not before or not after or not isinstance(result, dict):
            return None
        if result.get("executed") is False:
            return None
        name = action_key(action)
        if not name:
            return None
        b = state_hash(before, level)
        a = state_hash(after, level)
        edge = self.states.setdefault(b, {})
        prior = edge.get(name)
        if result.get("game_over"):
            outcome = "dead_end"
        elif b == a or result.get("board_changed") is False:
            outcome = "no_op"
        elif result.get("level_completed") or result.get("run_complete"):
            outcome = "progress"
        elif prior is not None and prior.get("next") == a:
            outcome = "loop"
        else:
            outcome = "active"
        edge[name] = {"next": a, "outcome": outcome,
                      "count": int((prior or {}).get("count", 0)) + 1}
        while len(edge) > MAX_EDGES_PER_STATE:
            edge.pop(next(iter(edge)))
        if outcome == "dead_end" and a not in self.dead_ends:
            self.dead_ends.append(a)
            del self.dead_ends[:-MAX_DEAD_ENDS]
        self.stats[outcome] = self.stats.get(outcome, 0) + 1
        while len(self.states) > MAX_STATES:
            self.states.pop(next(iter(self.states)))
        return outcome

    def tried(self, h) -> list[dict]:
        return [{"action": k, "next": v.get("next"),
                 "outcome": v.get("outcome"), "count": v.get("count")}
                for k, v in self.states.get(str(h), {}).items()]

    def is_pruned(self, h, action) -> bool:
        key = str(h)
        rec = self.states.get(key, {}).get(action_key(action))
        if rec is None:
            return False
        return (rec.get("outcome") in PRUNED
                or rec.get("next") in self.dead_ends
                or rec.get("next") == key)

    def safe_actions(self, h, valid) -> list:
        return [a for a in (valid or []) if not self.is_pruned(h, a)]

    def gate(self, before: str, level: int, requested, valid):
        if not valid or not requested:
            return None
        h = state_hash(before, level)
        valid_names = {action_name(a) for a in valid}
        considered = [action_key(a) for a in requested
                      if action_name(a) in valid_names]
        if not considered:
            return None
        blocked = [a for a in considered if self.is_pruned(h, a)]
        if not blocked:
            return None
        safe = self.safe_actions(h, valid)
        if not safe:
            self.stats["overrides"] += 1
            return None
        self.stats["pruned_calls"] += 1
        return {"blocked": blocked, "safe_actions": safe, "tried": self.tried(h)}

    def to_payload(self) -> dict:
        keys = list(self.states)[-SNAPSHOT_STATES:]
        return {
            "version": 1,
            "states": {k: [[n, v.get("next"), v.get("outcome"),
                            int(v.get("count", 1))]
                           for n, v in list(self.states[k].items())[-SNAPSHOT_EDGES:]]
                       for k in keys},
            "dead_ends": list(self.dead_ends[-MAX_DEAD_ENDS:]),
            "stats": dict(self.stats),
        }

    @classmethod
    def from_payload(cls, payload) -> "CompactStateGraph":
        mem = cls()
        if not isinstance(payload, dict):
            return mem
        for key, rows in (payload.get("states") or {}).items():
            edges = {}
            for row in rows or []:
                if not isinstance(row, (list, tuple)) or len(row) < 3:
                    continue
                try:
                    count = int(row[3])
                except (IndexError, TypeError, ValueError):
                    count = 1
                edges[str(row[0])] = {"next": str(row[1]),
                                      "outcome": str(row[2]), "count": count}
            if edges:
                mem.states[str(key)] = edges
        mem.dead_ends = [str(x) for x in (payload.get("dead_ends") or [])]
        for k, v in (payload.get("stats") or {}).items():
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                mem.stats[str(k)] = int(v)
        return mem

    def save(self, path: Path | None) -> None:
        if path is None:
            return
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            tmp = path.with_suffix(path.suffix + ".tmp")
            tmp.write_text(json.dumps(self.to_payload(), indent=1),
                           encoding="utf-8")
            tmp.replace(path)
        except OSError:
            pass

    @classmethod
    def load(cls, path: Path | None) -> "CompactStateGraph":
        if path is None or not path.is_file():
            return cls()
        try:
            return cls.from_payload(json.loads(path.read_text(encoding="utf-8")))
        except (OSError, ValueError):
            return cls()

    def __len__(self) -> int:
        return sum(len(e) for e in self.states.values())


# --------------------------------------------- sandbox-side view (a preamble)
_PREAMBLE = r"""
# The sandbox strips __build_class__ from its builtins, so `class` would raise
# NameError here. Functions are fine; the view is assembled with type().
def _k5_fnv(self, text, level=None):
    lv = self._level if level is None else int(level)
    h = (0xcbf29ce484222325 ^ ((lv & 0xFFFF) * 0x9E3779B97F4A7C15)) & 0xFFFFFFFFFFFFFFFF
    for b in str(text).encode("utf-8", "replace"):
        h ^= b
        h = (h * 0x100000001b3) & 0xFFFFFFFFFFFFFFFF
    return "%016x" % h


def _k5_key(self, action):
    if isinstance(action, dict):
        parts = [str(action.get("action") or "").strip().upper() or "?"]
        if action.get("row") is not None:
            parts.append("R" + str(action.get("row")).strip())
        if action.get("col") is not None:
            parts.append("C" + str(action.get("col")).strip())
        return "|".join(parts)
    return str(action or "").strip().upper()


def _k5_init(self, payload, level):
    payload = payload if isinstance(payload, dict) else {}
    self._level = int(level or 1)
    self._states = {}
    for key, rows in (payload.get("states") or {}).items():
        for row in rows or []:
            if isinstance(row, (list, tuple)) and len(row) >= 3:
                self._states.setdefault(str(key), {})[str(row[0])] = (
                    str(row[1]), str(row[2]),
                    int(row[3]) if len(row) > 3 else 1)
    self._dead = set(str(x) for x in (payload.get("dead_ends") or []))


def _k5_tried(self, h):
    out = []
    for k, v in self._states.get(str(h), {}).items():
        out.append({"action": k, "next": v[0], "outcome": v[1], "count": v[2]})
    return out


def _k5_is_pruned(self, h, action):
    key = str(h)
    rec = self._states.get(key, {}).get(self._key(action))
    if rec is None:
        return False
    return (rec[1] in ("no_op", "dead_end", "loop")
            or rec[0] in self._dead or rec[0] == key)


def _k5_safe(self, h, valid_actions):
    return [a for a in (valid_actions or []) if not self.is_pruned(h, a)]


def _k5_len(self):
    n = 0
    for v in self._states.values():
        n += len(v)
    return n


def _k5_repr(self):
    return "MemoryView(states=%d, edges=%d)" % (len(self._states), len(self))


_K5View = type("_K5MemoryView", (), {
    "__init__": _k5_init,
    "_key": _k5_key,
    "hash_grid": _k5_fnv,
    "tried": _k5_tried,
    "is_pruned": _k5_is_pruned,
    "safe_actions": _k5_safe,
    "__len__": _k5_len,
    "__repr__": _k5_repr,
})


def _k5_attach():
    global memory
    _view = _K5View(_K5_MEMORY, current_frame.level if current_frame is not None else 1)
    if current_frame is not None:
        current_frame.memory = _view
    memory = _view


_k5_attach()

_k5_original_action = action


def _k5_action(actions):
    _r = _k5_original_action(actions)
    _k5_attach()
    return _r


action = _k5_action
"""


# ------------------------------------------------------------ runtime patch
class _Runtime:
    def __init__(self, gate: bool) -> None:
        self.gate = bool(gate)
        self.memories: dict[str, CompactStateGraph] = {}
        self.state_path = None
        self.installed = False
        self.patched_module = None
        self.events: list[str] = []

    # memory is per game directory
    def memory_for(self, state_path) -> tuple[CompactStateGraph, Path | None]:
        if state_path is None:
            return CompactStateGraph(), None
        d = Path(state_path).parent
        key = str(d)
        sidecar = d / MEMORY_FILENAME
        if key not in self.memories:
            self.memories[key] = CompactStateGraph.load(sidecar)
        return self.memories[key], sidecar


_RUNTIME: _Runtime | None = None


def _patch_module(ta, runtime: _Runtime) -> None:
    missing = [n for n in ("ToolAgent", "run_sandboxed_python")
               if not hasattr(ta, n)]
    if missing:
        runtime.events.append("NOT PATCHED: %s is missing %s"
                              % (ta.__name__, ", ".join(missing)))
        return

    # 1) which game directory is live
    original_run = ta.ToolAgent._run_python_tool

    def _run_python_tool(self, state_path, arguments):
        previous = runtime.state_path
        runtime.state_path = state_path
        try:
            return original_run(self, state_path, arguments)
        finally:
            runtime.state_path = previous

    ta.ToolAgent._run_python_tool = _run_python_tool

    # 2) the gate + the recorder, around the sandbox call
    original_sandbox = ta.run_sandboxed_python

    def run_sandboxed_python(*, code, timeout_seconds, initial_state,
                             action_handler):
        memory, sidecar = runtime.memory_for(runtime.state_path)
        payload = memory.to_payload()
        state = dict(initial_state or {})
        state["memory"] = payload
        level = _level_of(state.get("current_frame"))
        before = _ascii_text(state.get("current_frame"))
        valid = list(state.get("valid_actions") or [])
        preamble = ("_K5_MEMORY = %s\n" % json.dumps(payload)) + _PREAMBLE

        def handler(actions):
            nonlocal before, valid, level, state
            if runtime.gate and before and valid:
                refusal = memory.gate(before, level, actions, valid)
                if refusal is not None:
                    blocked = ", ".join(str(b) for b in refusal["blocked"])
                    detail = (
                        "Episodic memory pruned %s: from this exact board those "
                        "actions are known no-ops, dead ends, or loops. Safe "
                        "actions now: %s. Already recorded here: %s. Pick from "
                        "safe_actions and continue in this same tool call."
                        % (blocked,
                           ", ".join(str(a) for a in refusal["safe_actions"]) or "none",
                           "; ".join("%s->%s" % (t["action"], t["outcome"])
                                     for t in refusal["tried"]) or "none"))
                    result = {
                        "executed": False,
                        "action_num": (state.get("last_action_result") or {}).get("action_num"),
                        "level": level, "score": None, "reward": 0.0,
                        "state": None, "valid_actions": list(valid),
                        "board_changed": False, "done": False,
                        "level_completed": False, "game_over": False,
                        "run_complete": False,
                        "requested_count": len(actions or []),
                        "executed_count": 0, "stopped_early": True,
                        "stop_reason": "k5_pruned", "stop_detail": detail,
                        "k5_pruned": {"blocked": list(refusal["blocked"]),
                                      "safe_actions": list(refusal["safe_actions"]),
                                      "tried": list(refusal["tried"])},
                    }
                    state["last_action_result"] = result
                    state["memory"] = memory.to_payload()
                    return {"action_result": result, "state": state}

            out = action_handler(actions)
            if not isinstance(out, dict):
                return out
            result = out.get("action_result") or {}
            new_state = out.get("state")
            after = _ascii_text((new_state or {}).get("current_frame"))
            if (len(actions or []) == 1 and before and after
                    and isinstance(result, dict)
                    and result.get("executed", True) is not False):
                memory.observe(before, level, actions[0], result, after)
                memory.save(sidecar)
            if after:
                before = after
                level = _level_of((new_state or {}).get("current_frame"))
            if isinstance(new_state, dict):
                if isinstance(new_state.get("valid_actions"), list):
                    valid = list(new_state["valid_actions"])
                new_state["memory"] = memory.to_payload()
                state = new_state
            return out

        return original_sandbox(code=preamble + code,
                                timeout_seconds=timeout_seconds,
                                initial_state=state,
                                action_handler=handler)

    ta.run_sandboxed_python = run_sandboxed_python

    # 3) the prompt
    if hasattr(ta, "_build_system_prompt"):
        original_build = ta._build_system_prompt

        def _build_system_prompt(*args, **kwargs):
            return original_build(*args, **kwargs) + ADDENDUM

        ta._build_system_prompt = _build_system_prompt
    elif hasattr(ta, "STRUCTURED_RUNTIME_STATE_ADDENDUM"):
        ta.STRUCTURED_RUNTIME_STATE_ADDENDUM = (
            ta.STRUCTURED_RUNTIME_STATE_ADDENDUM + ADDENDUM)
    else:
        runtime.events.append("prompt addendum skipped: no known prompt hook")

    runtime.patched_module = ta.__name__
    runtime.installed = True
    runtime.events.append("patched %s (gate=%s)" % (ta.__name__, runtime.gate))
    # Visible in the notebook stream at the moment the harness loads, and
    # written to disk so the outcome shows up in the kernel's output files.
    print("[k5] lever 3: PATCHED %s (gate=%s) -- episodic memory is live"
          % (ta.__name__, runtime.gate))
    try:
        status = Path(__file__).resolve().parent / "lever3_status.json"
        status.write_text(json.dumps(
            {"installed": True, "module": ta.__name__, "gate": runtime.gate,
             "events": runtime.events}, indent=1), encoding="utf-8")
    except Exception:
        pass


class _WatchingLoader(importlib.abc.Loader):
    def __init__(self, inner, runtime):
        self._inner = inner
        self._runtime = runtime

    def create_module(self, spec):
        return self._inner.create_module(spec)

    def exec_module(self, module):
        self._inner.exec_module(module)
        try:
            _patch_module(module, self._runtime)
        except Exception:
            self._runtime.events.append(
                "PATCH FAILED:\n" + traceback.format_exc())


class _Finder(importlib.abc.MetaPathFinder):
    def __init__(self, runtime):
        self._runtime = runtime

    def find_spec(self, fullname, path=None, target=None):
        if fullname != TARGET_MODULE:
            return None
        spec = importlib.machinery.PathFinder.find_spec(fullname, path)
        if spec is None or spec.loader is None:
            return None
        spec.loader = _WatchingLoader(spec.loader, self._runtime)
        return spec


def install(gate: bool = True) -> _Runtime:
    """Install Lever-3. Safe to call before or after the harness is imported."""
    global _RUNTIME
    runtime = _Runtime(gate)
    _RUNTIME = runtime
    if TARGET_MODULE in sys.modules:
        _patch_module(sys.modules[TARGET_MODULE], runtime)
        runtime.events.append("harness was already imported; patched in place")
    else:
        for i, finder in enumerate(sys.meta_path):
            if type(finder).__name__ == "_Finder":
                sys.meta_path.pop(i)
                break
        sys.meta_path.insert(0, _Finder(runtime))
        runtime.events.append("watching for %s" % TARGET_MODULE)
    return runtime
'''.lstrip('\n')
if not t.is_file() or t.stat().st_size < 100:
    t.write_text(EMBEDDED)
    print('EMBEDDED FALLBACK used ->', t.name, t.stat().st_size, 'bytes')
else:
    cur = t.read_text()
    print('using fetched', t.name, t.stat().st_size, 'bytes',
          '(identical to embedded)' if cur == EMBEDDED else
          '(DIFFERS from embedded: the repo fetch is the source of truth)')
print('  sha256', hashlib.sha256(t.read_bytes()).hexdigest()[:16], t.name)


In [ ]:
import hashlib, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
t = D / 'k5_gym.py'
EMBEDDED = r'''
#!/usr/bin/env python3
"""k5_gym.py — the k5 local filter: a tiny deterministic gym for the agent.

Five hand-built micro-games drive MyAgent through the exact frame/action
contract the harness uses (duck-typed frames; no arcengine needed — the
agent only ever getattr()s the fields it reads). Each game ends on WIN or
after the agent's own MAX_ACTIONS decisions, so the whole suite runs in
seconds on CPU.

THIS IS A FILTER, NEVER A SCORE. Pass/fail here means "not obviously
broken". It cannot and does not predict the hidden-game leaderboard —
the numbers it prints are smoke-test counters from five toy grids, and
they are labelled as such everywhere they are stored.

Determinism: no randomness anywhere (boards are formulas). One run is one
data point; the suite is never repeated-and-averaged.

Usage
  python3 k5_gym.py --agent program_synthesis_agent_v4.py [--knobs file.json]
                    [--json] [--require-levels N]
Exit code: 0 = PASS, 1 = FAIL, 2 = usage/IO error.
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import os
import sys
import time
import types

import numpy as np

GAMES = ("rarest_click", "largest_click", "move_chase", "frozen_void",
         "transient_anim")
FILTER_VERSION = 1


# --------------------------------------------------------------------------
# frame plumbing (duck-typed; matches what the agent actually reads)
# --------------------------------------------------------------------------

def _ns(**kw):
    return types.SimpleNamespace(**kw)


def frame_of(game, board, state):
    # COPY the board: the agent keeps frames by reference (astype copy=False),
    # so mutating one array in place would alias history[-1] == current frame.
    # The real harness always ships fresh arrays; the gym must too.
    return _ns(game_id=game.game_id,
               frame=[np.array(board, dtype=np.int64, copy=True)],
               state=state,                       # plain strings; the agent
               levels_completed=game.levels,      #   reads .value via getattr
               win_levels=game.win_levels,        #   with a str() fallback
               available_actions=list(game.available),
               action_input=None, guid=None, full_reset=False)


def extract_action(act):
    """(aid, x, y) from whatever action object the agent returned."""
    aid = 0
    try:
        aid = int(getattr(act, "value", getattr(act, "id", 0)) or 0)
    except Exception:
        aid = 0
    data = getattr(act, "action_data", None)
    x = int(getattr(data, "x", 0) or 0) if data is not None else 0
    y = int(getattr(data, "y", 0) or 0) if data is not None else 0
    return aid, x, y


def _cell_of(x, y):
    return (max(0, min(7, y // 8)), max(0, min(7, x // 8)))


def _put(board, r, c, h, w, color):
    r0, c0 = r * 8, c * 8
    board[r0:r0 + h, c0:c0 + w] = color


# --------------------------------------------------------------------------
# the five micro-games
# --------------------------------------------------------------------------

class Base:
    available = (6, 1, 2, 3, 4, 5)
    win_levels = 2

    def __init__(self, game_id):
        self.game_id = game_id
        self.state = "NOT_PLAYED"
        self.levels = 0
        self.board = np.zeros((64, 64), dtype=np.int64)
        self._build_level()

    def _build_level(self):
        raise NotImplementedError

    def _advance(self):
        self.levels += 1
        if self.levels >= self.win_levels:
            self.state = "WIN"
        else:
            self.board[:] = 0
            self._build_level()

    def observe(self):
        return frame_of(self, self.board, self.state)

    def apply(self, aid, x, y):
        if self.state == "NOT_PLAYED":
            if aid == 0:
                self.state = "NOT_FINISHED"
            return
        if self.state != "NOT_FINISHED":
            return
        self._act(aid, x, y)

    def _act(self, aid, x, y):
        raise NotImplementedError


class RarestClick(Base):
    """Click the uniquely rare blue block; it vanishes and the level clears.
    Correct click == the agent's recolor-to-zero prediction exactly."""

    game_kind = "rarest_click"

    def _build_level(self):
        _put(self.board, 2, 2, 8, 8, 3)                     # green  blocks
        _put(self.board, 2, 5, 8, 8, 3)
        _put(self.board, 5, 2, 8, 8, 3)
        _put(self.board, 5, 5, 8, 8, 1 if self.levels == 0 else 2)  # the rare one

    def _act(self, aid, x, y):
        if aid != 6:
            return
        if _cell_of(x, y) == (5, 5):
            self.board[40:48, 40:48] = 0
            self._advance()


class LargestClick(Base):
    """Only clicking the 16x16 block works; the three 8x8 blocks are decoys.
    Exercises selector demotion (rarest/smallest fail, largest verified)."""

    game_kind = "largest_click"

    def _build_level(self):
        _put(self.board, 1, 1, 8, 8, 3)
        _put(self.board, 1, 4, 8, 8, 2)
        _put(self.board, 4, 1, 8, 8, 5)
        _put(self.board, 4, 4, 16, 16, 4 if self.levels == 0 else 6)

    def _act(self, aid, x, y):
        if aid != 6:
            return
        if 32 <= y < 48 and 32 <= x < 48:
            self.board[32:48, 32:48] = 0
            self._advance()


class MoveChase(Base):
    """Move the 8x8 player onto the 8x8 goal. ACTION1..4 = up/down/left/right.
    Exercises control probing, direction learning, move predictions, and
    cross-level replay (level 2 is the same rule at a new location)."""

    game_kind = "move_chase"
    available = (1, 2, 3, 4)
    win_levels = 2
    DELTAS = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}

    def _build_level(self):
        self.p = (2, 2) if self.levels == 0 else (5, 1)   # player cell
        self.g = (6, 5) if self.levels == 0 else (1, 6)   # goal cell
        _put(self.board, self.g[0], self.g[1], 8, 8, 3)
        self._draw_player()

    def _draw_player(self):
        _put(self.board, self.p[0], self.p[1], 8, 8, 5)

    def _act(self, aid, x, y):
        d = self.DELTAS.get(aid)
        if d is None:
            return
        nr = max(0, min(7, self.p[0] + d[0]))
        nc = max(0, min(7, self.p[1] + d[1]))
        if (nr, nc) == self.p:
            return                                      # wall: NO_EFFECT
        self.board[:] = 0
        self.p = (nr, nc)
        if self.p == self.g:
            self._advance()
            return
        _put(self.board, self.g[0], self.g[1], 8, 8, 3)
        self._draw_player()


class FrozenVoid(Base):
    """Nothing ever changes. The never-hang regression detector: v1 burned
    71/80 actions on ACTION1 here; v2+ must rotate/blacklist and terminate."""

    game_kind = "frozen_void"
    available = (1, 2, 3, 4, 5)
    win_levels = 1

    def _build_level(self):
        _put(self.board, 3, 3, 4, 4, 7)

    def _act(self, aid, x, y):
        return


class TransientAnim(Base):
    """Wrong clicks flash a stripe for exactly one frame, then revert.
    Right click (the blue block) deletes it and clears the level. Teaches
    the filter that typed-transient evidence must not corrupt the rules."""

    game_kind = "transient_anim"

    def _build_level(self):
        self._flash = False
        self._flash_count = 0              # flashes are finite: after three
        _put(self.board, 2, 2, 8, 8, 3)    # the animation settles for good,
        _put(self.board, 2, 5, 8, 8, 3)    # so wrong clicks can never hide
        _put(self.board, 5, 3, 8, 8, 1)    # the settled world permanently

    def _act(self, aid, x, y):
        if self._flash:                                  # animation frame over:
            self.board[0, :16] = 0                       # revert to settled
            self._flash = False
        if aid != 6:
            return
        r, c = _cell_of(x, y)
        if (r, c) == (5, 3):
            self.board[40:48, 24:32] = 0
            self._advance()
        elif self._flash_count < 3:
            self._flash_count += 1
            self.board[0, :16] = 9                       # one-frame flash
            self._flash = True


SUITE = (RarestClick, LargestClick, MoveChase, FrozenVoid, TransientAnim)


# --------------------------------------------------------------------------
# driver + agent loading
# --------------------------------------------------------------------------

def load_agent(path, knobs_path=None):
    """Import the agent module by path, with the knob channel pointed at
    knobs_path (the same contract the Kaggle run uses). Fresh module object
    per call so variant runs never share GAME_MEMORY."""
    if knobs_path:
        os.environ["K5_KNOBS_JSON"] = knobs_path
    elif "K5_KNOBS_JSON" in os.environ:
        del os.environ["K5_KNOBS_JSON"]
    import tempfile
    os.environ["K5_TELEMETRY_DIR"] = tempfile.mkdtemp(prefix="k5gym_telem_")
    name = "k5_agent_" + hashlib.sha256(
        (os.path.abspath(path) + str(os.path.getmtime(path))
         + str(knobs_path)).encode()).hexdigest()[:12]
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def play(agent_mod, game, trace=None):
    """One game: observe -> is_done -> choose_action -> apply, capped at
    the agent's own MAX_ACTIONS decisions (mirrors the harness)."""
    MyAgent = agent_mod.MyAgent
    agent = MyAgent(game_id=game.game_id)
    decisions = 0
    slow = 0
    t_cap = time.perf_counter() + 60.0                   # per-game wall guard
    while True:
        frame = game.observe()
        if agent.is_done([frame], frame):
            break
        if decisions >= MyAgent.MAX_ACTIONS or time.perf_counter() > t_cap:
            break
        t0 = time.perf_counter()
        act = agent.choose_action([frame], frame)
        dt = time.perf_counter() - t0
        if dt > 5.0:
            return {"game": game.game_kind, "error": "decision exceeded 5s",
                    "levels": game.levels}
        if dt > MyAgent.TIME_BUDGET_S:
            slow += 1
        decisions += 1
        aid, x, y = extract_action(act)
        if trace is not None:
            trace.append(aid)
        game.apply(aid, x, y)
    return {"game": game.game_kind,
            "levels": int(game.levels),
            "win": 1 if game.state == "WIN" else 0,
            "decisions": decisions,
            "slow_decisions": slow}


def run_suite(agent_path, knobs_path=None):
    out = {"filter_version": FILTER_VERSION,
           "agent_path": os.path.basename(agent_path),
           "agent_sha256": None,
           "games": [], "levels": 0, "wins": 0, "errors": []}
    try:
        with open(agent_path, "rb") as fh:
            out["agent_sha256"] = hashlib.sha256(fh.read()).hexdigest()
        t0 = time.perf_counter()
        mod = load_agent(agent_path, knobs_path)
        for cls in SUITE:
            trace = []
            res = play(mod, cls(cls.game_kind), trace)
            res["trace_sha256_12"] = hashlib.sha256(
                json.dumps(trace).encode()).hexdigest()[:12]
            out["games"].append(res)
            if "error" in res:
                out["errors"].append(res["error"])
            out["levels"] += int(res.get("levels", 0))
            out["wins"] += int(res.get("win", 0))
        out["wall_seconds"] = round(time.perf_counter() - t0, 3)
    except Exception as exc:                              # a crash IS a result
        out["errors"].append("suite crashed: %r" % (exc,))
    out["ok"] = not out["errors"]
    return out


def verdict(base, cand, require_levels=0):
    """FAIL only on 'obviously broken': crash/hang, any win lost, or a drop
    of 2+ levels across the five toys. Anything else PASSES the filter —
    small shuffles between toys are exactly what a filter must not read
    into. Never stated or stored as a leaderboard prediction."""
    reasons = []
    if not cand.get("ok"):
        reasons.append("candidate crashed or hung: %s" % cand.get("errors"))
    if cand.get("wins", 0) < base.get("wins", 0):
        reasons.append("candidate lost a game win (%d -> %d)"
                       % (base.get("wins"), cand.get("wins")))
    if cand.get("levels", 0) <= base.get("levels", 0) - 2:
        reasons.append("candidate lost 2+ toy levels (%d -> %d)"
                       % (base.get("levels"), cand.get("levels")))
    if cand.get("levels", 0) < require_levels:
        reasons.append("below --require-levels floor (%d < %d)"
                       % (cand.get("levels"), require_levels))
    return ("FAIL" if reasons else "PASS"), reasons


def _print(result):
    print("  games:")
    for g in result.get("games", []):
        print("    %-15s levels=%d win=%d decisions=%d slow=%d trace=%s %s"
              % (g["game"], g.get("levels", 0), g.get("win", 0),
                 g.get("decisions", 0), g.get("slow_decisions", 0),
                 g.get("trace_sha256_12", "-"),
                 ("ERROR: " + g["error"]) if "error" in g else ""))
    print("  totals: levels=%d wins=%d wall=%ss errors=%s"
          % (result.get("levels", 0), result.get("wins", 0),
             result.get("wall_seconds", "?"), result.get("errors") or "none"))


def main():
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--agent", required=True, help="path to the agent .py")
    ap.add_argument("--knobs", default=None,
                    help="k5_knobs.json for the variant; omit for defaults")
    ap.add_argument("--base-knobs", default=None,
                    help="also run the incumbent from this JSON and compare")
    ap.add_argument("--require-levels", type=int, default=0)
    ap.add_argument("--json", action="store_true", help="machine-readable output")
    a = ap.parse_args()

    if not os.path.isfile(a.agent):
        print("agent not found: %s" % a.agent, file=sys.stderr)
        return 2

    cand = run_suite(a.agent, a.knobs)
    base = run_suite(a.agent, a.base_knobs) if a.base_knobs else None
    result = {"mode": "compare" if base else "single",
              "candidate": cand, "incumbent": base}
    if base:
        v, reasons = verdict(base, cand, a.require_levels)
        result["verdict"] = v
        result["verdict_reasons"] = reasons
    else:
        v, reasons = ("PASS" if cand["ok"]
                      and cand["levels"] >= a.require_levels else "FAIL"), \
                     ([] if cand["ok"] else cand["errors"])
        result["verdict"] = v
        result["verdict_reasons"] = reasons

    if a.json:
        print(json.dumps(result, indent=1, sort_keys=True))
    else:
        print("k5 gym (filter v%d) — a smoke test, NEVER a leaderboard score"
              % FILTER_VERSION)
        if base:
            print("incumbent:"); _print(base)
            print("candidate:"); _print(cand)
        else:
            _print(cand)
        print("verdict: %s %s"
              % (result["verdict"],
                 ("; ".join(result["verdict_reasons"])) or ""))
    return 0 if result["verdict"] == "PASS" else 1


if __name__ == "__main__":
    sys.exit(main())
'''.lstrip('\n')
if not t.is_file() or t.stat().st_size < 100:
    t.write_text(EMBEDDED)
    print('EMBEDDED FALLBACK used ->', t.name, t.stat().st_size, 'bytes')
else:
    cur = t.read_text()
    print('using fetched', t.name, t.stat().st_size, 'bytes',
          '(identical to embedded)' if cur == EMBEDDED else
          '(DIFFERS from embedded: the repo fetch is the source of truth)')
print('  sha256', hashlib.sha256(t.read_bytes()).hexdigest()[:16], t.name)


In [ ]:
import hashlib, pathlib
D = pathlib.Path('/kaggle/working/kaggle5'); D.mkdir(parents=True, exist_ok=True)
t = D / 'program_synthesis_agent_v4.py'
EMBEDDED = r'''
"""program_synthesis_agent_v4.py — self-contained ARC-AGI-3 agent (v2.2)

v2.2 (k5 between-game learning loop) adds exactly two things over v2.1:
  K1 knob channel: module constants below can be overridden from a JSON file
     the daily driver writes before this module is imported
     (K5_KNOBS_JSON env var, else /kaggle/working/k5_knobs.json).
     No file = the v2.1 defaults apply = behaviour identical to v2.1.
  K2 telemetry: per-game outcome rows appended to k5_per_game.csv next to
     the run's output (K5_TELEMETRY_DIR env var, else /kaggle/working),
     plus a one-off k5_run_info.json self-report. Telemetry is wrapped so
     it can never break gameplay; it changes no decisions.

Carried over from v2.1:
  F4 state-action exploration graph (novelty-seeking when no hypothesis)
  F5 typed evidence: transient/revert detection (settled vs animation)
Carried over from v2.0:
  F1 frame diffing + no-op blacklist    F2 cross-level verified replay
  F3 level-1-first exploration
Carried over from v1:
  Protocol layer, perception, micro-DSL, hypothesis bank, never-hang wrapper.

Contract: class MyAgent with choose_action(frames, latest_frame) and
is_done(frames, latest_frame). numpy + stdlib only; no network.
"""
import copy
import time
from collections import deque

import numpy as np

# ============================================================================
# SECTION 1 — PROTOCOL LAYER (unchanged from v2)
# ============================================================================
GRID = 64
CELL = 8

try:
    from arcengine import FrameData as _RealFrameData  # noqa: F401
    from arcengine import GameAction, GameState
    from arcengine.enums import (SimpleAction, ComplexAction, FrameData,
                                 FrameDataRaw, ActionInput)  # noqa: F401
    ARCENGINE_AVAILABLE = True
except Exception:
    ARCENGINE_AVAILABLE = False

    from enum import Enum

    class GameState(str, Enum):
        NOT_PLAYED = "NOT_PLAYED"
        NOT_FINISHED = "NOT_FINISHED"
        WIN = "WIN"
        GAME_OVER = "GAME_OVER"

    class SimpleAction:
        def __init__(self, game_id: str = ""):
            self.game_id = game_id

        def model_dump(self):
            return {"game_id": self.game_id}

    class ComplexAction:
        def __init__(self, game_id: str = "", x: int = 0, y: int = 0):
            if not (0 <= x <= 63 and 0 <= y <= 63):
                raise ValueError("x and y must be within [0, 63]")
            self.game_id = game_id
            self.x = x
            self.y = y

        def model_dump(self):
            return {"game_id": self.game_id, "x": self.x, "y": self.y}

    class GameAction(Enum):
        RESET = (0, SimpleAction)
        ACTION1 = (1, SimpleAction)
        ACTION2 = (2, SimpleAction)
        ACTION3 = (3, SimpleAction)
        ACTION4 = (4, SimpleAction)
        ACTION5 = (5, SimpleAction)
        ACTION6 = (6, ComplexAction)
        ACTION7 = (7, SimpleAction)

        def __init__(self, action_id, action_type):
            self._value_ = action_id
            self._atype = action_type
            self.action_data = action_type()

        def is_simple(self):
            return self._atype is SimpleAction

        def is_complex(self):
            return self._atype is ComplexAction

        def set_data(self, data):
            self.action_data = self._atype(**data)
            return self.action_data

        @classmethod
        def from_id(cls, action_id):
            for a in cls:
                if a.value == action_id:
                    return a
            raise ValueError(f"No GameAction with id {action_id}")

    class ActionInput:
        def __init__(self, id=GameAction.RESET, data=None, reasoning=None):
            self.id = id
            self.data = data or {}
            self.reasoning = reasoning

    class FrameData:
        def __init__(self, game_id="", frame=None, state=GameState.NOT_PLAYED,
                     levels_completed=0, win_levels=0, action_input=None,
                     guid=None, full_reset=False, available_actions=None):
            self.game_id = game_id
            self.frame = frame or []
            self.state = state
            self.levels_completed = levels_completed
            self.win_levels = win_levels
            self.action_input = action_input or ActionInput()
            self.guid = guid
            self.full_reset = full_reset
            self.available_actions = list(available_actions or [])

        def is_empty(self):
            return len(self.frame) == 0


def _pick_frame(frames, latest_frame):
    """Prefer the freshest frame that actually carries grid data.

    The kit's main() seeds `frames` with a dummy FrameData and passes the
    converted current observation as latest_frame, so latest_frame wins
    whenever it has pixels; the history covers callers that pass a stale
    or empty second argument but real frames in `frames`.
    """
    try:
        if getattr(latest_frame, "frame", None):
            return latest_frame
    except Exception:
        pass
    try:
        for cand in reversed(list(frames or [])):
            if getattr(cand, "frame", None):
                return cand
    except Exception:
        pass
    return latest_frame


def _state_name(state):
    try:
        return str(getattr(state, "value", state)).split(".")[-1]
    except Exception:
        return ""


def _grid_of(frame):
    try:
        layers = getattr(frame, "frame", None)
        if not layers:
            layers = getattr(frame, "_frame", None)
        if not layers:
            return None
        arr = np.asarray(layers[0])
        if arr.ndim != 2:
            return None
        return arr.astype(np.int64, copy=False)
    except Exception:
        return None


def merged_view(frame):
    try:
        layers = getattr(frame, "frame", None) or []
        if not layers:
            return None
        base = np.asarray(layers[0]).astype(np.int64, copy=False)
        for extra in layers[1:]:
            arr = np.asarray(extra)
            if arr.shape == base.shape:
                base = np.where(arr != 0, arr, base)
        return base
    except Exception:
        return None


# ============================================================================
# SECTION 2 — PERCEPTION (unchanged from v2)
# ============================================================================
def find_objects(grid):
    objects = []
    visited = set()
    h, w = grid.shape
    for r in range(h):
        for c in range(w):
            if grid[r, c] == 0 or (r, c) in visited:
                continue
            color = int(grid[r, c])
            obj_id = len(objects) + 1
            q = deque([(r, c)])
            visited.add((r, c))
            pixels = []
            min_r, max_r, min_c, max_c = r, r, c, c
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                min_r, max_r = min(min_r, cr), max(max_r, cr)
                min_c, max_c = min(min_c, cc), max(max_c, cc)
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    nr, nc = cr + dr, cc + dc
                    if (0 <= nr < h and 0 <= nc < w and (nr, nc) not in visited
                            and grid[nr, nc] == color):
                        visited.add((nr, nc))
                        q.append((nr, nc))
            cy = sum(p[0] for p in pixels) / len(pixels)
            cx = sum(p[1] for p in pixels) / len(pixels)
            objects.append({"id": obj_id, "color": color,
                            "pixels": frozenset(pixels),
                            "bbox": (min_r, min_c, max_r, max_c),
                            "size": len(pixels), "centroid": (cy, cx)})
    return objects


def frame_diff(grid_a, grid_b):
    if grid_a is None or grid_b is None or grid_a.shape != grid_b.shape:
        return np.ones_like(np.asarray(grid_b), dtype=bool)
    return grid_a != grid_b


def salience_rank(objects):
    color_count = {}
    for o in objects:
        color_count[o["color"]] = color_count.get(o["color"], 0) + 1
    ranked = sorted(objects,
                    key=lambda o: (color_count[o["color"]], o["size"], o["bbox"]))
    return ranked, color_count


# ============================================================================
# SECTION 3 — MICRO-DSL (unchanged from v2)
# ============================================================================
def prim_recolor_region(grid, mask, new_color):
    g = copy.deepcopy(grid)
    g[mask] = new_color
    return g


def prim_move_region(grid, mask, dr, dc):
    g = copy.deepcopy(grid)
    color = int(g[mask][0]) if mask.any() else 0
    g[mask] = 0
    rows = np.nonzero(mask)[0] + dr
    cols = np.nonzero(mask)[1] + dc
    keep = ((rows >= 0) & (rows < g.shape[0])
            & (cols >= 0) & (cols < g.shape[1]))
    g[rows[keep], cols[keep]] = color
    return g


# ============================================================================
# SECTION 4 — GOVERNANCE (UL-derived, unchanged from v2)
# ============================================================================
EFFECT_NO_EFFECT = "NO_EFFECT"
EFFECT_LOCAL = "LOCAL_CHANGE"
EFFECT_GLOBAL = "GLOBAL_CHANGE"
EFFECT_LEVEL = "LEVEL_ADVANCED"
EFFECT_TRANSIENT = "TRANSIENT_REVERT"   # v2.1 new

EVIDENCE_VERIFIED = "VERIFIED"
EVIDENCE_CORROBORATED = "CORROBORATED"

STATUS_ACTIVE = "ACTIVE"
STATUS_STALE = "STALE"
STATUS_QUARANTINED = "QUARANTINED"

BLACKLIST_CAP = 64
SEQ_STORE_CAP = 60
GLOBAL_CHANGE_FRACTION = 0.25

# v2.1 state-graph constants
GRAPH_HASH_STRIDE = 4               # downsample grid for hashing
GRAPH_RECENT_CAP = 12               # history of recent distinct states
GRAPH_TRANSIENT_MAX_DIFF = 32       # pixel diff threshold for transient
EXPLORATION_CAP = 12                # consecutive no-progress explore actions
                                    # (design §4.3: worst case = v2.0 minus
                                    #  at most EXPLORATION_CAP actions)


class EpisodeLedger:
    def __init__(self):
        self.events = []

    def append(self, kind, **payload):
        eid = len(self.events)
        self.events.append(dict(id=eid, kind=kind, **payload))
        return eid

    def __len__(self):
        return len(self.events)


class RuleRecord:
    def __init__(self, rule_id, kind):
        self.rule_id = rule_id
        self.kind = kind
        self.evidence_class = None
        self.status = STATUS_ACTIVE
        self.success = 0
        self.fail = 0
        self.rotations = 0
        self.evidence = []

    def cite(self, eid):
        self.evidence.append(eid)


# ============================================================================
# SECTION 4.5 — STATE-ACTION EXPLORATION GRAPH (v2.1 new)
# ============================================================================
class StateGraph:
    """Novelty-seeking state-action graph over settled frames.

    - States are hashed via a downsampled grid to filter single-pixel noise.
    - `recent` keeps distinct consecutive state hashes (auto-capped).
    - `visits` counts how many times each state was observed.
    - `transitions` records (from_state -> action_key -> to_state).
    - `action_uses` counts how many transitions each action participated in.

    Transient detection: a new state is flagged as transient (revert) iff
    it matches the state two distinct-changes ago AND the frame diff is
    small (< GRAPH_TRANSIENT_MAX_DIFF pixels). Small reverts are animation.
    """

    def __init__(self):
        self.recent = deque(maxlen=GRAPH_RECENT_CAP)
        self.visits = {}
        self.transitions = {}
        self.action_uses = {}

    @staticmethod
    def hash_grid(grid):
        if grid is None:
            return None
        try:
            arr = np.asarray(grid)
            if arr.ndim != 2:
                return None
            small = arr[::GRAPH_HASH_STRIDE, ::GRAPH_HASH_STRIDE]
            return hash(small.tobytes())
        except Exception:
            return None

    def is_revert(self, h, diff_pixels):
        """True if h matches the state two changes ago AND diff is small."""
        if h is None or diff_pixels > GRAPH_TRANSIENT_MAX_DIFF:
            return False
        if len(self.recent) < 2:
            return False
        return self.recent[-2] == h

    def observe(self, h):
        """Record this frame's hash. Idempotent for consecutive repeats."""
        if h is None:
            return
        if self.recent and self.recent[-1] == h:
            return
        self.recent.append(h)
        self.visits[h] = self.visits.get(h, 0) + 1

    def record_transition(self, from_h, key, to_h):
        if from_h is None or to_h is None or key is None:
            return
        self.transitions.setdefault(from_h, {})[key] = to_h
        aid = key if isinstance(key, int) else key[0]
        self.action_uses[aid] = self.action_uses.get(aid, 0) + 1

    def novelty_action(self, current_h, avail):
        """Pick a novelty-seeking action at the current state.
        Priority: untried simple action (least-used globally), then untried
        click cell (if 6 in avail), then action leading to least-visited
        state, else None."""
        if not avail:
            return None
        tried = self.transitions.get(current_h, {}) if current_h is not None else {}
        # 1) untried simple actions
        simple_untried = [a for a in avail if a != 6 and a not in tried]
        if simple_untried:
            return min(simple_untried,
                       key=lambda a: self.action_uses.get(a, 0))
        # 2) untried click cells (only if 6 is available)
        if 6 in avail:
            tried_cells = {k[1:] for k in tried
                           if isinstance(k, tuple) and k[0] == 6}
            for r in range(8):
                for c in range(8):
                    if (r, c) not in tried_cells:
                        return (6, r, c)
        # 3) least-visited successor (FIX audit B2: self-loops are dead
        # ends — a no-effect pair must never win the novelty pick)
        candidates = [(k, v) for k, v in tried.items() if v != current_h]
        if candidates:
            return min(candidates,
                       key=lambda p: self.visits.get(p[1], 0))[0]
        return None

    def clear_recent(self):
        self.recent.clear()


class GameKnowledge:
    def __init__(self, game_id):
        self.game_id = game_id
        self.rules = {}
        self.move = None
        self.sequences = {}
        self.graph = StateGraph()   # v2.1 persistent graph

    def rule(self, name):
        if name not in self.rules:
            self.rules[name] = RuleRecord(name, "click")
        return self.rules[name]

    def best_selector(self, order):
        def key(sel):
            rec = self.rules.get(sel)
            if rec is None:
                return (2, 0, 0)
            if rec.status == STATUS_QUARANTINED:
                return (9, 0, 0)
            if rec.status == STATUS_STALE:
                return (1, 0, 0)
            rank = {EVIDENCE_VERIFIED: 0, EVIDENCE_CORROBORATED: 1, None: 2}
            return (0, rank.get(rec.evidence_class, 2), -rec.success)
        return sorted(order, key=key)


def classify_effect(prev, cur, levels_before, levels_after, transient=False):
    if levels_after > levels_before:
        return EFFECT_LEVEL, 0
    d = frame_diff(prev, cur)
    n = int(d.sum())
    if n == 0:
        return EFFECT_NO_EFFECT, 0
    if transient:
        return EFFECT_TRANSIENT, n
    if n > GLOBAL_CHANGE_FRACTION * prev.size or n > 1024:
        return EFFECT_GLOBAL, n
    return EFFECT_LOCAL, n


# ============================================================================
# SECTION 5 — PROGRAM SYNTHESIS CORE v2.1
# ============================================================================
CLICK_SELECTORS = ["rarest_color", "smallest_object",
                   "corner_sample_color", "largest_object"]
PROBE_ATTEMPT_CAP = 3
REPLAY_ABORT_STALE = 2

# --------------------------------------------------------------------------
# K1 (v2.2 new) — k5 between-game learning loop: external knob overrides.
#
# The daily driver writes k5_knobs.json before this module is imported:
#   {"variant": "exp-001", "knobs": {"EXPLORATION_CAP": 24, ...}, ...}
# Only the names listed in K5_KNOB_NAMES may be overridden, and only with
# values of the matching type. GRID and CELL are structural and can never be
# overridden. With no file present (or any read failure) the module is
# byte-for-byte equivalent to v2.1 behaviour. K5_RUN_INFO records what was
# actually applied so a run can always be traced back to a ledger row.
# --------------------------------------------------------------------------
K5_KNOB_NAMES = (
    "BLACKLIST_CAP", "SEQ_STORE_CAP", "GLOBAL_CHANGE_FRACTION",
    "GRAPH_HASH_STRIDE", "GRAPH_RECENT_CAP", "GRAPH_TRANSIENT_MAX_DIFF",
    "EXPLORATION_CAP", "CLICK_SELECTORS", "PROBE_ATTEMPT_CAP",
    "REPLAY_ABORT_STALE",
)

K5_RUN_INFO = {"variant": None, "experiment": None, "reason": None,
               "applied_overrides": {}, "agent_sha256": None,
               "knobs": {}}


def _k5_self_sha256():
    import hashlib
    try:
        with open(__file__, "rb") as fh:
            return hashlib.sha256(fh.read()).hexdigest()
    except Exception:
        return None


def _k5_apply_knob_overrides():
    """Read the driver's JSON (if any) and rebind the knob constants.

    Runs at import time, before any class is built or game starts, so every
    consumer — definition-time (deque maxlen) and runtime (global lookups) —
    sees the same values. Never raises; returns the applied dict.
    """
    import json as _json
    import os as _os
    path = _os.environ.get("K5_KNOBS_JSON") or "/kaggle/working/k5_knobs.json"
    try:
        with open(path) as fh:
            payload = _json.load(fh)
    except Exception:
        return {}
    if not isinstance(payload, dict):
        return {}
    overrides = payload.get("knobs") if "knobs" in payload else payload
    if not isinstance(overrides, dict):
        overrides = {}
    g = globals()
    applied = {}
    for name in K5_KNOB_NAMES:
        if name not in overrides or overrides[name] is None:
            continue
        cur = g[name]
        val = overrides[name]
        try:
            if isinstance(cur, list):
                if not (isinstance(val, list) and val
                        and all(isinstance(v, str) for v in val)):
                    continue
                g[name] = list(val)
            elif isinstance(cur, bool):
                continue                      # no bool knobs; refuse coercion
            elif isinstance(cur, int):
                g[name] = int(val)
            elif isinstance(cur, float):
                g[name] = float(val)
            else:
                continue
        except (TypeError, ValueError):
            continue
        applied[name] = g[name]
    K5_RUN_INFO["variant"] = payload.get("variant")
    K5_RUN_INFO["experiment"] = payload.get("experiment")
    K5_RUN_INFO["reason"] = payload.get("reason")
    K5_RUN_INFO["agent_sha256"] = _k5_self_sha256()
    return applied


_K5_KNOBS_APPLIED = _k5_apply_knob_overrides()
K5_RUN_INFO["applied_overrides"] = dict(_K5_KNOBS_APPLIED)
K5_RUN_INFO["knobs"] = {n: globals()[n] for n in K5_KNOB_NAMES}
# Record this file's hash even when no knob file exists (baseline runs are
# the runs where provenance matters most).
if not K5_RUN_INFO.get("agent_sha256"):
    K5_RUN_INFO["agent_sha256"] = _k5_self_sha256()
if _K5_KNOBS_APPLIED:
    print("[k5] knob overrides applied at import: %s" % _K5_KNOBS_APPLIED)


def _action_key(pend):
    """Hashable key identifying the pending action for the state graph."""
    if not pend:
        return None
    aid = pend.get("aid")
    if aid is None:
        return None
    if aid == 6:
        data = pend.get("data") or {}
        x = int(data.get("x", 0))
        y = int(data.get("y", 0))
        return (6, y // CELL, x // CELL)
    return aid


class ProgramSynthesisV2Core:

    GAME_MEMORY = {}

    def __init__(self, verbose=False):
        self.verbose = verbose
        self._reset_episode()

    def _reset_episode(self):
        self.game_id = ""
        self.ledger = EpisodeLedger()
        self.history = []
        self.pending = None
        self.level = 0
        self.mode = None
        self.sel_order = list(CLICK_SELECTORS)
        self.queue = []
        self.queue_ptr = 0
        self.clicked_sig = set()
        self.blacklist = set()
        self.dirs = {}
        self.probe_attempts = {}
        self.player_color = None
        self.stuck = 0
        self.replay = None
        self.level_actions = []
        self.level_verified = False
        # v2.1 exploration budget (design §4.3)
        self.explore_streak = 0
        self.explore_blocked = False
        self._explore_rr = 0
        # v2.1: transient graph, replaced in new_episode with persistent one
        self.graph = StateGraph()

    def new_episode(self, game_id):
        know = ProgramSynthesisV2Core.GAME_MEMORY.get(game_id)
        self._reset_episode()
        self.game_id = game_id or ""
        if know is None:
            know = GameKnowledge(game_id)
            ProgramSynthesisV2Core.GAME_MEMORY[game_id] = know
        self.knowledge = know
        self.graph = know.graph                     # v2.1 persistent
        if know.move:
            self.dirs = dict(know.move.get("dirs") or {})
            self.player_color = know.move.get("player_color")
        self.sel_order = know.best_selector(list(CLICK_SELECTORS))
        self.ledger.append("EPISODE_START", game_id=self.game_id)

    # ------------------------------------------------------------- main entry
    def next_action(self, frame):
        gid = ""
        try:
            gid = getattr(frame, "game_id", "") or ""
        except Exception:
            pass
        if gid and gid != self.game_id:
            self.new_episode(gid)

        grid = self._grid(frame)
        if grid is None:
            return 0, {"game_id": self.game_id}, {"note": "no frame data"}

        if self.history:
            try:
                self._process_feedback(self.history[-1], grid, frame)
            except Exception:
                self.pending = None
        self.history.append(grid)

        state = _state_name(getattr(frame, "state", None))
        if state in ("NOT_PLAYED", "GAME_OVER"):
            self._clear_level_state()
            self.ledger.append("RESET_ISSUED", why=state)
            return 0, {"game_id": self.game_id}, {
                "strategy": "reset", "why": state,
                "hypothesis": self.sel_order[0] if self.sel_order else None}

        avail = self._available(frame)
        if not avail:
            return 0, {"game_id": self.game_id}, {"note": "no available actions"}
        self.mode = "click" if 6 in avail else "move"

        if self.replay is None and self.mode:
            seqrec = self.knowledge.sequences.get(self.level)
            if seqrec and not seqrec.get("stale"):
                self.replay = {"actions": seqrec["actions"], "idx": 0}
                self.ledger.append("REPLAY_STARTED", level=self.level,
                                   evidence_class=seqrec["evidence_class"],
                                   n_actions=len(seqrec["actions"]))
                if self.verbose:
                    print(f"    [v3] replay level {self.level}: "
                          f"{len(seqrec['actions'])} actions")
        if self.replay is not None:
            return self._replay_step(grid, frame, avail)

        if self.mode == "click":
            return self._click_policy(grid, frame, avail)
        return self._move_policy(grid, frame, avail)

    # ------------------------------------------------------------- feedback
    def _process_feedback(self, prev, cur, frame):
        levels_now = int(getattr(frame, "levels_completed", 0) or 0)
        pend = self.pending or {}
        self.pending = None
        aid = pend.get("aid")

        # v2.1 state graph: hash both frames
        h_prev = self.graph.hash_grid(prev)
        h_cur = self.graph.hash_grid(cur)
        diff_px = int(frame_diff(prev, cur).sum())

        # v2.1 transient detection: revert to state-2-ago with small diff
        is_revert = (h_cur is not None and h_prev != h_cur
                     and self.graph.is_revert(h_cur, diff_px))

        # v2.1 observe the new state (records visit + recent)
        if h_cur is not None:
            self.graph.observe(h_cur)

        effect, npx = classify_effect(prev, cur, self.level, levels_now,
                                      transient=is_revert)
        self.ledger.append("EFFECT_CLASSIFIED", action=aid, effect=effect,
                           changed=npx, level=levels_now,
                           replay=bool(pend.get("replay")))

        # v2.1 record transition (settled only; transients never recorded).
        # FIX (audit B2): NO_EFFECT pairs are recorded as SELF-LOOPS so the
        # graph counts them as tried — otherwise novelty_action re-proposes
        # the same dead action forever (the v1 _pending bug reborn: a frozen
        # game burned 71/80 actions on ACTION1).
        if (h_prev is not None and h_cur is not None
                and effect != EFFECT_TRANSIENT):
            key = _action_key(pend)
            if key is not None and (h_prev != h_cur
                                    or effect == EFFECT_NO_EFFECT):
                self.graph.record_transition(h_prev, key, h_cur)

        # v2.1 transients don't update rules; they're animation noise
        if effect == EFFECT_TRANSIENT:
            return

        # prediction validation
        verified = False
        predicted = pend.get("predicted")
        if predicted is not None and effect in (EFFECT_LOCAL, EFFECT_GLOBAL):
            ok = float((predicted == cur).mean()) if predicted.shape == cur.shape else 0.0
            if ok >= 0.97:
                verified = True

        if aid is not None:
            self.level_actions.append({
                "aid": aid, "data": pend.get("data") or {},
                "effect": effect, "verified": verified})
            if verified:
                self.level_verified = True
        if verified:
            self.explore_streak = 0   # progress: exploration budget resets

        if pend.get("replay"):
            expected = pend.get("expected")
            if effect != expected and effect != EFFECT_LEVEL:
                self._replay_abort(cause=f"expected {expected}, saw {effect}")
                return
            if effect == EFFECT_LEVEL:
                self.replay = None
                self.ledger.append("REPLAY_STEP_DONE", effect=effect)
                self._on_level_advanced(levels_now, pend)
                return
            self.ledger.append("REPLAY_STEP_OK", effect=effect)
            return

        if effect == EFFECT_LEVEL:
            self._on_level_advanced(levels_now, pend)
            return

        kind = pend.get("kind")
        if effect == EFFECT_NO_EFFECT:
            self._on_no_effect(pend)
            if kind == "probe":
                self._learn_direction(prev, cur, aid)
        elif kind == "click":
            self._on_click_feedback(pend, effect, verified)
        elif kind == "probe":
            self._learn_direction(prev, cur, aid)
        elif kind == "move":
            self._on_move_feedback(pend, prev, cur, aid, verified)
        # v2.1: "explore" actions update the graph but no rules

    def _on_level_advanced(self, levels_now, pend):
        seq = [a for a in self.level_actions if a["effect"] != EFFECT_NO_EFFECT]
        if 0 < len(self.level_actions) <= SEQ_STORE_CAP and seq:
            self.knowledge.sequences[self.level] = {
                "actions": seq,
                "evidence_class": (EVIDENCE_VERIFIED if self.level_verified
                                   else EVIDENCE_CORROBORATED),
                "aborts": 0, "stale": False}
            self.ledger.append("WIN_SEQUENCE_STORED", level=self.level,
                               n_actions=len(seq))
        rule_name = pend.get("rule")
        if self.mode == "move" and self.knowledge.move:
            rec = self.knowledge.move["record"]
            rec.success += 1
            if rec.evidence_class is None:
                rec.evidence_class = EVIDENCE_CORROBORATED
        elif rule_name:
            rec = self.knowledge.rule(rule_name)
            rec.success += 1
            if rec.evidence_class is None:
                rec.evidence_class = EVIDENCE_CORROBORATED
        self.ledger.append("LEVEL_ADVANCED", to=levels_now)
        self.level = levels_now
        self._clear_level_state()

    def _on_no_effect(self, pend):
        aid = pend.get("aid")
        kind = pend.get("kind")
        if kind == "click" and pend.get("sig"):
            self.clicked_sig.add(pend["sig"])
            if len(self.clicked_sig) > BLACKLIST_CAP:
                self.clicked_sig.pop()
        elif kind in ("simple", None) and self.mode != "move" and aid is not None:
            key = (self.level, aid)
            if key not in self.blacklist and len(self.blacklist) < BLACKLIST_CAP:
                self.blacklist.add(key)
                self.ledger.append("ACTION_BLACKLISTED", action=aid,
                                   level=self.level)
        if kind == "click" and pend.get("rule"):
            rec = self.knowledge.rule(pend["rule"])
            rec.fail += 1
        if kind == "move":
            self.stuck += 1

    def _on_click_feedback(self, pend, effect, verified):
        rule_name = pend.get("rule")
        if not rule_name:
            return
        rec = self.knowledge.rule(rule_name)
        if verified:
            rec.success += 1
            if rec.evidence_class != EVIDENCE_VERIFIED:
                rec.evidence_class = EVIDENCE_VERIFIED
                if self.verbose:
                    print(f"    [v3] VERIFIED '{rule_name}'")
            rec.cite(self.ledger.append("RULE_VERIFIED", rule=rule_name))
        elif pend.get("predicted") is not None:
            rec.fail += 1
            if rec.evidence_class == EVIDENCE_VERIFIED:
                rec.status = STATUS_STALE
                self.ledger.append("RULE_STALE", rule=rule_name,
                                   cause="prediction mismatch")
            self._demote(rule_name, why="prediction mismatch")
        else:
            rec.fail += 1

    def _on_move_feedback(self, pend, prev, cur, aid, verified):
        if self.knowledge.move is None:
            return
        rec = self.knowledge.move["record"]
        if verified:
            rec.success += 1
            if rec.evidence_class != EVIDENCE_VERIFIED:
                rec.evidence_class = EVIDENCE_VERIFIED
        self._learn_direction(prev, cur, aid)
        moved = bool(frame_diff(prev, cur).any())
        self.stuck = 0 if moved else self.stuck + 1

    def _learn_direction(self, prev, cur, aid):
        if aid is None or aid in self.dirs:
            return
        hole = (prev != 0) & (cur == 0)
        fill = (prev == 0) & (cur != 0)
        if hole.any() and fill.any():
            hr, hc = np.nonzero(hole)
            fr, fc = np.nonzero(fill)
            dr = int(round(fr.mean() - hr.mean()))
            dc = int(round(fc.mean() - hc.mean()))
            self.dirs[aid] = (int(np.sign(dr)) if abs(dr) > 1 else 0,
                              int(np.sign(dc)) if abs(dc) > 1 else 0)
            self.player_color = int(np.bincount(prev[hole]).argmax())
            if self.knowledge.move is None:
                self.knowledge.move = {
                    "dirs": {}, "player_color": None,
                    "record": RuleRecord("move", "control")}
            self.knowledge.move["dirs"] = dict(self.dirs)
            self.knowledge.move["player_color"] = self.player_color
        else:
            self.probe_attempts[aid] = self.probe_attempts.get(aid, 0) + 1
            if self.probe_attempts[aid] >= PROBE_ATTEMPT_CAP:
                self.dirs[aid] = (0, 0)

    # ------------------------------------------------------------- replay
    def _replay_step(self, grid, frame, avail):
        seq = self.replay["actions"]
        idx = self.replay["idx"]
        if idx >= len(seq):
            self.replay = None
            return (self._click_policy(grid, frame, avail) if self.mode == "click"
                    else self._move_policy(grid, frame, avail))
        stored = seq[idx]
        aid, data = stored["aid"], dict(stored["data"])
        self.replay["idx"] = idx + 1
        self.pending = {
            "aid": aid, "data": data, "kind": "replay", "replay": True,
            "replay_idx": idx, "expected": stored["effect"], "predicted": None}
        return aid, data, {"strategy": "verified_replay",
                           "step": f"{idx + 1}/{len(seq)}"}

    def _replay_abort(self, cause):
        seqrec = self.knowledge.sequences.get(self.level)
        if seqrec is not None:
            seqrec["aborts"] = seqrec.get("aborts", 0) + 1
            if seqrec["aborts"] >= REPLAY_ABORT_STALE:
                seqrec["stale"] = True
                self.ledger.append("SEQUENCE_STALE", level=self.level,
                                   cause=cause)
        self.ledger.append("REPLAY_ABORT", level=self.level, cause=cause)
        self.replay = None

    # ------------------------------------------------------------- click games
    def _select_targets(self, sel, objs):
        if sel == "rarest_color":
            counts = {}
            for o in objs:
                counts[o["color"]] = counts.get(o["color"], 0) + o["size"]
            if not counts:
                return []
            rare = min(counts, key=lambda c: (counts[c], c))
            return [o for o in objs if o["color"] == rare]
        if sel == "smallest_object":
            return [min(objs, key=lambda o: (o["size"], o["bbox"]))]
        if sel == "corner_sample_color":
            corner = [o for o in objs
                      if o["bbox"][0] < CELL and o["bbox"][1] < CELL]
            if not corner:
                return []
            sample = corner[0]
            return [o for o in objs
                    if o["color"] == sample["color"] and o["id"] != sample["id"]]
        if sel == "largest_object":
            return [max(objs, key=lambda o: o["size"])]
        return []

    def _click_policy(self, grid, frame, avail, _depth=0):
        objs = find_objects(grid)
        if not objs:
            return self._no_signal_explore(grid, avail)
        active = [s for s in self.sel_order
                  if self.knowledge.rules.get(s) is None
                  or self.knowledge.rules[s].status != STATUS_QUARANTINED]
        if not active:
            active = list(self.sel_order)
        sel = active[0]
        if not self.queue:
            targets = self._select_targets(sel, objs)
            counts = {}
            for o in objs:
                counts[o["color"]] = counts.get(o["color"], 0) + 1
            targets = sorted(targets, key=lambda o: (counts[o["color"]], o["size"],
                                                     o["bbox"]))
            self.queue = targets
            self.queue_ptr = 0
        target = None
        while self.queue_ptr < len(self.queue):
            cand = self.queue[self.queue_ptr]
            cy, cx = cand["centroid"]
            cell = (int(cy) // CELL, int(cx) // CELL)
            if (cand["color"], cell) not in self.clicked_sig:
                target = cand
                break
            self.queue_ptr += 1
        if target is None:
            if _depth >= len(CLICK_SELECTORS):
                # v2.1 M2' FIX: design §4.2 — no-signal click exploration
                # must be SALIENCE-ORDERED. The received code's blind cell
                # sweeps lost lp85's level-1 clear in the 25-game head-to-
                # head (v2.0's proven fallback clicked salient objects).
                return self._salience_explore(grid, objs, avail)
            self._demote(sel, why="all candidates clicked, no progress")
            return self._click_policy(grid, frame, avail, _depth + 1)
        mask = np.zeros_like(grid, dtype=bool)
        for (r, c) in target["pixels"]:
            mask[r, c] = True
        predicted = prim_recolor_region(grid, mask, 0)
        cy, cx = target["centroid"]
        x = min(max(int(round(cx)), 0), 63)
        y = min(max(int(round(cy)), 0), 63)
        sig = (target["color"], (int(cy) // CELL, int(cx) // CELL))
        self.pending = {"aid": 6, "kind": "click", "rule": sel,
                        "predicted": predicted, "sig": sig,
                        "data": {"x": x, "y": y, "game_id": self.game_id}}
        self.queue_ptr += 1
        rec = self.knowledge.rule(sel)
        return 6, dict(self.pending["data"]), {
            "strategy": "program_synthesis_v3", "hypothesis": sel,
            "evidence_class": rec.evidence_class}

    # ------------------------------------------------------------- move games
    def _move_policy(self, grid, frame, avail):
        avail = [a for a in avail if a != 0]
        unprobed = [a for a in avail
                    if a not in self.dirs
                    and self.probe_attempts.get(a, 0) < PROBE_ATTEMPT_CAP]
        if unprobed:
            aid = unprobed[0]
            self.pending = {"aid": aid, "kind": "probe", "predicted": None,
                            "data": {"game_id": self.game_id}}
            return aid, {"game_id": self.game_id}, {
                "strategy": "program_synthesis_v3", "phase": "probe controls"}
        objs = find_objects(grid)
        if not objs or self.player_color is None:
            return self._no_signal_explore(grid, avail)
        players = [o for o in objs if o["color"] == self.player_color]
        others = [o for o in objs if o["color"] != self.player_color]
        if not players or not others:
            return self._no_signal_explore(grid, avail)
        player = players[0]
        goal = salience_rank(others)[0][0]
        py, px = player["centroid"]
        gy, gx = goal["centroid"]

        def dist_after(a):
            dr, dc = self.dirs.get(a, (0, 0))
            return abs(py + dr * CELL - gy) + abs(px + dc * CELL - gx)

        ordered = sorted(avail, key=dist_after)
        if self.stuck >= 2:
            ordered = ordered[1:] + ordered[:1]
            self.stuck = 0
        aid = ordered[0]
        dr, dc = self.dirs.get(aid, (0, 0))
        mask = np.zeros_like(grid, dtype=bool)
        for (r, c) in player["pixels"]:
            mask[r, c] = True
        predicted = prim_move_region(grid, mask, dr * CELL, dc * CELL)
        self.pending = {"aid": aid, "kind": "move", "predicted": predicted,
                        "data": {"game_id": self.game_id}}
        return aid, {"game_id": self.game_id}, {
            "strategy": "program_synthesis_v3",
            "controls": {str(k): v for k, v in self.dirs.items()}}

    # ------------------------------------------------------------- v2.1 explore
    def _salience_explore(self, grid, objs, avail):
        """M2' FIX (design §4.2): salience-ordered no-signal click
        exploration — v2.0's proven exhaustion fallback, restored, with
        graph bookkeeping (objects already tried at this state are
        skipped). These clicks are v2-baseline behaviour and deliberately
        do NOT count against EXPLORATION_CAP (worst case = v2.0 exactly).
        Blind cell enumeration remains the last resort."""
        ranked, _ = salience_rank(objs)
        h = self.graph.hash_grid(grid)
        tried = self.graph.transitions.get(h, {}) if h is not None else {}
        tried_cells = {k[1:] for k in tried
                       if isinstance(k, tuple) and k[0] == 6}
        for o in ranked:
            cy, cx = o["centroid"]
            cell = (int(cy) // CELL, int(cx) // CELL)
            if cell in tried_cells:
                continue
            if (o["color"], cell) in self.clicked_sig:
                continue
            x = min(max(int(round(cx)), 0), 63)
            y = min(max(int(round(cy)), 0), 63)
            self.pending = {"aid": 6, "kind": "explore", "predicted": None,
                            "sig": (o["color"], cell),
                            "data": {"x": x, "y": y, "game_id": self.game_id}}
            return 6, dict(self.pending["data"]), {
                "strategy": "salience_explore", "object": o["id"],
                "color": o["color"], "cell": cell,
                "note": "salience-ordered no-signal click"}
        # every salient object tried at this state: blind cell sweep
        return self._no_signal_explore(grid, avail)

    def _no_signal_explore(self, grid, avail):
        """Novelty-seeking fallback. Uses the state-action graph to pick
        an untried action at the current state, or the least-visited
        successor. Never returns None unless avail is empty."""
        if not avail:
            return 0, {"game_id": self.game_id}, {"note": "no actions"}

        def _rotate(note):
            # FIX (audit B2/B3): bounded fallback — never repeat one dead
            # action forever; rotate through the legal actions instead
            aid = int(avail[self._explore_rr % len(avail)])
            self._explore_rr += 1
            data = {"game_id": self.game_id}
            if aid == 6:
                data = {"x": 32, "y": 32, "game_id": self.game_id}
            self.pending = {"aid": aid, "kind": "explore",
                            "predicted": None, "data": data}
            return aid, dict(data), {
                "strategy": "state_graph_explore", "action": aid,
                "note": note}

        # FIX (audit B3): exploration budget cap — after EXPLORATION_CAP
        # consecutive explore actions without progress, degrade to the
        # bounded rotation for the rest of the level (design §4.3)
        if self.explore_blocked:
            return _rotate("exploration capped")

        h = self.graph.hash_grid(grid)
        choice = self.graph.novelty_action(h, avail)

        # Novelty might return (6, r, c) tuple for untried click cells
        if isinstance(choice, tuple) and len(choice) == 3 and choice[0] == 6:
            _, r, c = choice
            x = min(max(c * CELL + CELL // 2, 0), 63)
            y = min(max(r * CELL + CELL // 2, 0), 63)
            cell = (r, c)
            self.pending = {"aid": 6, "kind": "explore", "predicted": None,
                            "sig": cell,
                            "data": {"x": x, "y": y, "game_id": self.game_id}}
            self.explore_streak += 1
            if self.explore_streak >= EXPLORATION_CAP:
                self.explore_blocked = True
                self.ledger.append("EXPLORATION_CAPPED",
                                   streak=self.explore_streak)
            return 6, dict(self.pending["data"]), {
                "strategy": "state_graph_explore", "action": 6,
                "cell": cell, "state": h,
                "note": "novel click cell"}
        if choice is not None:
            aid = int(choice)
            data = {"game_id": self.game_id}
            if aid == 6:
                data = {"x": 32, "y": 32, "game_id": self.game_id}
            self.pending = {"aid": aid, "kind": "explore", "predicted": None,
                            "data": data}
            self.explore_streak += 1
            if self.explore_streak >= EXPLORATION_CAP:
                self.explore_blocked = True
                self.ledger.append("EXPLORATION_CAPPED",
                                   streak=self.explore_streak)
            return aid, dict(data), {
                "strategy": "state_graph_explore", "action": aid,
                "state": h, "note": "novel action"}

        # Graph exhausted at this state: bounded rotation
        return _rotate("graph exhausted")

    # ------------------------------------------------------------- helpers
    def _demote(self, sel, why):
        if sel in self.sel_order:
            self.sel_order.remove(sel)
            self.sel_order.append(sel)
        self.queue, self.queue_ptr = [], 0
        rec = self.knowledge.rule(sel)
        rec.rotations += 1
        if rec.rotations >= 2 and rec.success == 0:
            rec.status = STATUS_QUARANTINED
            self.ledger.append("RULE_QUARANTINED", rule=sel, cause=why)
        else:
            self.ledger.append("RULE_DEMOTED", rule=sel, cause=why)

    def _clear_level_state(self):
        self.queue, self.queue_ptr = [], 0
        self.clicked_sig = set()
        self.blacklist = set()
        self.pending = None
        self.stuck = 0
        self.level_actions = []
        self.level_verified = False
        self.replay = None
        # v2.1: recent buffer restarts fresh on new level; the exploration
        # cap also resets (design §4.3: cap is per level)
        self.explore_streak = 0
        self.explore_blocked = False
        if self.graph is not None:
            self.graph.clear_recent()

    def _grid(self, frame):
        g = merged_view(frame)
        if g is None:
            g = _grid_of(frame)
        return g

    def _available(self, frame):
        try:
            avail = list(getattr(frame, "available_actions", None) or [])
            return [int(a) for a in avail]
        except Exception:
            return []


# ============================================================================
# SECTION 6 — KIT-FACING AGENT (never-hang wrapper + protocol discipline)
# ============================================================================
# FIX (audit B1): the ARC-AGI-3-Agents framework drives agents via
# Agent.main(); when the framework is importable we subclass its base so
# main()/frames/counter all work (local `make play-local` and the Kaggle
# gateway). Standalone (no framework), the class duck-types the same
# surface. Restored verbatim from the M2-proven v2 wrapper.
try:
    from agents.agent import Agent as _KitAgent   # ARC-AGI-3-Agents framework
    KIT_FRAMEWORK = True
except Exception:
    _KitAgent = object
    KIT_FRAMEWORK = False


class ProgramSynthesisV2Agent(_KitAgent):
    """MyAgent for the ARC-AGI-3 Kaggle starter kit.

    - choose_action / is_done match the official Agent ABC surface; when the
      framework is present the inherited main() loop drives this class.
    - Every decide path is wrapped: any exception or missing field falls
      back to the cheapest legal action (RESET when required) — it can
      degrade, never hang, never raise.
    - Constructor accepts any kit-style arguments (game_id is sniffed from
      kwargs or the second positional argument when present).
    - v2.2: per-game telemetry (K2). Observes frames via choose_action and
      keeps k5_per_game.csv current. Fully guarded: a telemetry failure is
      logged once and telemetry disables itself; gameplay is unaffected.
    """

    MAX_ACTIONS = 80
    TIME_BUDGET_S = 1.0

    def __init__(self, *args, **kwargs):
        verbose = bool(kwargs.pop("verbose", False))
        self._kit = False
        if _KitAgent is not object:
            try:
                super().__init__(*args, **kwargs)
                self._kit = True
            except Exception:
                self._kit = False   # bad args for the kit signature: standalone
        if not self._kit:
            self.agent_name = (kwargs.get("agent_name")
                               or (args[2] if len(args) > 2 else None)
                               or "ProgramSynthesisV4")
        gid = (kwargs.get("game_id")
               or (args[1] if len(args) > 1 and isinstance(args[1], str) else "")
               or "")
        self.core = ProgramSynthesisV2Core(verbose=verbose)
        self.action_counter = 0     # bumped by the harness (official main())
        self._steps = 0             # internal mirror, bumped by choose_action
        # v2.2 telemetry state
        self._k5_games = {}
        self._k5_io_ok = True
        self.begin_episode(gid or "")

    # -- lifecycle -----------------------------------------------------------
    def begin_episode(self, game_id):
        self.core.new_episode(game_id)

    # -- v2.2 telemetry (K2): observe-only; must never influence decisions ----
    def _k5_dir(self):
        import os
        return (os.environ.get("K5_TELEMETRY_DIR") or "/kaggle/working")

    def _k5_note(self, frame):
        """Update the row for the current game and flush the CSV.

        Called from choose_action AFTER the action was decided (never before),
        so telemetry cannot feed back into play. CSV content is a pure
        function of the observed frame sequence + action counts: same run,
        same bytes. Flushed every call (write early, write often).
        """
        if not self._k5_io_ok:
            return
        try:
            gid = str(getattr(frame, "game_id", "") or "")
            if not gid:
                return
            row = self._k5_games.get(gid)
            if row is None:
                row = {"game_id": gid, "levels_completed": 0, "win_levels": 0,
                       "win": 0, "actions_used": 0, "decision_events": 0,
                       "final_state": ""}
                self._k5_games[gid] = row
            state = _state_name(getattr(frame, "state", None))
            row["levels_completed"] = max(
                row["levels_completed"],
                int(getattr(frame, "levels_completed", 0) or 0))
            row["win_levels"] = max(
                row["win_levels"], int(getattr(frame, "win_levels", 0) or 0))
            row["win"] = 1 if state == "WIN" else row["win"]
            row["actions_used"] = max(self.action_counter, self._steps)
            row["decision_events"] = len(self.core.ledger)
            if state:
                row["final_state"] = state
            self._k5_flush()
        except Exception:
            self._k5_io_ok = False
            try:
                print("[k5] telemetry disabled after an I/O error (gameplay unaffected)")
            except Exception:
                pass

    def _k5_flush(self):
        import csv
        import hashlib
        import json
        import os
        d = self._k5_dir()
        os.makedirs(d, exist_ok=True)
        cols = ["run_variant", "run_agent_sha256", "run_knobs_sha256",
                "game_id", "levels_completed", "win_levels", "win",
                "actions_used", "decision_events", "final_state"]
        variant = str(K5_RUN_INFO.get("variant") or "")
        asha = str(K5_RUN_INFO.get("agent_sha256") or "")
        ksha = hashlib.sha256(json.dumps(K5_RUN_INFO.get("knobs") or {},
                                         sort_keys=True).encode()).hexdigest()
        tmp = os.path.join(d, "k5_per_game.csv.tmp")
        final = os.path.join(d, "k5_per_game.csv")
        # Merge-on-write: the harness may build a fresh MyAgent per game, so
        # self._k5_games never accumulates across games. Re-read whatever is
        # already on disk and union it in before writing, so the CSV always
        # covers every game of the run. Counters take max (monotonic), win
        # is monotonic too, final_state prefers the live row. A corrupt or
        # absent prior file is ignored: telemetry degrades, never breaks.
        rows = {gid: dict(row) for gid, row in self._k5_games.items()}
        try:
            import io
            with open(final) as fh:
                body = "".join(ln for ln in fh if not ln.startswith("#"))
            if body.strip():
                for old in csv.DictReader(io.StringIO(body)):
                    gid = old.get("game_id") or ""
                    if not gid:
                        continue
                    cur = rows.get(gid)
                    if cur is None:
                        rows[gid] = old
                    else:
                        for k in ("levels_completed", "win_levels", "win",
                                  "actions_used", "decision_events"):
                            try:
                                cur[k] = max(int(cur[k] or 0), int(old[k] or 0))
                            except (TypeError, ValueError):
                                pass
                        if not cur.get("final_state") and old.get("final_state"):
                            cur["final_state"] = old["final_state"]
        except Exception:
            pass
        with open(tmp, "w", newline="") as fh:
            fh.write("# k5 telemetry v1 (a FILTER/observation aid, not a score)\n")
            fh.write("# variant=%s agent_sha256=%s knobs_sha256=%s\n"
                     % (variant, asha, ksha[:16]))
            w = csv.DictWriter(fh, fieldnames=cols)
            w.writeheader()
            for gid in sorted(rows):
                row = dict(rows[gid])
                row.update(run_variant=variant, run_agent_sha256=asha,
                           run_knobs_sha256=ksha[:16])
                w.writerow(row)
        os.replace(tmp, final)
        info = dict(K5_RUN_INFO)
        info["knobs_sha256"] = ksha
        with open(os.path.join(d, "k5_run_info.json"), "w") as fh:
            json.dump(info, fh, indent=1, sort_keys=True)

    # -- official ABC surface --------------------------------------------------
    def is_done(self, frames, latest_frame):
        try:
            frame = _pick_frame(frames, latest_frame)
            # Telemetry here too: the terminal (WIN) frame is only ever
            # offered to is_done, never to choose_action, so noting here is
            # the only way the final level total reaches the CSV row.
            self._k5_note(frame)
            if _state_name(getattr(frame, "state", None)) == "WIN":
                return True
            return (self.action_counter >= self.MAX_ACTIONS
                    or self._steps >= self.MAX_ACTIONS)
        except Exception:
            return False

    def choose_action(self, frames, latest_frame):
        try:
            # kit main() passes the converted current observation as arg 2
            # (frames[0] is a dummy); runner mode passes frames[-1] itself
            frame = _pick_frame(frames, latest_frame)
            t0 = time.perf_counter()
            aid, data, reasoning = self.core.next_action(frame)
            action = self._make_action(aid, data, reasoning)
            self._steps += 1
            if time.perf_counter() - t0 > self.TIME_BUDGET_S:
                self.core.ledger.append(
                    "SLOW_DECISION",
                    seconds=round(time.perf_counter() - t0, 3))
            self._k5_note(frame)   # telemetry AFTER the decision (observe-only)
            return action
        except Exception:
            self._steps += 1
            return self._fallback(frames, latest_frame)

    @staticmethod
    def _make_action(aid, data, reasoning):
        action = GameAction.from_id(int(aid))
        if data:
            try:
                action.set_data({k: (int(v) if k in ("x", "y") else v)
                                 for k, v in data.items()})
            except Exception:
                pass
        try:
            action.reasoning = reasoning
        except Exception:
            pass
        return action

    def _fallback(self, frames, latest_frame):
        try:
            state = _state_name(getattr(latest_frame, "state", None))
            if state in ("NOT_PLAYED", "GAME_OVER"):
                return self._make_action(0, {"game_id": self.core.game_id},
                                         {"strategy": "fallback_reset"})
            avail = []
            try:
                avail = [int(a) for a in
                         (getattr(latest_frame, "available_actions", None) or [])]
            except Exception:
                avail = []
            if not avail:
                avail = [1, 2, 3, 4, 6]
            aid = avail[0]
            if aid == 6:
                data = {"x": 32, "y": 32, "game_id": self.core.game_id}
            else:
                data = {"game_id": self.core.game_id}
            return self._make_action(aid, data, {"strategy": "safe_fallback"})
        except Exception:
            try:
                return GameAction.RESET
            except Exception:
                return None


MyAgent = ProgramSynthesisV2Agent
'''.lstrip('\n')
if not t.is_file() or t.stat().st_size < 100:
    t.write_text(EMBEDDED)
    print('EMBEDDED FALLBACK used ->', t.name, t.stat().st_size, 'bytes')
else:
    cur = t.read_text()
    print('using fetched', t.name, t.stat().st_size, 'bytes',
          '(identical to embedded)' if cur == EMBEDDED else
          '(DIFFERS from embedded: the repo fetch is the source of truth)')
print('  sha256', hashlib.sha256(t.read_bytes()).hexdigest()[:16], t.name)


In [ ]:
# ## 3 — Auth (never aborts)
#
# Order: `KAGGLE_API_TOKEN`, then `KAGGLE_API_V1_TOKEN` (what Kaggle injects itself — mapped into the modern name), then token files or `KAGGLE_USERNAME`+`KAGGLE_KEY`, then an interactive prompt **only when a TTY exists**, and finally Kaggle's ambient notebook auth. A scheduled (papermill) run has no stdin, so no code path here may raise.
pass


In [ ]:
import os, pathlib, sys
def _have_file():
    for p in ('~/.kaggle/access_token', '~/.kaggle/access_token.txt',
              '~/.kaggle/kaggle.json'):
        f = pathlib.Path(os.path.expanduser(p))
        if f.is_file() and f.read_text().strip():
            return p
    return None
if os.environ.get('KAGGLE_API_TOKEN'):
    print('auth: KAGGLE_API_TOKEN is set')
elif os.environ.get('KAGGLE_API_V1_TOKEN'):
    os.environ['KAGGLE_API_TOKEN'] = os.environ['KAGGLE_API_V1_TOKEN']
    print('auth: mapped KAGGLE_API_V1_TOKEN (Kaggle-injected) into KAGGLE_API_TOKEN')
elif _have_file():
    print('auth: using', _have_file())
elif os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
    print('auth: using KAGGLE_USERNAME/KAGGLE_KEY')
elif sys.stdin is not None and sys.stdin.isatty():
    import getpass
    tok = getpass.getpass('Paste a KGAT_ token (blank = use ambient auth): ').strip()
    if tok:
        os.environ['KAGGLE_API_TOKEN'] = tok
    print('continuing with whatever the environment provides')
else:
    print('auth: no token and no TTY (scheduled run) — using Kaggle '
          'ambient notebook auth and CONTINUING')

In [ ]:
# ## 4 — Free dry run
#
# Probes the CLI, verifies the 5 inputs + agent pin + defaults, dry-runs the lever+knob injection, scans the fork for the K1 knob channel, then probes submission-limits (informational). **Pushes nothing.**
pass


In [ ]:
%cd /kaggle/working/kaggle5
!python kaggle5.py --stage check

In [ ]:
# ## 5 — Read-only: what would today do?
#
# Reconciles any new submission against the pending row and prints the experiment table + the next plan. Pushes nothing.
pass


In [ ]:
!python kaggle5.py --stage plan

In [ ]:
# ## 6 — THE DAILY CELL (this is the one you schedule)
#
# | situation | what it does |
# |---|---|
# | yesterday's submission scored | reconciles, then plans from it |
# | pending row still unanswered | prints why, pushes nothing |
# | variant planned | toy-filter -> pending row -> push -> **stop** |
# | filter rejects the variant | records it, pushes nothing, moves on tomorrow |
# | queue exhausted | prints the incumbent, pushes nothing, ever |
#
# A scheduled run never blocks 9 h and never submits. It emails you the run log; the bottom of a push log is the human card (what to tap).
pass


In [ ]:
!python kaggle5.py --stage daily

In [ ]:
# ## 7 — Manual ops (optional, run only by hand when needed)
pass


In [ ]:
# Optional manual ops — uncomment if needed:
# !python kaggle5.py --stage plan
# !python kaggle5.py --stage score
# !python kaggle5.py --abandon
pass

In [ ]:
# ## The machine never submits — what you do instead
#
# After a push, the run prints the card. The taps:
#
# 1. open the fork: https://www.kaggle.com/code/mohamedmalikturay/arc-agi-3-milestone-2-solution-3e9adb
# 2. wait for the pushed version to finish, then '...' -> **Submit to arc-prize-2026-arc-agi-3** -> pick the pushed version
# 3. paste the message the card prints (`kaggle5 <variant> v<N>`) — it is how tomorrow's run matches the score to the experiment with certainty
#
# Skip a day? The driver waits ~2 days, marks the row stale, and moves on — nothing is silently dropped. Want to skip deliberately? Run `--abandon` from cell 7 or tell the next run to.
#
# ## Scheduling it
#
# Settings -> **Schedule a notebook to run** -> Daily. CPU-only notebooks can be scheduled (this one is); runs fire roughly on the UTC date; every run emails you its log, pass or fail; save a version first — the schedule runs the saved notebook.
#
# LOCAL_UTC_OFFSET_HOURS in kaggle5.py is 0 (Freetown is UTC+0 all year).
pass
